# gnn-spot: spot capacity for synchronous training — full project in one notebook

Reproduction of *When Does Spot Capacity Pay for Synchronous Training* (analytical + simulated; **no real cluster measured**).

**How to use:** Run all cells top to bottom. Section 1 writes every project file to disk (works offline; needs only numpy/scipy/pandas/matplotlib/pyyaml/mpmath, which Kaggle has). Section 2 runs tests and the reproduction. Known result: 59/60 verification checks pass; the Fig. 5 halting crossing (computed ≈1.77 vs printed ≈1.9) is unresolved. Reconstructed experiments are labelled in the reports.

## 1. Create the project files

In [ ]:
import os, sys, subprocess
PROJ = '/kaggle/working/gnn-spot-training' if os.path.exists('/kaggle/working') else os.path.abspath('gnn-spot-training')
dirs = ["configs", "docs", "paper", "scripts", "src/gnn_spot", "src/gnn_spot/checkpointing", "src/gnn_spot/deadlines", "src/gnn_spot/experiments", "src/gnn_spot/models", "src/gnn_spot/optimization", "src/gnn_spot/plotting", "src/gnn_spot/recovery", "src/gnn_spot/reporting", "src/gnn_spot/simulation", "src/gnn_spot/verification", "tests", "tests/integration", "tests/property", "tests/regression", "tests/unit"]
for d in dirs:
    os.makedirs(os.path.join(PROJ, d), exist_ok=True)
for d in ['results/tables','results/figures','results/raw','results/manifests','notebooks']:
    os.makedirs(os.path.join(PROJ, d), exist_ok=True)
os.chdir(PROJ)
print('project dir:', PROJ)

In [ ]:
%%writefile README.md
# gnn-spot-training

Executable, tested reproduction of **“When Does Spot Capacity Pay for Synchronous Training: Cost, Deadline and Recovery-Design Regimes”**
(Katkuri, Ananya JPS, Srividyambika). The paper is analytical + simulated; **no real cluster was measured, here or in the paper.**

## What is here
* `src/gnn_spot/models` – performance model (Eq. 1–2, Prop. 1), Lambert-W checkpoint optimum (Lemma 1, numerically stable), spot cost factor, break-even.
* `checkpointing`, `recovery`, `deadlines`, `optimization` – Young/Daly baselines, recovery designs & warnings (Thm 3, Prop. 2), completion-time law & Cantelli/Chernoff (Prop. 3, App. A), deadline slack (Thm 1–2).
* `simulation` – two independent rollback simulators (vectorised + explicit state machine), halting/elastic simulators, Weibull/burst processes.
* `verification` – 60 executable checks (`checks.py`); `reference.py` holds paper-printed numbers as comparison targets only.
* `experiments`, `plotting`, `reporting` – tables, figures, robustness reconstruction, sensitivity/ablation/stress extensions, generated reports.
* `docs/` – `MATHEMATICAL_MODEL.md` (equations + deviation log), `VALIDATION_REPORT.md` (generated), `LIMITATIONS.md`, `REPRODUCIBILITY.md`, `IMPLEMENTATION_AUDIT.md`, `IMPLEMENTATION_PLAN.md`. `paper/reproduction_report.md` (generated).

## Install and run
```bash
python -m pip install -e ".[dev]"
pytest
python -m gnn_spot --help
python scripts/reproduce_paper.py --mode quick   # ~2 min
python scripts/reproduce_paper.py --mode full    # ~4 min on one core
python scripts/generate_report.py
```

## Headline results (full mode; see `docs/VALIDATION_REPORT.md` for every check)
* 59 of 60 verification checks pass; **1 unresolved**: Fig. 5 halting crossing computed at u≈1.77 vs ≈1.9 printed.
* Table IX theory (K, mean, std) reproduced exactly; Chernoff/Cantelli examples reproduced; Tables III, IV, VI, Remark 1 numbers reproduced (Table VI needs the exact rate −ln0.8/730).
* Simulation agrees with analytical moments (|z|<3.5) for Table IX, overlapping stalls, warnings (incl. a derived exact-K mean), halting and elastic designs.
* **Reconstructed** (paper omits configs): robustness study (32/32 series have no significant interior optimum; Weibull 0.6 +10…13%, Weibull 1.5 −3.5…−6%, burst 1.34 vs 1.97 at MTBR 3 h), Prop. 2 runs, Thm 2 brute force (continuous n*≈21.5 vs n_D=9; integer optimum n=20, n_s=18).

## Limitations
See `docs/LIMITATIONS.md`. Equations in the supplied `.docx` were illegible and were reconstructed and independently verified.


In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]
name = "gnn-spot"
version = "0.1.0"
description = "Executable reproduction of 'When Does Spot Capacity Pay for Synchronous Training: Cost, Deadline and Recovery-Design Regimes'"
requires-python = ">=3.10"
dependencies = [
  "numpy>=1.24",
  "scipy>=1.10",
  "pandas>=2.0",
  "matplotlib>=3.7",
  "pyyaml>=6.0",
  "mpmath>=1.3",
]

[project.optional-dependencies]
dev = ["pytest>=7.4", "hypothesis>=6.80", "sympy>=1.12"]

[project.scripts]
gnn-spot = "gnn_spot.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-q"
markers = ["slow: statistically heavy tests (deselect with -m 'not slow')"]


In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
*.egg-info/
.pytest_cache/
.hypothesis/
.venv/
build/
dist/
.ipynb_checkpoints/


In [ ]:
%%writefile configs/baseline.yaml
# Paper's main numerical-verification parameters (Section X).  Test parameters, NOT claims about any cloud.
# Durations R, delta are in SECONDS here and converted to hours at load time.
perf:                      # Eq. (1); theta=1, A=alpha=0 reproduces the closed forms of Prop. 1(c),(d)
  a: 3600.0                # s, single-worker compute time
  theta: 1.0
  A: 0.0
  kappa0: 1.0
  gamma: 0.0
  sigma: 20.0              # s
  alpha: 0.0
spot:
  R_s: 300.0
  delta_s: 30.0
  rho: 0.35
  n_s: 8                   # spot workers in the Table IX / moment checks
  T_N_h: 7.40
  z: 40.0
  stall: fixed             # (S4a); 'overlap' = (S4b)


In [ ]:
%%writefile configs/paper_verification.yaml
# Run sizes for the verification suite.  'full' uses the trial counts reported in the paper where stated.
modes:
  quick:
    prop1_grid: 20000
    lemma1_points: 60
    thm1_configs: 3000
    mc_trials: 8000
    prop2_trials: [8000, 40000]
    robustness_runs: 60
  full:
    prop1_grid: 400000       # paper: 4e5 points
    lemma1_points: 400
    thm1_configs: 3000       # paper: 3,000
    mc_trials: 40000         # paper: 40,000 (Table IX)
    prop2_trials: [40000, 200000]   # paper: 40,000 and 200,000
    robustness_runs: 400     # paper: 250-400 per point
seeds:
  master: 20261016


In [ ]:
%%writefile configs/robustness.yaml
# RECONSTRUCTION of Sec. IX-B from its published description (original per-configuration data unavailable).
T_N_h: 8.0
delta_s: 30.0
R_s: 300.0
n: 16
rho: 0.35
processes: [exponential, weibull0.6, weibull1.5, burst]
burst_group: 4
mtbr_h: [0.3, 0.5, 1.2, 3.0, 6.0, 12.0, 24.0, 48.0]     # eight values spanning 0.3-48 h (exact values not published)
n_s_values: [0, 2, 4, 6, 8, 10, 12, 14, 16]               # sweep 0..16 (exact grid not published)
runs_per_point: 400
seed: 20261016


In [ ]:
%%writefile configs/sensitivity.yaml
base:
  n: 32
  s: 0.5
  rho: 0.35
  R_s: 300.0
  delta_s: 30.0
  mtbr_h: 12.0
  stall: fixed
ranges:
  n: [4, 8, 16, 32, 64, 128, 256]
  s: [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
  delta_s: [5, 15, 30, 60, 120, 300]
  R_s: [60, 150, 300, 600, 1800, 3600]
  rho: [0.1, 0.2, 0.35, 0.5, 0.7, 0.9]
  mtbr_h: [1, 2, 3, 6, 12, 24, 48, 100]
  deadline_multiple: [1.05, 1.1, 1.2, 1.5, 2.0, 3.0]


In [ ]:
%%writefile docs/IMPLEMENTATION_AUDIT.md
# Implementation audit

Status key: DONE (implemented and executed), PARTIAL, BLOCKED, NOT DONE.

## Starting state
Empty repository; only `GNN.docx` (PDF-style export; equations illegible). Everything was built from the paper text.

## Paper-to-code mapping
See the traceability matrix in `paper/reproduction_report.md` §9 (generated from `reporting/report.py: TRACE`).

## Deliverables
| Item | Status | Note |
|---|---|---|
| A Performance model, Prop. 1 | DONE | derivative/grid/root checks pass |
| B Spot model, Lemma 1, Lambert-W | DONE | stable Newton path; 60-digit mpmath check |
| C Theorem 1 | DONE | 3,000 random configs, 0 violations (our ranges) |
| D Theorem 2 | DONE / RECONSTRUCTED | local vs global conditions separate; config reconstructed |
| E Theorem 3 | DONE | plus new halting/elastic simulations |
| F Prop. 2 warnings | DONE / RECONSTRUCTED | exact-K mean derived and verified |
| G Appendix A, Prop. 3 | DONE | variance derived (printed Eq. 11 illegible); Table IX theory exact |
| H Baselines, Table III | DONE | one borderline cell (26.84 vs 26.9) |
| Simulation framework | DONE | two independent simulators cross-validated |
| Robustness (Sec. IX-B) | RECONSTRUCTED | original data unavailable |
| Figures 3–6 + extras | DONE | PNG+PDF+data CSV |
| Tests | DONE | 107 tests (unit, property, regression, integration) pass |
| Clean-install test | DONE | fresh venv: `pip install -e ".[dev]"`, CLI, `reproduce_paper.py --mode quick` all ran; 59/60 checks |
| Fig. 5 crossing (≈1.9) | UNRESOLVED | computed ≈1.77 |

## Issues found while building
* `floor((D-T_N)/(δ_e+R))` off-by-one risk in the φ=1 Poisson tail → fixed with a tolerance (found by a unit test).
* Table VI only matches with the exact λ_w, not rounded 3.1e-4.
* Several of my own test criteria were wrong (naive residual at v=1e-12, stale variable, comparing against the spot endpoint instead of min(spot, no-spot)); fixed after diagnosis, not by loosening tolerances.
* Paper-side ambiguities: Eq. (11), Prop. 2 run configuration, Thm 2 brute-force configuration, halting billing, Fig. 5 crossing.

## Phase log
Phases 1–9 executed. Phase 9 was run for `--mode quick` only (full mode was run in the dev environment, not re-run in the clean venv).


In [ ]:
%%writefile docs/IMPLEMENTATION_PLAN.md
# Implementation plan and status

| Milestone | Acceptance criterion | Verification command | Status |
|---|---|---|---|
| M1 Core models | unit tests pass | `pytest tests/unit` | DONE |
| M2 Simulators | cross-validated vs analytical and each other | `python -m gnn_spot verify --only sim_crossval overlap` | DONE |
| M3 Verification suite | all checks executed, statuses recorded | `python -m gnn_spot verify --mode full` | DONE (59/60) |
| M4 Reproduction | tables, figures, manifests | `python scripts/reproduce_paper.py --mode full` | DONE |
| M5 Reports | generated from saved results | `python scripts/generate_report.py` | DONE |
| M6 Hardening | property + integration tests | `pytest` | DONE (107 pass) |
| M7 Clean install | fresh venv, README commands | `python -m venv .venv && .venv/bin/pip install -e ".[dev]" && .venv/bin/pytest` | DONE (quick mode) |

## Next steps
1. Investigate the Fig. 5 crossing (try halting with finite-job first-passage; ask authors for the figure script).
2. Ask authors for the Sec. IX-B, Prop. 2 and Thm 2 scripts/seeds to convert RECONSTRUCTED items into exact reproductions.
3. Replay public spot-interruption traces (the paper's own future work).


In [ ]:
%%writefile docs/LIMITATIONS.md
# Limitations

## Inherited from the paper (preserved, not hidden)
1. **No empirical validation.** All results are derived or checked in simulation of the *same* model. No epoch times, bills or revocation traces from real systems are used here or in the paper.
2. **Independent exponential revocations.** Theorems 1–3 assume them. The Weibull/burst robustness check does not cover market-wide events whose rate does not fall with cluster size; the Laplace-transform results are for the fail-free stall (S4a) only.
3. **Model forms.** Homogeneous workers, power-law `a n^-θ` and `κ0 n^γ`, constant prices, no on-demand fallback, separator law needs bounded degree; warnings, anchor workers and elastic continuation are modelled at the level of rates; **elastic continuation ignores statistical efficiency**.
4. **Published inputs.** The AWS interruption bucket is open-ended and read as a per-worker probability; `R ∈ [5,60]` min is assumed; the 44–83% fetch share is one DGL preprint.
5. **Targeted (not systematic) literature search**; Prop. 2 and Appendix A overlap with prior work.

## Specific to this reproduction
6. **Illegible source equations.** The supplied `.docx` is a PDF-style export; formulas were reconstructed from text and independently verified (see `MATHEMATICAL_MODEL.md`). Any equation whose *printed form* we could not read (notably Eq. (11) and Prop. 2's proof details) was *derived*, not transcribed; agreement with printed numbers is the evidence that the reading is right, and is reported per item.
7. **Unavailable data/scripts.** The paper states that the authors' simulation scripts, seeds and per-configuration outputs are available only on request. Therefore:
   * the Sec. IX-B robustness experiment (Table VII), the Prop. 2 runs (9.072 vs 9.066 h; 0.4002 vs 0.4003) and the Theorem 2 brute force (`n=21` vs `n_D=9`) are **reconstructed**, with all assumed parameters in `configs/` and flagged `RECONSTRUCTED` in the reports;
   * we cannot and do not claim exact reproduction of those numbers.
8. **Unresolved numerical discrepancies** are listed in `docs/VALIDATION_REPORT.md` ("Unresolved items") — at the time of writing: the Fig. 5 halting crossing (≈1.77 computed vs ≈1.9 printed for `n_s=16`) and one borderline Table III cell (26.84 vs 26.9). We did not alter tolerances to hide them.
9. **Halting billing** is not specified in the paper; (S5) all-billed is assumed. Elastic billing follows the paper (down spot unbilled); the S5-consistent variant is also reported.
10. **Integer effects.** Analytical optima are continuous; integer spot counts change the optimum (e.g. continuous `n*≈21.5`, integer `(n, n_s)` optimum `n=20, n_s=18` in the reconstructed Theorem 2 setting). Both are reported.
11. **Monte Carlo power.** The robustness test with 250–400 runs per point (paper-sized) can rule out large, not small, departures from concavity; we report standard errors and use an explicit significance rule (interior point > 2 SE below both endpoints). `full` mode uses 400 runs/point; larger runs are possible via `configs/robustness.yaml`.
12. **Computational limits.** Everything runs on one CPU core. `full` mode takes tens of minutes; no GPU is used or needed. Weibull/burst simulations use a Python event loop (transparent, not fast).
13. **Statistical protocol.** MC checks use `|z| < 3.5` per comparison (about 60 comparisons); a genuine mismatch must be investigated, not absorbed by widening this threshold.
14. **Exploratory extensions** (sensitivity, ablation, fine break-even scan, market-wide correlated stress) are *new* analyses, not claims of the paper, and are labelled as such.


In [ ]:
%%writefile docs/MATHEMATICAL_MODEL.md
# Mathematical model, as implemented

Source of truth: `GNN.docx`. **Important:** the supplied `.docx` is a PDF-style export — equations are *not* Word math objects and
are largely illegible in text extraction. Every equation below was therefore reconstructed from the surrounding text and then
verified *independently* (numerical differentiation, brute-force minimisation, 60-digit `mpmath`, Monte Carlo, and — where the paper prints
numbers — agreement with those numbers). Where a printed number could pin down an interpretation, that is noted.

## 1. Units and variables

| Symbol | Meaning | Code | Unit |
|---|---|---|---|
| `N, n, z` | epochs; workers; time-proportional charge (worker-equivalents) | `Job.N`, `n`, `z` | – |
| `a, θ, A, κ0, γ, σ, α` | Eq. (1) parameters | `PerfParams` | seconds (epoch time) |
| `π, ρ` | on-demand price/worker-hour; spot price ratio | `pi`, `rho` | – |
| `λ_w, λ_a = n s λ_w` | per-worker / aggregate revocation rate | `lam_w`, `lam_a` | per **hour** |
| `R, δ` | stall after revocation; checkpoint write time | `R`, `delta` | **hours** inside functions (YAML gives seconds) |
| `u = λ_a R, v = λ_a δ` | stall load; checkpoint-to-failure ratio | `u`, `v` | – |
| `m(s) = 1-(1-ρ)s` | price-weighted bill fraction | `m_of_s` | – |
| `Λ(u)` | `1+u` (S4a, `stall="fixed"`) or `e^u` (S4b, `"overlap"`) | `Lambda` | – |
| `Ψ(v) = -1/W0(-e^{-1-v})`, `x* = 1+W0(-e^{-1-v})` | checkpoint optimum | `psi`, `x_star` | – |
| `T_N = N T(n)` | failure-free job duration | `Job.T_N` | hours |

## 2. Equations and where they live

* **Eq. (1)–(2), Prop. 1** — `models/performance.py`.
  `T(n)=a n^-θ + (A/n) r(n) + σ(1-1/n) + αn`, `r=κ(1-1/n)`, `κ=min{1,κ0 n^γ}`; `J=π(n+z)T`; `ε=-d ln T/d ln n`.
  The closed form `ε = θp_c + (1-γ)p_f - (p_f+p_s)/(n-1) - p_l` is implemented in the singularity-free form
  `ε·T = θc + (1-γ_eff)f - Aκ/n² - σ/n - αn` (valid at n=1). `κ` has a kink at the cap (`γ_eff=0` there); derivative tests avoid the kink.
  Prop. 1(c): `n* = √(zb/σ)`, `J(n)-J* = πσ(n-n*)²/n`, `J* = π(√b+√(zσ))²`, `b=a-σ` (θ=1, A=α=0).
  Prop. 1(d): `n_D=(a-σ)/(D/N-σ)`.
* **Eq. (3)–(4), Lemma 1** — `models/spot.py`, `models/lambert.py`.
  Segment of exposure `X`: `E[S] = (Λ(u)/λ_a)(e^{λ_a X}-1)`. Overhead per unit work at `x=λ_aτ`: `Ω(x)=Λ(u)(e^{x+v}-1)/x`, minimised at
  `x*` (stationarity `(1-x)e^{x+v}=1`), minimum `Λ(u)Ψ(v)`; `F*=m(s)Λ(u)Ψ(v)`.
  **Numerics:** with `t=-W0(-e^{-1-v})`, `v = t-1-ln t`; with `L=ln Ψ=-ln t`, `h(L)=L-1+e^{-L}=v`. Newton on this equation (monotone, convex) replaces
  direct `W0` evaluation, which cancels catastrophically for small `v` (naive scipy returns NaN at `v=0` and ~1e-5 relative error near `v=1e-12`).
* **Theorem 1** — corner structure from strict concavity of `ln F*` in `s` (`ln m` strictly concave; `ln Λ` concave; `ln Ψ` concave since `L''<0`);
  deadline shares `[0, s_D]` from monotonicity of `ΛΨ` in `s` (`deadlines/feasibility.py`). Break-even `Λ(u*)Ψ(u*δ/R)=1/m` (`breakeven_u`);
  first-order (Young) break-even `1+u+√(2uδ/R)=1/m` solved in `√u` (`breakeven_u_young`). `z>0`: `m_z(s)=(nm(s)+z)/(n+z)` (affine in `s`).
* **Theorem 2** — `optimization/slack.py`. `H(y)=Λ(yR)Ψ(yδ)`, `ℓ=(ln H)'=R(lnΛ)'(yR)+δ(lnΨ)'(yδ)` with `(lnΨ)'=1/(1-e^{-L})`.
  `y_D(n)` solves `T_N(n)H(y)=D`; `C(n)=πD[n-(1-ρ)y_D/λ_w]`; `y_D'=ε/(nℓ)`; `C'(n)=πD[1-(1-ρ)ε/(λ_w nℓ)]`. The **local** test (Eq. 5)
  and the **global** test (`(1-ρ)y_D/λ_w > n-n_D`) are separate functions; the continuous optimum and the integer brute force are separate too.
* **Theorem 3** — `recovery/designs.py`. `Γ_R=Λ(u)Ψ(v)`; `Γ_P=(1+λ_wR)^{n_s}` with `1+u ≤ Γ_P ≤ e^u`; `Γ_E = 1/(1-sq)`, `q=λ_wR/(1+λ_wR)`;
  elastic cost factor `(m - sqρ)/(1-sq)` (down spot unbilled, as in the paper), `m/(1-sq)` if down spot are billed (S5-consistent variant).
  **Cost factor ≠ stretch**: hourly-bill factor × stretch. Billing assumption for **halting** is *not stated in the paper*; we use (S5) (all billed) — flagged in `LIMITATIONS.md`.
* **Prop. 2 (warnings)** — `Ω_W=Ψ((1-φ)v)(1+u+φ v_e)`, `v_e=λ_aδ_e`; `φ=1`: `E[W]=T_N(1+u+v_e)`, `P(W>D)=P(Poisson(λ_aT_N) > (D-T_N)/(δ_e+R))`.
  *Derivation (ours, to confirm the formula):* warned events are Poisson in **running** time with rate `φλ_a`, each adding `δ_e+R` of wall time and no lost work;
  unwarned events (`λ_u=(1-φ)λ_a`) cause rollback. Per segment: `E[S]=(ω-1)[(1+φλ_a(δ_e+R))/λ_u + R]`, `ω=e^{λ_uX}`; minimising over a continuous
  interval gives the paper's formula. `warning_mean_exact_K` gives the exact mean for the integer number of chunks (verified by simulation).
* **Prop. 3 / Appendix A** — `deadlines/completion.py`. Laplace transform Eq. (10); `E[S]=(ω-1)(1/λ+R)`; variance **derived by us** (the printed Eq. (11) is illegible):
  `Var S = Var(G)(m_y+R)² + E[G] Var(Y|Y≤X)`, `G` = failures before success (`E[G]=ω-1`, `Var G=ω(ω-1)`), `Y` truncated exponential. It reproduces the paper's printed
  Table IX standard deviations to 3 decimals and agrees with finite-difference derivatives of Eq. (10). `K=round(T_N/τ*)`, `τ*=x*/λ_a`.
  Chernoff `inf_t e^{-tD}M_{w+δ}(t)^{K-1}M_w(t)` for `0<t<t_c`; Cantelli `P(W>D) ≤ Var/(Var+(D-μ)²)`; 5% budget multiplier `√19=4.3589`.
* **Remark 1 / Table III** — `checkpointing/baselines.py`. Young `x_Y=√(2v)`, first-order overhead `1+√(2v)+u`; Daly (2006) `τ=√(2δM)[1+⅓√(δ/2M)+(1/9)(δ/2M)]-δ` (`δ<2M`, else `M`);
  "per-worker-rate" interval: `x_used = n_s x*(v/n_s)` in aggregate units; regret `=(e^{x+v}-1)/x / Ψ(v) - 1`.

## 3. Simulation semantics (S1)–(S5)

`simulation/rollback.py` implements two independent simulators of the same model (vectorised segment-level, and an explicit WORK/CKPT/STALL/DONE
state machine). Revocations are Poisson in **running time** (ages frozen during stalls); a revocation during a write loses the write and the chunk;
the last chunk has no write (S3); (S4a) stall `R` failure-free, (S4b) stall = busy period of an M/D/∞ queue (`mean (e^u-1)/λ_a`); all workers billed throughout (S5).
`simulation/designs_sim.py`: halting and elastic with alternating-renewal spot workers (up `Exp(λ_w)`, down `R`), stationary start.
`simulation/processes.py`: per-worker exponential/Weibull (mean-matched) renewals and burst-correlated events (group of 4).

## 4. Deviation / interpretation log

| # | Item | What we did | Why |
|---|---|---|---|
| D1 | Illegible equations | reconstructed from text, verified independently | docx has no math objects |
| D2 | Eq. (11) variance | derived and verified (Laplace-derivative + MC + paper's printed Table IX) | printed form illegible |
| D3 | Table IV `m` labels | `0.67→2/3`, `0.33→1/3`, `0.14` as printed | reproduces printed values to printed precision |
| D4 | Table VI rate | exact `-ln(0.8)/730` instead of rounded `3.1e-4` | printed `u` (e.g. 0.0016) matches exact rate only |
| D5 | Prop. 2 run configuration | not given; we use MTBR 3 h, `δ_e=δ` and label RECONSTRUCTED; we also *infer* the MTBR (≈4.52 h) that would give the printed 9.072 h | unavailable data |
| D6 | Theorem 2 brute-force configuration | not given; `a=3600,σ=20`, deadline with `n_D=9`, MTBR ∈ {6,3,1.5} h | cost scales with `D`, so `N` is immaterial |
| D7 | Robustness experiment | reconstructed (see `configs/robustness.yaml`) | per-configuration data unavailable (paper says so) |
| D8 | Halting billing | (S5) all billed | not stated for halting |
| D9 | Elastic/halting simulations | new (paper reports none for Theorem 3) | requested validation |
| D10 | Random-configuration ranges (Thm 1(a), Remark 1(i)) | our own ranges | paper gives counts only |

## 5. Edge cases handled

`v=0` (`Ψ=1`, `x*=0`), `n=1`, `λ_a=0` (`Var W = 0`), `κ` cap, `T_N>D` (infeasible → `None`), `D=T_N` (no slack), `n<n_D` (`y_D=nan`), `n=n_D` (`y_D=0`),
overflow of `e^u`/`Ψ` for very large `u,v` (finite only for `v<~700`), float-fragile `floor` in the φ=1 Poisson tail (fixed with a tolerance — found by a test).


In [ ]:
%%writefile docs/REPRODUCIBILITY.md
# Reproducibility

## Environment
* Python ≥ 3.10 (developed on 3.12), NumPy, SciPy, pandas, Matplotlib, PyYAML, mpmath; dev: pytest, hypothesis, sympy. CPU only.
* `pip install -e ".[dev]"` **or** `conda env create -f environment.yml && conda activate gnn-spot`.
* Environment details of every run are stored in `results/manifests/*.json` (Python/NumPy/SciPy/pandas/Matplotlib versions, platform, git revision, argv, configuration, seeds).

## Commands
```bash
python -m pip install -e ".[dev]"
pytest                                   # unit + property + regression + integration (~minutes)
pytest -m "not slow"                     # (no tests are currently marked slow)
python -m gnn_spot --help
python -m gnn_spot verify --mode quick   # deterministic small run, ~1 min
python -m gnn_spot verify --mode full    # paper-scale trial counts
python scripts/reproduce_paper.py --mode quick   # verify + tables + robustness + extensions + figures + reports
python scripts/reproduce_paper.py --mode full
python scripts/generate_report.py        # regenerate reports from saved results only (no simulation)
python -m gnn_spot figures               # results-only plotting mode (re-plot from saved CSVs / equations)
python -m gnn_spot simulate --n-s 8 --mtbr 3 --trials 40000 --seed 1
```
`GNN_SPOT_RESULTS=/some/dir` redirects all outputs (used by the integration tests so they never overwrite committed results).

## Seeds
* Master seed `20261016` (`configs/paper_verification.yaml`); every Monte Carlo block derives independent streams with `numpy.random.SeedSequence(seed).spawn(k)`.
* Verification checks use fixed integer seeds written in `verification/checks.py` (e.g. Table IX: 11; Prop. 2: 31–43; Theorem 3 simulations: 53, 59). Same seed ⇒ identical output on the same NumPy version (the bit-stream of `Generator` can differ across NumPy major versions).
* Robustness: seed `20261016`, one child stream per (process, MTBR, n_s) configuration.

## Outputs
* `results/raw/` — `verification_{quick,full}.json` (every check: expected, computed, discrepancy, criterion, status), `robustness_{mode}.csv`.
* `results/tables/` — Tables III, IV, V, VI, IX analogues, recovery-design table, robustness summaries, sensitivity, ablation, integer-vs-continuous, break-even scan, market-wide stress.
* `results/figures/` — PNG (300 dpi) + PDF, and `results/figures/data/*.csv` with the plotted data.
* `results/manifests/` — run manifests.
* `docs/VALIDATION_REPORT.md`, `paper/reproduction_report.md` — generated from the saved JSON/CSV, never typed by hand.

## What a result's provenance means
* `REPRODUCTION`: compared with a number printed in the paper (`verification/reference.py` — comparison targets only; never fed to plots).
* `RECONSTRUCTED`: the paper omits the configuration; ours is documented in `configs/` and `MATHEMATICAL_MODEL.md` §4.


In [ ]:
%%writefile docs/VALIDATION_REPORT.md
# Validation report (mode: `full`)

Generated 2026-10-16 08:21 UTC from `results/raw/verification_full.json` (68 s runtime). **59 PASS, 1 FAIL, 0 INCONCLUSIVE of 60 checks.**

Evidence types: ANALYTICAL (algebra re-evaluated), NUMERICAL (independent numerics), MONTE_CARLO (simulation vs theory, |z| criterion), REPRODUCTION (compared with a number printed in the paper), RECONSTRUCTED (paper omits the configuration; ours is labelled).

All statuses are produced by executing the checks in `src/gnn_spot/verification/checks.py`; none is hand-entered.

## Unresolved items (not PASS)

- **Fig5-crossing** (Fig. 5 caption): halting curve crosses above fail-free rollback near u~1.9 (n_s=16, d/R=0.1) — expected 1.9, computed 1.77117; criterion: |u_c - 1.9| < 0.1. 

## All checks

| ID | Paper ref | Claim | Evidence | Expected | Computed | Discrepancy | Criterion | Status |
|---|---|---|---|---|---|---|---|---|
| P1c-root | Prop. 1(c); Table VIII | closed-form n* = sqrt(z b/sigma) equals grid argmin (theta=1) | NUMERICAL | 84.6168 | 84.6166 | 0.000139164 | /n*-argmin/ <= grid step 9.98e-04 | PASS |
| P1c-regret | Prop. 1(c); Table VIII ('12 digits') | J(n)=J* + sigma (n-n*)^2/n and J*=(sqrt b + sqrt(z sigma))^2 over t... | NUMERICAL | 0 | 5.99057e-16 | 5.99057e-16 | max relative error <= 1e-12 | PASS |
| P1c-theta0.9-model1 | Prop. 1(c); Table VIII | root of eps(n)=n/(n+z) equals grid argmin, theta=0.9, variant=model1 | NUMERICAL | 87.3689 | 87.3689 | 0 | within one grid step of a fine local grid | PASS |
| P1c-theta0.9-a_minus_sigma | Prop. 1(c); Table VIII | root of eps(n)=n/(n+z) equals grid argmin, theta=0.9, variant=a_min... | NUMERICAL | 87.3553 | 87.3553 | 1.42109e-14 | within one grid step of a fine local grid | PASS |
| P1c-theta0.7-model1 | Prop. 1(c); Table VIII | root of eps(n)=n/(n+z) equals grid argmin, theta=0.7, variant=model1 | NUMERICAL | 68.6088 | 68.6088 | 1.42109e-14 | within one grid step of a fine local grid | PASS |
| P1c-theta0.7-a_minus_sigma | Prop. 1(c); Table VIII | root of eps(n)=n/(n+z) equals grid argmin, theta=0.7, variant=a_min... | NUMERICAL | 68.6531 | 68.6531 | 1.42109e-14 | within one grid step of a fine local grid | PASS |
| P1b-elasticity | Prop. 1(b) | analytical elasticity = numerical d ln T/d ln n (kappa<1 region and... | NUMERICAL | [0.8522651980887492, 0.8479585315616469, 0.8243902865689988, 0.7783... | [0.8522651981020035, 0.847958531460876, 0.8243902864801899, 0.77830... | 1.00771e-10 | max abs diff <= 1e-7 | PASS |
| P1a-dJ | Prop. 1(a) | J'(n)=pi T [1-(n+z) eps/n] vs central difference | NUMERICAL | [-33680.66136390553, -15748.615008851628, -3147.422277470468, -394.... | [-33680.66136414596, -15748.615009396306, -3147.4222776725837, -394... | 4.9295e-09 | max rel diff <= 1e-6 | PASS |
| P1d-nD | Prop. 1(d) | n_D=(a-sigma)/(D/N-sigma) solves N T(n_D)=D | ANALYTICAL | 9 | 9 | 1.77636e-15 | /n_D - 9/ <= 1e-9 | PASS |
| L1-psi | Lemma 1; Table VIII | closed-form Psi(v)=min_x (e^{x+v}-1)/x vs numerical minimisation, v... | NUMERICAL | 0 | 7.59294e-16 | 7.59294e-16 | max rel err <= 1e-9 | PASS |
| L1-xstar | Lemma 1 | closed-form x* vs numerical argmin (flat minimum -> looser tol) | NUMERICAL | 0 | 1.75024e-07 | 1.75024e-07 | max rel err <= 1e-4 | PASS |
| L1-lambertw | Lemma 1 | stable solver agrees with scipy W0 where W0 is well-conditioned | NUMERICAL | 0 | 8.53104e-15 | 8.53104e-15 | max rel diff <= 1e-9 | PASS |
| L1-props | Lemma 1 | Psi(0)=1, Psi strictly increasing, ln Psi concave, Psi=1+sqrt(2v)+O(v) | NUMERICAL | True | True | {"asym_const": 1.44271955082666} | monotone & concave on grid; (Psi-1-sqrt(2v))/v bounded | PASS |
| L1-stationary | Lemma 1 proof | (1-x) e^{x+v} = 1 at x*; dOmega/dx = 0 | NUMERICAL | 0 | 0 | 0 | /dOmega/dx/ <= 1e-6 | PASS |
| L1-smallv | numerical stability | branch-point regime v in {1e-12,1e-9,1e-6}: ln Psi vs 60-digit mpma... | NUMERICAL | 0 | 1.51521e-16 | 1.51521e-16 | max rel error <= 1e-12 | PASS |
| T1a-concave | Thm 1(a); Table VIII | ln F*(s) strictly concave in s over 3000 random configs (z=0 and z>... | NUMERICAL | 0 | 0 | 0 | 0 configurations with positive second difference > 1e-9 (rel.) | PASS |
| T1a-corner | Thm 1(a); Table VIII | argmin over s in [0,1] at an endpoint in all configs | NUMERICAL | 0 | 0 | 0 | 0 interior minima | PASS |
| T1a-updown | Thm 1(a); Table VIII ('up-then-down') | F*(s) pattern: once decreasing, never increasing again (concave shape) | NUMERICAL | 0 | 0 | 0 | 0 violations | PASS |
| T1b-monotone | Thm 1(b) | F* strictly increasing in n and in lam_w (z=0) | NUMERICAL | True | True | - | strict increase on grids | PASS |
| T1b-breakeven | Thm 1(b) | root of Lambda(u)Psi(u delta/R)=1/m satisfies F*=1 | NUMERICAL | 0 | 5.21805e-15 | 5.21805e-15 | <=1e-12 | PASS |
| T2-deriv | Thm 2 proof | C'(n)=pi D[1-(1-rho) eps/(lam_w n ell(y_D))] vs finite difference | NUMERICAL | 0 | 6.99455e-09 | 6.99455e-09 | max rel diff <= 1e-5 | PASS |
| T2-yprime | Thm 2 proof | y_D'(n)=eps(n)/(n ell(y_D)) | NUMERICAL | 0.475361 | 0.475361 | 2.24464e-10 | rel diff <= 1e-6 | PASS |
| T2-local | Eq. (5) | (1-rho)eps > lam_w n ell  <=>  C'(n) < 0 on a grid of n | ANALYTICAL | True | True | - | equivalence at all grid points | PASS |
| T2-nD-local | Thm 2 / App. B | C'(n) -> pi D > 0 just above n_D (condition (5) fails there) | NUMERICAL | 11.6049 | 11.6049 | 0 | rel diff <= 1e-3 | PASS |
| T2-brute | Thm 2; Table VIII | deadline-binding formula (integer n, continuous s) vs exhaustive in... | RECONSTRUCTED | formula <= brute force and gap <= one spot worker's bill | [{"mtbr_h": 6.0, "n_D": 9.000000000000002, "n_opt_cont": 13.3030217... | {"relative_gaps": [0.0, 0.007125875246430289, 0.0]} | formula never exceeds the integer optimum (relaxation) and the gap ... | PASS |
| T2-n21 | Thm 2; Table VIII ('optimum n=21 vs n_D=9 at MTBR 3 h') | cost-optimal cluster at MTBR 3 h exceeds deadline minimum; paper re... | RECONSTRUCTED | {"mtbr_h": 3.0, "n_opt": 21, "n_D": 9} | {"n_opt_continuous": 21.5278433071348, "n_int_continuous_s": 22, "n... | 0.527843 | continuous optimum within 1 worker of 21 | PASS |
| R1i | Remark 1(i); Table VIII | Young first-order overhead 1+sqrt(2v)+u < Lambda(u)Psi(v) over 20,0... | NUMERICAL | 0 | 0 | 0 | 0 violations | PASS |
| R1ii | Remark 1(ii) | Young regret exceeds 1%,5%,10% at v ~ 0.17, 0.50, 0.79 | REPRODUCTION | {"0.01": 0.17, "0.05": 0.5, "0.1": 0.79} | {"0.01": 0.1671574422804091, "0.05": 0.4969268612522571, "0.1": 0.7... | {"0.01": -0.00284255771959091, "0.05": -0.0030731387477428984, "0.1... | /v_computed - v_paper/ < 0.006 (2-decimal rounding) | PASS |
| R1iii | Remark 1(iii) | Daly regret <= 0.19% over v in [1e-4,1e2] (numerical) | REPRODUCTION | 0.19 | 0.189664 | -0.000335618 | computed max <= 0.195% | PASS |
| R1ii-unbounded | Remark 1(ii) | Young's regret grows without bound in v (strictly increasing, excee... | NUMERICAL | unbounded | [0.14172629412017868, 6.201475975452791, 36060.350245087386, 629945... | - | strictly increasing along v=1,10,100,500 and last > 1e6 | PASS |
| TabIII | Table III | relative excess cost of per-worker-rate interval, 16 cells | REPRODUCTION | printed values | {"n_s=4,v=0.01": [3.8, 3.795], "n_s=4,v=0.05": [9.3, 9.263], "n_s=4... | 0.00392574 | each cell within 0.4% relative (3-significant-digit rounding) | PASS |
| TabIV | Table IV | break-even stall load u* (first-order, exact S4a, exact S4b), m in ... | REPRODUCTION | printed | {"m=0.667,d/R=0.1": {"paper": [0.27, 0.21, 0.2], "computed": [0.268... | {} | each entry equals the printed value to its printed precision (m=0.6... | PASS |
| TabVI | Table VI | stall load at lam_w=-ln(0.8)/730 /h and n_s needed to reach u*=0.9 | REPRODUCTION | printed | {"R=5min,n_s=64": [0.0016, 0.00163], "R=5min,n_s=256": [0.0065, 0.0... | {} | u equals the printed value to its printed precision (2 s.f.); n_s w... | PASS |
| SecIX-rate | Sec. IX-A | lam_w >= -ln(0.8)/730 h = 3.1e-4 /h; MTBR below ~3300 h | REPRODUCTION | [0.00031, 3300] | [0.00030567609769069826, 3271.436685938922] | - | 3.1e-4 to 2 digits; MTBR within 2% of 3300 | PASS |
| SecIX-stress | Sec. IX-A (third finding) | u~0.5 at n_s=64,R=5 min needs per-worker MTBR ~ 10 h, ~300x more fr... | REPRODUCTION | [10, 300] | [10.666666666666666, 309.375] | - | MTBR within 10% of 10 h; factor within 10% of 300 | PASS |
| TabIX-theory | Table IX (theory columns); Lemma 2 | K, mean and std from Eqs. (10)-(11)-equivalent moments match printe... | REPRODUCTION | {"12": [48, 8.686, 8.684, 0.418, 0.42], "6": [70, 9.57, 9.57, 0.52,... | {"MTBR=12h": [48, 8.6861, 0.4178], "MTBR=6h": [70, 9.5696, 0.5198],... | - | K equal; mean,std within 6e-4 h (3-decimal rounding) | PASS |
| TabIX-sim | Table IX; Table VIII | analytical mean/std vs 40,000 simulated runs per row | MONTE_CARLO | /z/<3.5 | {"MTBR=12h": {"K": 48, "mean_th": 8.6861, "std_th": 0.4178, "mean_s... | - | /z/ < 3.5 for mean and std in all rows | PASS |
| AppA-laplace | Appendix A, Eq. (10) | closed-form segment mean/variance vs finite-difference derivatives ... | NUMERICAL | [1.0023310666011098, 0.43326674643801694] | [1.0023310714524225, 0.433266750365253] | [4.840030252037022e-09, 9.064245222275845e-09] | rel diff <= 1e-5 | PASS |
| AppA-phi0 | Appendix A | transform equals 1 at theta=0 | ANALYTICAL | 1 | 1 | 0 | <=1e-12 | PASS |
| Cantelli-4.36 | Eq. (8) | 5% miss budget -> multiplier sqrt(19) = 4.36 | REPRODUCTION | 4.36 | 4.3589 | -0.00110106 | /mult - 4.36/ < 0.005 | PASS |
| AppA-bounds | Appendix A; Table VIII | Chernoff & Cantelli bounds hold and reproduce the quoted values at ... | REPRODUCTION | {"1.1": [0.058, 0.311, 0.272], "1.25": [0.0002, 0.002, 0.056]} | {"D=1.1mu": {"cantelli": [0.2719, 0.272], "chernoff": [0.3107, 0.31... | - | bounds match printed values to rounding; simulated miss <= both bounds | PASS |
| AppA-K | Appendix A | K=round(T_N/tau*), tau*=x*/lam_a | REPRODUCTION | {"12": 48, "6": 70, "3": 101, "2": 125} | {"12": 48, "6": 70, "3": 101, "2": 125} | - | equal | PASS |
| Fig6-text | Sec. IX-A; Fig. 6 | deadline multiple 'about 1.2-1.8 at u=0.01-0.1 depending on T_N/R, ... | REPRODUCTION | 1.2-1.8 (u=0.01-0.1); >2 near 0.3 | {"u=0.01-0.1 range": [1.147782610214232, 1.7792078771695423], "u~0.... | - | range lies within [1.1,1.85] and the T_N/R=30 curve exceeds 2 near ... | PASS |
| P2-phi0.9-exactK | Prop. 2 (derivation check) | simulation vs EXACT integer-K mean (derived here), phi=0.9 (MTBR3h)... | MONTE_CARLO | 9.82248 | 9.82131 | -0.00117693 | /z/ < 3.5 | PASS |
| P2-phi0.9 | Prop. 2; Table VIII | closed form Psi((1-phi)v)(1+u+phi v_e) (continuous interval) vs exa... | RECONSTRUCTED | 9.83353 | 9.82248 | -0.0110424 | relative difference <= 0.5% (continuous-K approximation; paper repo... | PASS |
| P2-infer | Prop. 2; Table VIII | which per-worker MTBR (8 spot workers, delta_e=delta, T_N=7.40 h) m... | RECONSTRUCTED | 9.072 | 9.072 | {"inferred_mtbr_h": 4.52481645746142} | informational (inference, not a claim about the paper) | PASS |
| P2-phi1-tail | Prop. 2; Table VIII | phi=1: exact Poisson-tail miss probability vs 200,000 runs (D=9.142... | RECONSTRUCTED | 0.505908 | 0.50714 | 0.00123176 | /z/ < 3.5; also equals direct pmf sum to 1e-10 | PASS |
| P2-phi1-mean | Prop. 2 | phi=1: E[W]=T_N(1+u+v_e) | MONTE_CARLO | 9.20889 | 9.20917 | 0.000280653 | /z/ < 3.5 | PASS |
| P2-event-vs-vec | Prop. 2 | explicit state-machine simulator vs vectorised simulator agree (sam... | MONTE_CARLO | 9.82361 | 9.8332 | 0.00958685 | /z/<3.5 | PASS |
| T3-bounds | Thm 3(b),(c); Eq. (6) | 1+u <= Gamma_P <= e^u and Gamma_E <= Gamma_P <= e^u Psi(v) over ran... | NUMERICAL | 0 | 0 | 0 | 0 violations | PASS |
| T3-n-scaling | Thm 3 | at fixed s, lam_w, R: Gamma_E independent of n; Gamma_R, Gamma_P in... | ANALYTICAL | True | True | {"rollback": [1.0336591412299623, 1.1426097777105717, 1.91709254669... | strict ordering | PASS |
| T3-cost-vs-time | Thm 3 / Eq. (7) | elastic cost factor (m - s q rho)/(1 - s q) differs from the stretc... | ANALYTICAL | cost != stretch | [0.6752704996737717, 1.0008323066885283] | 0.325562 | differ by > 1e-6 | PASS |
| T3-sim-halting | Thm 3 (new: paper reports no simulation) | halting: simulated stretch & cost factor vs formulas (n=16, s=0.5, ... | MONTE_CARLO | {"stretch": 1.104486101181412, "cost_factor": 0.7455281182974531} | {"stretch": 1.1047469704645672, "cost_factor": 0.7457042050635829} | {"z_stretch": 1.6927829033616886} | /z/ < 3.5 on stretch; cost-factor relative error <= 1% | PASS |
| T3-sim-elastic | Thm 3 (new: paper reports no simulation) | elastic: simulated stretch & cost factor vs formulas (n=16, s=0.5, ... | MONTE_CARLO | {"stretch": 1.0062111801242237, "cost_factor": 0.6770186335403727} | {"stretch": 1.0062087509320146, "cost_factor": 0.6770178440529048} | {"z_stretch": -0.2739856557476768} | /z/ < 3.5 on stretch; cost-factor relative error <= 1% | PASS |
| Fig5-crossing | Fig. 5 caption | halting curve crosses above fail-free rollback near u~1.9 (n_s=16, ... | REPRODUCTION | 1.9 | 1.77117 | -0.128832 | /u_c - 1.9/ < 0.1 | FAIL |
| S4b-busy | (S4b); Lemma 1 | M/D/inf busy period mean (e^u-1)/lam_a | MONTE_CARLO | 0.0989031 | 0.0988662 | -3.69218e-05 | /z/<3.5 | PASS |
| S4b-mean | Lemma 1 (S4b) | mean completion under overlapping replacements = sum e^u (e^{lam X}... | MONTE_CARLO | 9.70506 | 9.70819 | 0.00312182 | /z/<3.5 | PASS |
| SIM-xval-fixed | Sec. 'Simulation framework' | event-driven state machine vs vectorised segment simulator (fixed s... | MONTE_CARLO | 4.2567 | 4.24477 | -0.0119311 | /z/<3.5 | PASS |
| SIM-xval-overlap | Sec. 'Simulation framework' | event-driven state machine vs vectorised segment simulator (overlap... | MONTE_CARLO | 4.30924 | 4.30875 | -0.00048642 | /z/<3.5 | PASS |
| SIM-xval-perworker | (S1) | per-worker renewal processes (n_s=8) vs aggregate Poisson (aggregat... | MONTE_CARLO | 4.66038 | 4.66531 | 0.00492432 | /z/<3.5 | PASS |

## Notes on individual checks

- `L1-smallv`: naive scipy W0 gives Psi(1e-12)=1.0000014142495635 (true ~1+1.41e-6)
- `T1a-concave`: random ranges are ours (paper does not state them)
- `T2-brute`: Revocation rates, N, D are not stated in the paper; MTBR in {6,3,1.5} h and D with n_D=9 are our reconstruction.
- `T2-n21`: Integer search over (n,n_s) gives n=20 (n_s=18) because spot counts are integral.
- `R1i`: (u,v) sampling ranges are ours
- `TabIII`: Max relative deviation 0.39% (cell n_s=24,v=0.01 shows 26.84 vs printed 26.9).
- `TabIV`: Young's first-order u* > exact u* in every row: True
- `TabVI`: With the rounded rate 3.1e-4 the R=5 min, n_s=64 cell would be 0.00165, i.e. 0.0017 when rounded: the paper evidently used the exact rate.
- `TabIX-theory`: Variance formula derived independently here (paper's Eq. (11) is illegible in the supplied file).
- `Fig6-text`: T_N/R=300 stays slightly below 2 at u=0.3 (about 1.93): 'above 2' holds only for shorter jobs.
- `P2-phi0.9-exactK`: z=-0.43; K=30
- `P2-phi0.9`: K-rounding effect -0.112%. MTBR, delta_e, K are not stated in the paper (assumed MTBR 3 h, delta_e=delta).
- `P2-infer`: Inference only: the paper does not state the rate; do not treat as the authors' configuration.
- `P2-phi1-tail`: z=1.10; paper's 0.4002 vs 0.4003 used an unstated D.
- `P2-event-vs-vec`: z=0.41
- `T3-sim-halting`: Long-run formulas; first-passage of a finite job adds an O(R/T_N) boundary effect, reduced by the stationary start.
- `T3-sim-elastic`: Long-run formulas; first-passage of a finite job adds an O(R/T_N) boundary effect, reduced by the stationary start.
- `S4b-busy`: z=-0.71
- `S4b-mean`: z=0.86
- `SIM-xval-fixed`: z=-1.42
- `SIM-xval-overlap`: z=-0.06
- `SIM-xval-perworker`: z=0.48


In [ ]:
%%writefile environment.yml
name: gnn-spot
channels: [conda-forge]
dependencies:
  - python>=3.10
  - numpy>=1.24
  - scipy>=1.10
  - pandas>=2.0
  - matplotlib>=3.7
  - pyyaml>=6.0
  - mpmath>=1.3
  - pytest>=7.4
  - hypothesis>=6.80
  - sympy>=1.12
  - pip
  - pip:
      - -e .


In [ ]:
%%writefile paper/reproduction_report.md
# Reproduction report

Paper: *When Does Spot Capacity Pay for Synchronous Training: Cost, Deadline and Recovery-Design Regimes* (Katkuri, Ananya JPS, Srividyambika).
Run mode: `full`; generated 2026-10-16. Everything below is computed by this repository; nothing was measured on a real cluster, exactly as in the paper.

## 1. Summary

- **REPRODUCTION**: 12/13 pass
- **ANALYTICAL**: 5/5 pass
- **NUMERICAL**: 26/26 pass
- **MONTE_CARLO**: 11/11 pass
- **RECONSTRUCTED**: 5/5 pass

## 2. Exact reproduction of printed numbers (REPRODUCTION)

Values printed in the paper compared with values recomputed from the equations:

- [PASS] `R1ii` — Young regret exceeds 1%,5%,10% at v ~ 0.17, 0.50, 0.79. Criterion: |v_computed - v_paper| < 0.006 (2-decimal rounding). 
- [PASS] `R1iii` — Daly regret <= 0.19% over v in [1e-4,1e2] (numerical). Criterion: computed max <= 0.195%. 
- [PASS] `TabIII` — relative excess cost of per-worker-rate interval, 16 cells. Criterion: each cell within 0.4% relative (3-significant-digit rounding). Max relative deviation 0.39% (cell n_s=24,v=0.01 shows 26.84 vs printed 26.9).
- [PASS] `TabIV` — break-even stall load u* (first-order, exact S4a, exact S4b), m in {2/3,1/3,0.14}. Criterion: each entry equals the printed value to its printed precision (m=0.67 read as 2/3, 0.33 as 1/3). Young's first-order u* > exact u* in every row: True
- [PASS] `TabVI` — stall load at lam_w=-ln(0.8)/730 /h and n_s needed to reach u*=0.9. Criterion: u equals the printed value to its printed precision (2 s.f.); n_s within 2% ('~'). With the rounded rate 3.1e-4 the R=5 min, n_s=64 cell would be 0.00165, i.e. 0.0017 when rounded: the paper evidently used the exact rate.
- [PASS] `SecIX-rate` — lam_w >= -ln(0.8)/730 h = 3.1e-4 /h; MTBR below ~3300 h. Criterion: 3.1e-4 to 2 digits; MTBR within 2% of 3300. 
- [PASS] `SecIX-stress` — u~0.5 at n_s=64,R=5 min needs per-worker MTBR ~ 10 h, ~300x more frequent than 3300 h. Criterion: MTBR within 10% of 10 h; factor within 10% of 300. 
- [PASS] `TabIX-theory` — K, mean and std from Eqs. (10)-(11)-equivalent moments match printed Table IX. Criterion: K equal; mean,std within 6e-4 h (3-decimal rounding). Variance formula derived independently here (paper's Eq. (11) is illegible in the supplied file).
- [PASS] `Cantelli-4.36` — 5% miss budget -> multiplier sqrt(19) = 4.36. Criterion: |mult - 4.36| < 0.005. 
- [PASS] `AppA-bounds` — Chernoff & Cantelli bounds hold and reproduce the quoted values at MTBR 3 h (D=1.1mu, 1.25mu). Criterion: bounds match printed values to rounding; simulated miss <= both bounds. 
- [PASS] `AppA-K` — K=round(T_N/tau*), tau*=x*/lam_a. Criterion: equal. 
- [PASS] `Fig6-text` — deadline multiple 'about 1.2-1.8 at u=0.01-0.1 depending on T_N/R, above 2 near u=0.3'. Criterion: range lies within [1.1,1.85] and the T_N/R=30 curve exceeds 2 near u=0.3. T_N/R=300 stays slightly below 2 at u=0.3 (about 1.93): 'above 2' holds only for shorter jobs.
- [FAIL] `Fig5-crossing` — halting curve crosses above fail-free rollback near u~1.9 (n_s=16, d/R=0.1). Criterion: |u_c - 1.9| < 0.1. 

## 3. Independent verification of derivations (ANALYTICAL / NUMERICAL)

- [PASS] `P1d-nD` — n_D=(a-sigma)/(D/N-sigma) solves N T(n_D)=D (|n_D - 9| <= 1e-9)
- [PASS] `T2-local` — (1-rho)eps > lam_w n ell  <=>  C'(n) < 0 on a grid of n (equivalence at all grid points)
- [PASS] `AppA-phi0` — transform equals 1 at theta=0 (<=1e-12)
- [PASS] `T3-n-scaling` — at fixed s, lam_w, R: Gamma_E independent of n; Gamma_R, Gamma_P increase with n (strict ordering)
- [PASS] `T3-cost-vs-time` — elastic cost factor (m - s q rho)/(1 - s q) differs from the stretch Gamma_E (billing of down spot workers) (differ by > 1e-6)
- [PASS] `P1c-root` — closed-form n* = sqrt(z b/sigma) equals grid argmin (theta=1) (|n*-argmin| <= grid step 9.98e-04)
- [PASS] `P1c-regret` — J(n)=J* + sigma (n-n*)^2/n and J*=(sqrt b + sqrt(z sigma))^2 over the whole grid (max relative error <= 1e-12)
- [PASS] `P1c-theta0.9-model1` — root of eps(n)=n/(n+z) equals grid argmin, theta=0.9, variant=model1 (within one grid step of a fine local grid)
- [PASS] `P1c-theta0.9-a_minus_sigma` — root of eps(n)=n/(n+z) equals grid argmin, theta=0.9, variant=a_minus_sigma (within one grid step of a fine local grid)
- [PASS] `P1c-theta0.7-model1` — root of eps(n)=n/(n+z) equals grid argmin, theta=0.7, variant=model1 (within one grid step of a fine local grid)
- [PASS] `P1c-theta0.7-a_minus_sigma` — root of eps(n)=n/(n+z) equals grid argmin, theta=0.7, variant=a_minus_sigma (within one grid step of a fine local grid)
- [PASS] `P1b-elasticity` — analytical elasticity = numerical d ln T/d ln n (kappa<1 region and capped) (max abs diff <= 1e-7)
- [PASS] `P1a-dJ` — J'(n)=pi T [1-(n+z) eps/n] vs central difference (max rel diff <= 1e-6)
- [PASS] `L1-psi` — closed-form Psi(v)=min_x (e^{x+v}-1)/x vs numerical minimisation, v in [1e-4,5] (max rel err <= 1e-9)
- [PASS] `L1-xstar` — closed-form x* vs numerical argmin (flat minimum -> looser tol) (max rel err <= 1e-4)
- [PASS] `L1-lambertw` — stable solver agrees with scipy W0 where W0 is well-conditioned (max rel diff <= 1e-9)
- [PASS] `L1-props` — Psi(0)=1, Psi strictly increasing, ln Psi concave, Psi=1+sqrt(2v)+O(v) (monotone & concave on grid; (Psi-1-sqrt(2v))/v bounded)
- [PASS] `L1-stationary` — (1-x) e^{x+v} = 1 at x*; dOmega/dx = 0 (|dOmega/dx| <= 1e-6)
- [PASS] `L1-smallv` — branch-point regime v in {1e-12,1e-9,1e-6}: ln Psi vs 60-digit mpmath reference (max rel error <= 1e-12)
- [PASS] `T1a-concave` — ln F*(s) strictly concave in s over 3000 random configs (z=0 and z>0, both stalls) (0 configurations with positive second difference > 1e-9 (rel.))
- [PASS] `T1a-corner` — argmin over s in [0,1] at an endpoint in all configs (0 interior minima)
- [PASS] `T1a-updown` — F*(s) pattern: once decreasing, never increasing again (concave shape) (0 violations)
- [PASS] `T1b-monotone` — F* strictly increasing in n and in lam_w (z=0) (strict increase on grids)
- [PASS] `T1b-breakeven` — root of Lambda(u)Psi(u delta/R)=1/m satisfies F*=1 (<=1e-12)
- [PASS] `T2-deriv` — C'(n)=pi D[1-(1-rho) eps/(lam_w n ell(y_D))] vs finite difference (max rel diff <= 1e-5)
- [PASS] `T2-yprime` — y_D'(n)=eps(n)/(n ell(y_D)) (rel diff <= 1e-6)
- [PASS] `T2-nD-local` — C'(n) -> pi D > 0 just above n_D (condition (5) fails there) (rel diff <= 1e-3)
- [PASS] `R1i` — Young first-order overhead 1+sqrt(2v)+u < Lambda(u)Psi(v) over 20,000 random (u,v), both stalls (0 violations)
- [PASS] `R1ii-unbounded` — Young's regret grows without bound in v (strictly increasing, exceeds 1e6 by v=500) (strictly increasing along v=1,10,100,500 and last > 1e6)
- [PASS] `AppA-laplace` — closed-form segment mean/variance vs finite-difference derivatives of the Laplace transform (rel diff <= 1e-5)
- [PASS] `T3-bounds` — 1+u <= Gamma_P <= e^u and Gamma_E <= Gamma_P <= e^u Psi(v) over random parameters (0 violations)

## 4. Monte Carlo agreement (MONTE_CARLO)

- [PASS] `TabIX-sim` — analytical mean/std vs 40,000 simulated runs per row (|z| < 3.5 for mean and std in all rows). 
- [PASS] `P2-phi0.9-exactK` — simulation vs EXACT integer-K mean (derived here), phi=0.9 (MTBR3h), 40,000 runs (|z| < 3.5). z=-0.43; K=30
- [PASS] `P2-phi1-mean` — phi=1: E[W]=T_N(1+u+v_e) (|z| < 3.5). 
- [PASS] `P2-event-vs-vec` — explicit state-machine simulator vs vectorised simulator agree (same K), phi=0.9 (|z|<3.5). z=0.41
- [PASS] `T3-sim-halting` — halting: simulated stretch & cost factor vs formulas (n=16, s=0.5, T_N=400.0 h, stationary start, 3000 runs) (|z| < 3.5 on stretch; cost-factor relative error <= 1%). Long-run formulas; first-passage of a finite job adds an O(R/T_N) boundary effect, reduced by the stationary start.
- [PASS] `T3-sim-elastic` — elastic: simulated stretch & cost factor vs formulas (n=16, s=0.5, T_N=400.0 h, stationary start, 3000 runs) (|z| < 3.5 on stretch; cost-factor relative error <= 1%). Long-run formulas; first-passage of a finite job adds an O(R/T_N) boundary effect, reduced by the stationary start.
- [PASS] `S4b-busy` — M/D/inf busy period mean (e^u-1)/lam_a (|z|<3.5). z=-0.71
- [PASS] `S4b-mean` — mean completion under overlapping replacements = sum e^u (e^{lam X}-1)/lam (|z|<3.5). z=0.86
- [PASS] `SIM-xval-fixed` — event-driven state machine vs vectorised segment simulator (fixed stall) (|z|<3.5). z=-1.42
- [PASS] `SIM-xval-overlap` — event-driven state machine vs vectorised segment simulator (overlap stall) (|z|<3.5). z=-0.06
- [PASS] `SIM-xval-perworker` — per-worker renewal processes (n_s=8) vs aggregate Poisson (aggregation principle) (|z|<3.5). z=0.48

## 5. Reconstructed experiments (parameters not in the paper)

- [PASS] `T2-brute` — deadline-binding formula (integer n, continuous s) vs exhaustive integer (n,n_s) search at 3 revocation rates. Revocation rates, N, D are not stated in the paper; MTBR in {6,3,1.5} h and D with n_D=9 are our reconstruction.
- [PASS] `T2-n21` — cost-optimal cluster at MTBR 3 h exceeds deadline minimum; paper reports n=21. Integer search over (n,n_s) gives n=20 (n_s=18) because spot counts are integral.
- [PASS] `P2-phi0.9` — closed form Psi((1-phi)v)(1+u+phi v_e) (continuous interval) vs exact integer-K mean, phi=0.9 (MTBR3h, delta_e=delta). K-rounding effect -0.112%. MTBR, delta_e, K are not stated in the paper (assumed MTBR 3 h, delta_e=delta).
- [PASS] `P2-infer` — which per-worker MTBR (8 spot workers, delta_e=delta, T_N=7.40 h) makes the paper's quoted theory mean 9.072 h?. Inference only: the paper does not state the rate; do not treat as the authors' configuration.
- [PASS] `P2-phi1-tail` — phi=1: exact Poisson-tail miss probability vs 200,000 runs (D=9.142 h, assumed MTBR 3 h). z=1.10; paper's 0.4002 vs 0.4003 used an unstated D.

## 6. Unresolved discrepancies

- **Fig5-crossing**: halting curve crosses above fail-free rollback near u~1.9 (n_s=16, d/R=0.1) — expected 1.9; computed 1.77117. 

## 7. Robustness experiment (Sec. IX-B) — RECONSTRUCTED

- 32 (process, MTBR) series. Raw argmin of cost at an endpoint: 32/32; statistically significant interior minimum (2 SE below both endpoints): 0/32.
- The paper's 32 'configurations' are (process × MTBR) = 4 × 8; we sweep n_s in {0,2,...,16} within each. The n_s grid and the eight MTBRs are our choice.
- Runs per point: `full` (paper: 250-400). With few runs the test has limited power, as the paper itself notes.

Stretch at n_s=16 relative to exponential (paper claims: Weibull 0.6 +11..13% at MTBR 3,12,48 h; Weibull 1.5 −4..6%; burst 1.35 vs 1.98 at 3 h):

| process | MTBR h | stretch | exp. stretch (sim) | exp. theory (ΛΨ) | rel. change % | ±SE % |
|---|---|---|---|---|---|---|
| exponential | 3 | 1.970 | 1.970 | 1.977 | +0.0 | 0.5 |
| exponential | 12 | 1.293 | 1.293 | 1.295 | +0.0 | 0.3 |
| exponential | 48 | 1.105 | 1.105 | 1.108 | +0.0 | 0.3 |
| weibull0.6 | 3 | 2.175 | 1.970 | 1.977 | +10.4 | 0.7 |
| weibull0.6 | 12 | 1.460 | 1.293 | 1.295 | +12.9 | 0.6 |
| weibull0.6 | 48 | 1.229 | 1.105 | 1.108 | +11.2 | 0.5 |
| weibull1.5 | 3 | 1.901 | 1.970 | 1.977 | -3.5 | 0.4 |
| weibull1.5 | 12 | 1.216 | 1.293 | 1.295 | -6.0 | 0.3 |
| weibull1.5 | 48 | 1.062 | 1.105 | 1.108 | -3.9 | 0.2 |
| burst | 3 | 1.344 | 1.970 | 1.977 | -31.8 | 0.2 |
| burst | 12 | 1.129 | 1.293 | 1.295 | -12.7 | 0.2 |
| burst | 48 | 1.055 | 1.105 | 1.108 | -4.5 | 0.2 |

## 8. Extensions beyond the paper (clearly new, not claims of the paper)

### Ablation of modelling choices (base n=32, s=0.5, MTBR 12 h)

| variant                                     |   stretch |   cost_factor |
|:--------------------------------------------|----------:|--------------:|
| baseline: rollback, S4a, optimal interval   |    1.2946 |        0.8739 |
| S4b overlapping replacements                |    1.3021 |        0.8789 |
| rollback S4a, Young interval                |    1.2949 |        0.874  |
| rollback S4a, Daly interval                 |    1.2946 |        0.8739 |
| rollback S4a, interval from per-worker rate |    1.5512 |        1.047  |
| halting (S5 billing)                        |    1.1171 |        0.754  |
| elastic (down spot unbilled)                |    1.0035 |        0.6761 |
| elastic (all billed, S5)                    |    1.0035 |        0.6773 |
| warnings phi=0.9 + rollback                 |    1.1757 |        0.7936 |

### One-at-a-time ±50% sensitivity of F*

| parameter   |   F_low |   F_high |   F_base |   swing |
|:------------|--------:|---------:|---------:|--------:|
| s           |  0.9842 |   0.7218 |   0.8739 |  0.2623 |
| rho         |  0.7606 |   0.9872 |   0.8739 |  0.2266 |
| mtbr_h      |  1.0265 |   0.8209 |   0.8739 |  0.2056 |
| n           |  0.7932 |   0.9507 |   0.8739 |  0.1575 |
| R_s         |  0.8302 |   0.9176 |   0.8739 |  0.0874 |
| delta_s     |  0.8349 |   0.9054 |   0.8739 |  0.0705 |

### Fine-grained simulated scan around the predicted break-even u*

|   u_over_ustar |      u |   K |   F_continuous |   F_exactK |   F_sim |   F_sim_se | predicted_spot_pays   | sim_spot_pays   | sim_significant   |
|---------------:|-------:|----:|---------------:|-----------:|--------:|-----------:|:----------------------|:----------------|:------------------|
|            0.6 | 0.5041 | 170 |         0.7361 |     0.7354 |  0.7352 |     0.0001 | True                  | True            | True              |
|            0.8 | 0.6722 | 200 |         0.8648 |     0.864  |  0.8638 |     0.0002 | True                  | True            | True              |
|            0.9 | 0.7562 | 213 |         0.9316 |     0.9306 |  0.9311 |     0.0002 | True                  | True            | True              |
|            1   | 0.8402 | 227 |         1      |     0.999  |  0.9991 |     0.0002 | True                  | True            | True              |
|            1.1 | 0.9242 | 239 |         1.0701 |     1.0691 |  1.0688 |     0.0002 | False                 | False           | True              |
|            1.2 | 1.0082 | 252 |         1.142  |     1.1408 |  1.1412 |     0.0002 | False                 | False           | True              |
|            1.4 | 1.1763 | 275 |         1.291  |     1.2897 |  1.2897 |     0.0003 | False                 | False           | True              |

### Market-wide correlated reclamation (all spot workers revoked together) — exploratory

|   mtbr_h |   n_s |   stretch |   stretch_sem |   cost_factor |
|---------:|------:|----------:|--------------:|--------------:|
|      1.2 |     0 |    1      |        0      |        1      |
|      1.2 |     4 |    1.207  |        0.0028 |        1.0109 |
|      1.2 |     8 |    1.2066 |        0.0029 |        0.8144 |
|      1.2 |    12 |    1.2037 |        0.0029 |        0.6169 |
|      1.2 |    16 |    1.2046 |        0.0029 |        0.4216 |
|      3   |     0 |    1      |        0      |        1      |
|      3   |     4 |    1.1047 |        0.0023 |        0.9252 |
|      3   |     8 |    1.1054 |        0.0022 |        0.7462 |
|      3   |    12 |    1.1064 |        0.0022 |        0.567  |
|      3   |    16 |    1.1107 |        0.0022 |        0.3888 |
|     12   |     0 |    1      |        0      |        1      |
|     12   |     4 |    1.0445 |        0.0018 |        0.8748 |
|     12   |     8 |    1.0442 |        0.0018 |        0.7048 |
|     12   |    12 |    1.0483 |        0.0018 |        0.5372 |
|     12   |    16 |    1.0438 |        0.0017 |        0.3653 |

### Integer spot counts vs continuous share

|   mtbr_h |   n |   F_continuous_min |   s_cont_argmin |   F_integer_min |   n_s_int_argmin |   rounding_penalty |
|---------:|----:|-------------------:|----------------:|----------------:|-----------------:|-------------------:|
|        3 |   2 |             0.4113 |               1 |          0.4113 |                2 |                 -0 |
|        3 |  12 |             0.6112 |               1 |          0.6112 |               12 |                 -0 |
|        3 |  32 |             1      |               0 |          1      |                0 |                  0 |
|        3 |  64 |             1      |               0 |          1      |                0 |                  0 |
|        6 |   2 |             0.3879 |               1 |          0.3879 |                2 |                  0 |
|        6 |  12 |             0.4929 |               1 |          0.4929 |               12 |                  0 |
|        6 |  32 |             0.6919 |               1 |          0.6919 |               32 |                  0 |
|        6 |  64 |             1      |               0 |          1      |                0 |                  0 |
|       12 |   2 |             0.3742 |               1 |          0.3742 |                2 |                  0 |
|       12 |  12 |             0.4326 |               1 |          0.4326 |               12 |                  0 |
|       12 |  32 |             0.5322 |               1 |          0.5322 |               32 |                  0 |
|       12 |  64 |             0.6919 |               1 |          0.6919 |               64 |                  0 |

## 9. Paper-to-code traceability matrix

| Paper item | Source | Test | Experiment / output |
|---|---|---|---|
| Eq. (1) T(n); Eq. (2) J(n), eps(n) | `models/performance.py` | `tests/unit/test_performance.py` | verify: prop1 (P1b, P1a) |
| Prop. 1(a)-(d) | `models/performance.py` | `tests/unit/test_performance.py` | verify: prop1 (P1c-*, P1d-nD) |
| Eq. (3) lambda_a,u,v,m(s),Lambda(u) | `models/spot.py` | `tests/unit/test_spot.py` | verify: theorem1 |
| Lemma 1 (x*, Psi, Eq. 4) | `models/lambert.py, checkpointing/baselines.py` | `tests/unit/test_lambert.py, tests/property/test_properties.py` | verify: lemma1 (L1-*) |
| Theorem 1(a) corner/log-concavity | `models/spot.py (log_cost_factor)` | `tests/property/test_properties.py` | verify: theorem1 (T1a-*) |
| Theorem 1(b) break-even; Table IV; Fig. 4 | `models/spot.py (breakeven_u*)` | `tests/regression/test_paper_values.py` | verify: table4_6; results/tables/table4_*.csv; fig4 |
| Theorem 1(c) deadline feasible share | `deadlines/feasibility.py` | `tests/unit/test_deadlines.py` | verify: theorem2/theorem3 paths |
| Theorem 2; Eq. (5) | `optimization/slack.py` | `tests/unit/test_slack.py` | verify: theorem2 (T2-*); fig_thm2 |
| Theorem 3 (R/P/E designs, Eq. 6-7) | `recovery/designs.py, simulation/designs_sim.py` | `tests/unit/test_designs.py` | verify: theorem3 (T3-*); Fig. 5; results/tables/table_recovery_designs.csv |
| Prop. 2 (warnings) | `recovery/designs.py, simulation/rollback.py` | `tests/unit/test_designs.py` | verify: prop2 (P2-*) |
| Prop. 3 / Appendix A (Eq. 8, 10, 11), Table IX | `deadlines/completion.py` | `tests/unit/test_completion.py` | verify: appendix_a; table9_completion_moments.csv; Fig. 6 |
| Remark 1 (Young/Daly), Table III | `checkpointing/baselines.py` | `tests/regression/test_paper_values.py` | verify: remark1, table3; fig_baselines |
| Sec. IX-A, Table VI | `experiments/tables.py` | `tests/regression/test_paper_values.py` | verify: table4_6; table6_stall_load.csv |
| Sec. IX-B robustness (Table VII) | `experiments/robustness.py, simulation/processes.py` | `tests/integration/test_pipeline.py` | robustness_*.csv; fig_robustness_reconstructed (RECONSTRUCTED) |
| Fig. 3 regime maps | `plotting/figures.py` | `tests/integration/test_pipeline.py` | fig3_scaleout_regime_map |
| Simulator (S1)-(S5) state machine | `simulation/rollback.py` | `tests/unit/test_simulation.py` | verify: sim_crossval, overlap |

## 10. What is NOT established

- No real-cluster measurement, trace replay, or fitted θ/σ exists in the paper or here; all evidence is analytical or simulation of the same model.
- The original robustness data, Prop. 2/Thm 2 run configurations and simulation scripts are unavailable; those experiments are reconstructed and cannot be claimed as exact reproductions.
- The supplied document's equations are illegible (PDF-style export); equations were reconstructed from surrounding text and then verified independently (see MATHEMATICAL_MODEL.md).



In [ ]:
%%writefile scripts/generate_report.py
#!/usr/bin/env python
"""Regenerate docs/VALIDATION_REPORT.md and paper/reproduction_report.md from saved results (no re-simulation)."""
import argparse, sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from gnn_spot.reporting.report import generate_reports

if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--mode", choices=["quick", "full"], default="full", help="which saved verification run to report")
    generate_reports(ap.parse_args().mode)


In [ ]:
%%writefile scripts/reproduce_paper.py
#!/usr/bin/env python
"""Reproduce the paper's verification, tables, figures and reports.  --mode quick | full"""
import argparse, sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from gnn_spot.experiments.pipeline import reproduce

if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--mode", choices=["quick", "full"], default="quick")
    reproduce(ap.parse_args().mode)


In [ ]:
%%writefile scripts/run_all.py
#!/usr/bin/env python
"""Alias of reproduce_paper.py (kept for the suggested layout)."""
import runpy, sys
from pathlib import Path
runpy.run_path(str(Path(__file__).with_name("reproduce_paper.py")), run_name="__main__")


In [ ]:
%%writefile src/gnn_spot/__init__.py
"""gnn_spot: analytical models, simulators and verification for spot-capacity synchronous training.

Unit convention (used by every function unless a name says otherwise):
  * rates are per HOUR, durations passed to model functions are in HOURS;
  * configuration files give R and delta in SECONDS and convert at load time.
"""
__version__ = "0.1.0"


In [ ]:
%%writefile src/gnn_spot/__main__.py
import sys
from .cli import main
sys.exit(main())


In [ ]:
%%writefile src/gnn_spot/checkpointing/__init__.py



In [ ]:
%%writefile src/gnn_spot/checkpointing/baselines.py
"""Checkpoint-interval baselines: Young (first order), Daly (higher order), per-worker-rate misuse.

All intervals are expressed as x = lambda_a * tau (work interval in units of the mean time
between aggregate revocations).  The overhead per unit work of a chunk of work tau followed by a
checkpoint write delta is  Omega(x) = Lambda(u) (exp(x+v) - 1) / x  (Lemma 1); Lambda cancels
in every regret ratio.
"""
from __future__ import annotations

import numpy as np

from ..models.lambert import psi, x_star
from ..models.spot import Lambda


def omega(x, u, v, stall="fixed"):
    """Overhead per unit work Omega(x) = Lambda(u)(e^{x+v}-1)/x (Lemma 1)."""
    x = np.asarray(x, dtype=float)
    return Lambda(u, stall) * np.expm1(x + v) / x


def reduced_overhead(x, v):
    """Omega / Lambda = (e^{x+v}-1)/x, the part that depends on the interval."""
    x = np.asarray(x, dtype=float)
    return np.expm1(x + v) / x


def young_x(v):
    """Young's first-order interval: tau = sqrt(2 delta M), M = 1/lambda_a  ->  x = sqrt(2 v)."""
    return np.sqrt(2.0 * np.asarray(v, dtype=float))


def daly_x(v):
    """Daly's higher-order estimate (Daly 2006, Eq. 20): for delta < 2M,
    tau = sqrt(2 delta M)[1 + (1/3) sqrt(delta/2M) + (1/9)(delta/2M)] - delta ; else tau = M.
    In units x = tau/M with v = delta/M."""
    v = np.asarray(v, dtype=float)
    x = np.sqrt(2*v) * (1 + np.sqrt(v/2)/3 + v/18) - v
    return np.where(v < 2.0, x, 1.0)


def young_first_order_overhead(u, v):
    """The first-order overhead of Remark 1(i): 1 + sqrt(2v) + u (an approximation, not an achievable value)."""
    return 1.0 + np.sqrt(2.0 * np.asarray(v, dtype=float)) + np.asarray(u, dtype=float)


def regret(x, v):
    """Relative excess overhead of interval x over the optimum: Omega(x)/Omega(x*) - 1."""
    return reduced_overhead(x, v) / psi(v) - 1.0


def young_regret(v):
    return regret(young_x(v), v)


def daly_regret(v):
    return regret(daly_x(v), v)


def per_worker_rate_excess(n_s, v):
    """Relative excess cost of an interval computed from the per-worker rate (Table III).

    The per-worker optimum is x*(v/n_s) in per-worker units; in aggregate units the interval
    actually used is x = n_s * x*(v/n_s)  (tau fixed, lambda_a = n_s lambda_w).  Returns the
    excess (fraction) over the aggregate-rate optimum, may exceed 1.
    """
    n_s = np.asarray(n_s, dtype=float)
    v = np.asarray(v, dtype=float)
    x_used = n_s * x_star(v / n_s)
    return regret(x_used, v)


In [ ]:
%%writefile src/gnn_spot/cli.py
"""Command-line interface:  python -m gnn_spot <command> [options]"""
from __future__ import annotations

import argparse
import sys

from .config import ConfigError


def _build():
    p = argparse.ArgumentParser(prog="gnn_spot", description="Spot capacity for synchronous training: models, simulators, verification.")
    sub = p.add_subparsers(dest="cmd", required=True)
    mode = lambda sp: sp.add_argument("--mode", choices=["quick", "full"], default="quick", help="quick = small deterministic run; full = paper-scale trial counts")
    sp = sub.add_parser("verify", help="run the verification suite"); mode(sp)
    sp.add_argument("--only", nargs="*", help="subset of checks (e.g. prop1 lemma1 theorem2)")
    sp = sub.add_parser("tables", help="regenerate analytical tables"); mode(sp)
    sp = sub.add_parser("robustness", help="reconstructed Sec. IX-B robustness experiment"); mode(sp)
    sp = sub.add_parser("extensions", help="sensitivity, ablation, break-even scan, stress tests"); mode(sp)
    sp = sub.add_parser("figures", help="regenerate figures from saved results/equations (results-only plotting mode)"); mode(sp)
    sp = sub.add_parser("report", help="generate validation/reproduction reports from saved results"); mode(sp)
    sp = sub.add_parser("reproduce", help="everything: verify + tables + robustness + extensions + figures + reports"); mode(sp)
    sp = sub.add_parser("simulate", help="one-off Monte Carlo of the full-rollback model")
    sp.add_argument("--T-N", type=float, default=7.40, help="failure-free duration (h)")
    sp.add_argument("--n-s", type=int, default=8); sp.add_argument("--mtbr", type=float, default=3.0, help="per-worker MTBR (h)")
    sp.add_argument("--R", type=float, default=300.0, help="stall (s)"); sp.add_argument("--delta", type=float, default=30.0, help="checkpoint write (s)")
    sp.add_argument("--trials", type=int, default=40000); sp.add_argument("--seed", type=int, default=1)
    sp.add_argument("--stall", choices=["fixed", "overlap"], default="fixed")
    return p


def main(argv=None):
    args = _build().parse_args(argv)
    try:
        if args.cmd == "verify":
            from .experiments.pipeline import run_verification
            res = run_verification(args.mode, only=args.only)
            return 0 if all(r["status"] == "PASS" for r in res) else 1
        if args.cmd == "tables":
            from .experiments.pipeline import run_tables; run_tables(args.mode)
        elif args.cmd == "robustness":
            from .experiments.pipeline import run_robustness; run_robustness(args.mode)
        elif args.cmd == "extensions":
            from .experiments.pipeline import run_extensions; run_extensions(args.mode)
        elif args.cmd == "figures":
            from .experiments.pipeline import run_figures; run_figures(args.mode)
        elif args.cmd == "report":
            from .reporting.report import generate_reports; generate_reports(args.mode)
        elif args.cmd == "reproduce":
            from .experiments.pipeline import reproduce; reproduce(args.mode)
        elif args.cmd == "simulate":
            from .deadlines.completion import completion_moments
            from .simulation.rollback import simulate_rollback_vectorized
            from .simulation.results import make_rngs, summarize
            import json
            lam = args.n_s / args.mtbr; R = args.R / 3600; d = args.delta / 3600
            m = completion_moments(args.T_N, lam, R, d)
            if args.stall == "overlap":
                from .models.lambert import x_star
            W, info = simulate_rollback_vectorized(args.T_N, lam, R, d, m.K, args.trials, make_rngs(args.seed, 1)[0], stall=args.stall)
            res = summarize(W, vars(args), args.seed, diagnostics={"K": m.K, "mean_revocations": float(info["n_revocations"].mean())})
            out = res.to_dict(); out["analytical_S4a"] = {"mean": m.mean, "std": m.std}
            print(json.dumps(out, indent=2, default=float))
    except ConfigError as e:
        print(f"configuration error: {e}", file=sys.stderr)
        return 2
    except ValueError as e:
        print(f"invalid input: {e}", file=sys.stderr)
        return 2
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile src/gnn_spot/config.py
"""YAML configuration loading with validation."""
from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Dict

import yaml

from .models.performance import PerfParams

ROOT = Path(__file__).resolve().parents[2]
CONFIG_DIR = ROOT / "configs"
RESULTS_DIR = Path(os.environ.get("GNN_SPOT_RESULTS", ROOT / "results"))   # override for tests / scratch runs


class ConfigError(ValueError):
    pass


def load_yaml(name_or_path) -> Dict[str, Any]:
    p = Path(name_or_path)
    if not p.exists():
        p = CONFIG_DIR / str(name_or_path)
    if not p.exists():
        raise ConfigError(f"config file not found: {name_or_path} (looked in {CONFIG_DIR})")
    with open(p, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if not isinstance(cfg, dict):
        raise ConfigError(f"{p} must contain a YAML mapping")
    return cfg


def _need(cfg, keys, where):
    miss = [k for k in keys if k not in cfg]
    if miss:
        raise ConfigError(f"{where}: missing keys {miss}")


def load_baseline(name="baseline.yaml"):
    cfg = load_yaml(name)
    _need(cfg, ["perf", "spot"], name)
    _need(cfg["spot"], ["R_s", "delta_s", "rho", "stall"], f"{name}:spot")
    sp = cfg["spot"]
    if not (0 < sp["rho"] < 1):
        raise ConfigError("spot.rho must be in (0,1)")
    if sp["stall"] not in ("fixed", "overlap"):
        raise ConfigError("spot.stall must be 'fixed' or 'overlap'")
    if sp["R_s"] <= 0 or sp["delta_s"] < 0:
        raise ConfigError("R_s must be >0 and delta_s >= 0")
    perf = PerfParams(**cfg["perf"])
    cfg["_perf"] = perf
    cfg["_R_h"] = sp["R_s"] / 3600.0
    cfg["_delta_h"] = sp["delta_s"] / 3600.0
    return cfg


def load_mode(mode: str, name="paper_verification.yaml"):
    cfg = load_yaml(name)
    if mode not in cfg["modes"]:
        raise ConfigError(f"unknown mode {mode!r}; choose from {list(cfg['modes'])}")
    return cfg["modes"][mode], cfg["seeds"]["master"]


In [ ]:
%%writefile src/gnn_spot/deadlines/__init__.py



In [ ]:
%%writefile src/gnn_spot/deadlines/completion.py
"""Completion-time distribution under full rollback with fixed failure-free stalls (S4a): Appendix A.

A segment of exposure X (work + checkpoint write) at aggregate rate lam, stall R has duration S with
Laplace transform (Eq. 10)
    E[exp(-theta S)] = exp(-(lam+theta) X) / (1 - exp(-theta R) lam (1-exp(-(lam+theta)X))/(lam+theta)).
W = S_1 + ... + S_K: K-1 segments of exposure w+delta and one of exposure w = T_N/K.

The variance formula below is DERIVED here (the manuscript's Eq. (11) is not legible in the supplied
document): with G ~ Geometric failures before success, S = X + sum_{i<=G}(Y_i + R), Y_i truncated
exponential on [0, X].  It is verified independently against numerical differentiation of Eq. (10)
and against Monte Carlo (tests/unit/test_completion.py).
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy.optimize import brentq, minimize_scalar

from ..models.lambert import x_star


# ---------------------------------------------------------------------- single segment
def segment_laplace(theta, X, lam, R):
    """Eq. (10) (theta may be negative inside the convergence region)."""
    a = lam + theta
    g = X if abs(a) < 1e-14 else -np.expm1(-a * X) / a
    return np.exp(-a * X) / (1.0 - np.exp(-theta * R) * lam * g)


def segment_mean(X, lam, R):
    """E[S] = (omega - 1)(1/lam + R), omega = exp(lam X)."""
    return np.expm1(lam * X) * (1.0 / lam + R)


def segment_var(X, lam, R):
    """Var[S] = Var(G)(m_y+R)^2 + E[G] Var(Y | Y<=X), G geometric (failures before success)."""
    w = np.exp(lam * X)
    q = 1.0 / w
    EG = w - 1.0
    VG = w * (w - 1.0)
    my = 1.0 / lam - X * q / (1.0 - q)
    ey2 = (2.0 / lam**2 - np.exp(-lam * X) * (X**2 + 2*X/lam + 2/lam**2)) / (1.0 - q)
    vy = ey2 - my**2
    return VG * (my + R)**2 + EG * vy


# ---------------------------------------------------------------------- whole job
@dataclass(frozen=True)
class CompletionMoments:
    K: int
    w: float
    mean: float
    var: float

    @property
    def std(self):
        return float(np.sqrt(self.var))


def n_chunks(T_N, lam_a, v):
    """K = round(T_N / tau*), tau* = x*/lam_a  (Appendix A); at least 1."""
    return max(1, int(round(T_N * lam_a / float(x_star(v)))))


def completion_moments(T_N, lam_a, R, delta, K=None) -> CompletionMoments:
    """Exact mean/variance of W for K equal chunks (K-1 with checkpoint write, last without)."""
    if lam_a <= 0:
        return CompletionMoments(1, T_N, T_N, 0.0)
    if K is None:
        K = n_chunks(T_N, lam_a, lam_a * delta)
    w = T_N / K
    mean = (K - 1) * segment_mean(w + delta, lam_a, R) + segment_mean(w, lam_a, R)
    var = (K - 1) * segment_var(w + delta, lam_a, R) + segment_var(w, lam_a, R)
    return CompletionMoments(K, w, float(mean), float(var))


# ---------------------------------------------------------------------- tail bounds
def cantelli_bound(D, mean, var):
    """P(W > D) <= var/(var + (D-mean)^2) for D > mean (else 1)."""
    if D <= mean:
        return 1.0
    return var / (var + (D - mean)**2)


def cantelli_multiplier(eta: float) -> float:
    """sqrt((1-eta)/eta): 4.3589 for eta = 5%."""
    return float(np.sqrt((1.0 - eta) / eta))


def cantelli_deadline(mean, var, eta=0.05):
    """Smallest D with Cantelli bound <= eta: mu + sqrt(Var) sqrt((1-eta)/eta)   (Eq. 8)."""
    return mean + np.sqrt(var) * cantelli_multiplier(eta)


def _log_mgf_segment(t, X, lam, R):
    """ln E[exp(t S)] (t>0); +inf outside the convergence region."""
    a = lam - t
    g = X if abs(a) < 1e-14 else -np.expm1(-a * X) / a
    den = 1.0 - np.exp(t * R) * lam * g
    if den <= 0:
        return np.inf
    return -a * X - np.log(den)


def chernoff_tc(X, lam, R):
    """Smallest positive root t_c of exp(tR) lam (1-exp(-(lam-t)X))/(lam-t) = 1 (longest segment)."""
    def f(t):
        a = lam - t
        g = X if abs(a) < 1e-14 else -np.expm1(-a * X) / a
        return np.exp(t * R) * lam * g - 1.0
    # f(0) = exp(0)*lam*(1-e^{-lam X})/lam - 1 = -e^{-lam X} < 0 ; f increases to +inf
    hi = 1e-6
    while f(hi) < 0:
        hi *= 2.0
        if hi > 1e6:
            return np.inf
    return brentq(f, 0.0, hi, xtol=1e-14)


def chernoff_bound(D, T_N, lam_a, R, delta, K=None):
    """inf_t exp(-tD) M_{w+delta}(t)^{K-1} M_w(t) over 0 < t < t_c (valid for D > E[W])."""
    mom = completion_moments(T_N, lam_a, R, delta, K)
    if D <= mom.mean:
        return 1.0
    K, w = mom.K, mom.w
    tc = chernoff_tc(w + delta, lam_a, R)
    def obj(t):
        val = -t * D + (K - 1) * _log_mgf_segment(t, w + delta, lam_a, R) + _log_mgf_segment(t, w, lam_a, R)
        return val if np.isfinite(val) else 1e300
    res = minimize_scalar(obj, bounds=(1e-12, tc * (1 - 1e-9)), method="bounded",
                          options={"xatol": 1e-13})
    return float(min(1.0, np.exp(res.fun)))


In [ ]:
%%writefile src/gnn_spot/deadlines/feasibility.py
"""Mean-deadline feasibility and the corner decision (Theorem 1(a),(c)); Cantelli feasibility (Prop. 3).

Mean-deadline feasibility (T_N * Gamma <= D) is NOT a probabilistic guarantee: with D equal to the mean
a substantial fraction of runs miss it.  `cantelli_feasible_counts` implements the tail-aware rule of Eq. (8).
"""
from __future__ import annotations

from typing import Optional

import numpy as np
from scipy.optimize import brentq

from ..recovery.designs import stretch_of_design, hourly_bill_factor
from ..models.spot import m_of_s
from .completion import completion_moments, cantelli_deadline


def largest_feasible_share(n, T_N, D, design, lam_w, R, delta, stall="fixed") -> Optional[float]:
    """Largest s in [0,1] with T_N * Gamma(n,s) <= D (Gamma increasing in s).

    Returns None if even all-on-demand misses the deadline (T_N > D); returns 1.0 if all-spot is feasible.
    """
    if T_N > D:
        return None
    f = lambda s: T_N * float(stretch_of_design(design, n, s, lam_w, R, delta, stall)) - D
    if f(1.0) <= 0:
        return 1.0
    return float(brentq(f, 0.0, 1.0, xtol=1e-14))


def expected_cost(n, s, T_N, design, lam_w, R, delta, rho, pi=1.0, z=0.0, stall="fixed", bill_down_spot=False):
    """E[cost] = pi (n * bill_factor(s) + z) * T_N * Gamma   (hours x price)."""
    g = stretch_of_design(design, n, s, lam_w, R, delta, stall)
    b = hourly_bill_factor(design, s, rho, lam_w, R, bill_down_spot)
    return pi * (n * b + z) * T_N * g


def corner_decision(n, T_N, D, design, lam_w, R, delta, rho, pi=1.0, z=0.0, stall="fixed"):
    """Theorem 1(c): choose all on-demand or the largest deadline-feasible share.

    Returns a dict with the endpoint shares, costs and the winner; feasible=False if T_N > D.
    """
    sbar = largest_feasible_share(n, T_N, D, design, lam_w, R, delta, stall)
    if sbar is None:
        return {"feasible": False, "s_bar": None, "choice": None}
    c0 = float(expected_cost(n, 0.0, T_N, design, lam_w, R, delta, rho, pi, z, stall))
    c1 = float(expected_cost(n, sbar, T_N, design, lam_w, R, delta, rho, pi, z, stall))
    return {"feasible": True, "s_bar": sbar, "cost_on_demand": c0, "cost_at_s_bar": c1,
            "choice": "spot" if c1 < c0 else "on-demand", "s_opt": sbar if c1 < c0 else 0.0}


def cantelli_feasible_counts(n, T_N, D, lam_w, R, delta, eta=0.05):
    """Integer spot-worker counts n_s in 0..n for which mu + sqrt(Var) sqrt((1-eta)/eta) <= D (S4a, rollback).

    Uses the exact integer number of chunks K (hence need NOT be monotone in n_s, as the paper notes).
    Returns (feasible_counts, rhs_values) where rhs[n_s] is the Eq. (8) right-hand side.
    """
    rhs = np.empty(n + 1)
    rhs[0] = T_N
    for k in range(1, n + 1):
        m = completion_moments(T_N, k * lam_w, R, delta)
        rhs[k] = cantelli_deadline(m.mean, m.var, eta)
    return np.flatnonzero(rhs <= D), rhs


In [ ]:
%%writefile src/gnn_spot/experiments/__init__.py



In [ ]:
%%writefile src/gnn_spot/experiments/analysis.py
"""Sensitivity, ablation, fine-grained break-even exploration and stress tests (extensions beyond the paper)."""
from __future__ import annotations

from typing import Dict

import numpy as np
import pandas as pd

from ..checkpointing import baselines as bl
from ..deadlines.completion import completion_moments, n_chunks
from ..deadlines.feasibility import largest_feasible_share
from ..models import spot
from ..models.lambert import psi, x_star
from ..recovery import designs as des
from ..simulation import rollback as rb
from ..simulation.results import make_rngs


def _eval(n, s, rho, R_h, d_h, mtbr, stall="fixed"):
    lam_w = 1.0 / mtbr
    n_s = n * s
    lam_a, u, v = spot.aggregate(n, s, lam_w, R_h, d_h)
    out = {"u": float(u), "v": float(v), "F_rollback": float(spot.cost_factor(n, s, rho, lam_w, R_h, d_h, stall)),
           "stretch_rollback": float(spot.stretch(u, v, stall)),
           "F_halting": float(des.halting_cost_factor(s, rho, lam_w, R_h, n_s)),
           "F_elastic": float(des.elastic_cost_factor(s, rho, lam_w, R_h))}
    return out


def sensitivity(cfg: Dict) -> pd.DataFrame:
    b, rg = cfg["base"], cfg["ranges"]
    rows = []
    base = dict(n=b["n"], s=b["s"], rho=b["rho"], R_s=b["R_s"], delta_s=b["delta_s"], mtbr_h=b["mtbr_h"])
    def ev(p):
        return _eval(p["n"], p["s"], p["rho"], p["R_s"] / 3600, p["delta_s"] / 3600, p["mtbr_h"], b["stall"])
    for name in ("n", "s", "delta_s", "R_s", "rho", "mtbr_h"):
        for val in rg[name]:
            p = dict(base); p[name] = val
            rows.append({"parameter": name, "value": val, **ev(p)})
    # deadline multiple: largest feasible share (mean deadline) by design
    R_h, d_h = base["R_s"] / 3600, base["delta_s"] / 3600
    for mult in rg["deadline_multiple"]:
        for design in ("rollback", "halting", "elastic"):
            sD = largest_feasible_share(base["n"], 1.0, mult, design, 1 / base["mtbr_h"], R_h, d_h, b["stall"])
            rows.append({"parameter": "deadline_multiple", "value": mult, "design": design, "s_D": sD})
    return pd.DataFrame(rows)


def tornado(cfg: Dict, frac=0.5) -> pd.DataFrame:
    """Change in F*(rollback) when each parameter is scaled by (1 -/+ frac) one at a time."""
    b = cfg["base"]
    base = dict(n=b["n"], s=b["s"], rho=b["rho"], R_s=b["R_s"], delta_s=b["delta_s"], mtbr_h=b["mtbr_h"])
    f = lambda p: _eval(p["n"], min(1.0, p["s"]), p["rho"], p["R_s"] / 3600, p["delta_s"] / 3600, p["mtbr_h"], b["stall"])["F_rollback"]
    f0 = f(base)
    rows = []
    for name in base:
        lo, hi = dict(base), dict(base)
        lo[name] = base[name] * (1 - frac)
        hi[name] = base[name] * (1 + frac)
        if name == "rho":
            hi[name] = min(0.95, hi[name])
        rows.append({"parameter": name, "F_low": f(lo), "F_high": f(hi), "F_base": f0, "swing": abs(f(hi) - f(lo))})
    return pd.DataFrame(rows).sort_values("swing", ascending=False)


def ablation(cfg: Dict) -> pd.DataFrame:
    """Which modelling choices move the cost factor most (all else fixed)?"""
    b = cfg["base"]
    n, s, rho = b["n"], b["s"], b["rho"]
    R_h, d_h, mt = b["R_s"] / 3600, b["delta_s"] / 3600, b["mtbr_h"]
    lam_w = 1 / mt
    n_s = n * s
    lam_a, u, v = spot.aggregate(n, s, lam_w, R_h, d_h)
    m = float(spot.m_of_s(s, rho))
    rows = []
    def add(name, stretch, billing_m=m):
        rows.append({"variant": name, "stretch": float(stretch), "cost_factor": float(billing_m * stretch)})
    add("baseline: rollback, S4a, optimal interval", spot.stretch(u, v, "fixed"))
    add("S4b overlapping replacements", spot.stretch(u, v, "overlap"))
    for nm, xfun in (("Young interval", bl.young_x), ("Daly interval", bl.daly_x)):
        add(f"rollback S4a, {nm}", float(spot.Lambda(u, "fixed")) * float(bl.reduced_overhead(xfun(v), v)))
    xw = n_s * x_star(v / n_s)
    add("rollback S4a, interval from per-worker rate", float(spot.Lambda(u, "fixed")) * float(bl.reduced_overhead(xw, v)))
    add("halting (S5 billing)", des.gamma_halting(lam_w, R_h, n_s))
    add("elastic (down spot unbilled)", des.gamma_elastic(s, lam_w, R_h), float(des.hourly_bill_factor("elastic", s, rho, lam_w, R_h)))
    add("elastic (all billed, S5)", des.gamma_elastic(s, lam_w, R_h))
    add("warnings phi=0.9 + rollback", des.warning_overhead(u, v, 0.9, v))
    return pd.DataFrame(rows)


def integer_vs_continuous(R_h=300/3600, d_h=30/3600, rho=0.35, mtbr_list=(3.0, 6.0, 12.0), n_range=(2, 12, 32, 64)) -> pd.DataFrame:
    """Continuous break-even/optimal shares vs integer spot-worker counts: effect of rounding n_s on F*."""
    rows = []
    for mt in mtbr_list:
        for n in n_range:
            lam_w = 1 / mt
            s_grid = np.linspace(0, 1, 2001)
            F = spot.cost_factor(n, s_grid, rho, lam_w, R_h, d_h)
            ks = np.arange(0, n + 1)
            Fi = spot.cost_factor(n, ks / n, rho, lam_w, R_h, d_h)
            rows.append({"mtbr_h": mt, "n": n, "F_continuous_min": float(F.min()), "s_cont_argmin": float(s_grid[np.argmin(F)]),
                         "F_integer_min": float(Fi.min()), "n_s_int_argmin": int(ks[np.argmin(Fi)]), "rounding_penalty": float(Fi.min() / F.min() - 1)})
    return pd.DataFrame(rows)


def breakeven_scan(n=16, rho=0.35, T_N=8.0, R_s=300.0, d_s=30.0, trials=20000, seed=5, multiples=(0.6, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4)) -> pd.DataFrame:
    """Fine-grained exploration around the predicted break-even u* (all-spot, s=1, n=n_s): compare the
    simulated sign of (F-1) with the prediction; the exact integer-K analytic mean is shown too."""
    R, d = R_s / 3600, d_s / 3600
    m = rho                                  # s = 1 -> m = rho
    ustar = spot.breakeven_u(m, d / R, "fixed")
    rows = []
    for mult, rng in zip(multiples, make_rngs(seed, len(multiples))):
        u = ustar * mult
        lam_a = u / R
        K = n_chunks(T_N, lam_a, lam_a * d)
        W, _ = rb.simulate_rollback_vectorized(T_N, lam_a, R, d, K, trials, rng)
        mom = completion_moments(T_N, lam_a, R, d, K)
        F_sim = m * W.mean() / T_N
        se = m * W.std(ddof=1) / np.sqrt(trials) / T_N
        rows.append({"u_over_ustar": mult, "u": u, "K": K, "F_continuous": float(m * spot.stretch(u, lam_a * d)), "F_exactK": m * mom.mean / T_N, "F_sim": F_sim, "F_sim_se": se,
                     "predicted_spot_pays": bool(m * spot.stretch(u, lam_a * d) < 1), "sim_spot_pays": bool(F_sim < 1), "sim_significant": bool(abs(F_sim - 1) > 3 * se)})
    return pd.DataFrame(rows)


def market_wide_stress(n=16, rho=0.35, T_N=8.0, R_s=300.0, d_s=30.0, runs=200, seed=9, mtbrs=(1.2, 3.0, 12.0)) -> pd.DataFrame:
    """EXPLORATORY (not in the paper): every revocation event takes out ALL spot workers (market-wide reclamation).
    The aggregate event rate then does not grow with n_s, so the stretch is independent of n_s and F(s)=m(s) G,
    linear (hence trivially corner-optimal) -- the cluster-size penalty disappears under full correlation."""
    from ..simulation.processes import make_process
    R, d = R_s / 3600, d_s / 3600
    rows = []
    cfgs = [(mt, ns) for mt in mtbrs for ns in (0, 4, 8, 12, 16)]
    for (mt, ns), rng in zip(cfgs, make_rngs(seed, len(cfgs))):
        if ns == 0:
            W = np.full(runs, T_N); K = 1
        else:
            K = n_chunks(T_N, 1.0 / mt, d / mt)            # interval tuned to the true EVENT rate 1/mt
            W = np.array([rb.simulate_rollback_event_driven(T_N, K, R, d, make_process("burst", ns, mt, rng, group=ns), rng)[0] for _ in range(runs)])
        s = ns / n
        rows.append({"mtbr_h": mt, "n_s": ns, "stretch": W.mean() / T_N, "stretch_sem": W.std(ddof=1) / np.sqrt(runs) / T_N, "cost_factor": float(spot.m_of_s(s, rho)) * W.mean() / T_N})
    return pd.DataFrame(rows)


In [ ]:
%%writefile src/gnn_spot/experiments/manifest.py
"""Run manifests: configuration, seeds, environment, git revision, command line."""
from __future__ import annotations

import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import scipy
import pandas as pd
import matplotlib

from ..config import ROOT, RESULTS_DIR


def git_rev():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return None


def environment():
    return {"python": sys.version.split()[0], "platform": platform.platform(), "numpy": np.__version__, "scipy": scipy.__version__,
            "pandas": pd.__version__, "matplotlib": matplotlib.__version__}


def write_manifest(name, config, seeds=None, extra=None, outdir=None):
    outdir = Path(outdir or RESULTS_DIR / "manifests")
    outdir.mkdir(parents=True, exist_ok=True)
    m = {"name": name, "utc_time": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "git_rev": git_rev(),
         "argv": sys.argv, "environment": environment(), "config": config, "seeds": seeds, "extra": extra or {}}
    p = outdir / f"{name}.json"
    with open(p, "w") as f:
        json.dump(m, f, indent=2, default=str)
    return p


In [ ]:
%%writefile src/gnn_spot/experiments/pipeline.py
"""End-to-end reproduction pipeline: verification, tables, figures, robustness, extensions, manifests."""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ..config import ROOT, RESULTS_DIR, load_yaml, load_mode
from ..verification.checks import run_checks
from . import tables as T, analysis as A, robustness as RB
from .manifest import write_manifest

RES = RESULTS_DIR


def _w(df: pd.DataFrame, name: str, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)
    df.to_csv(outdir / f"{name}.csv", index=False)
    return outdir / f"{name}.csv"


def run_verification(mode="quick", only=None, log=print):
    cfg, seed = load_mode(mode)
    t0 = time.time()
    res = run_checks(mode, only=only, log=log)
    d = [r.to_dict() for r in res]
    (RES / "raw").mkdir(parents=True, exist_ok=True)
    path = RES / "raw" / f"verification_{mode}.json"
    with open(path, "w") as f:
        json.dump({"mode": mode, "seconds": time.time() - t0, "results": d}, f, indent=1, default=str)
    flat = pd.DataFrame([{k: (v if not isinstance(v, (dict, list)) else "<structured>") for k, v in x.items() if k in ("id", "paper_ref", "evidence", "status", "criterion", "claim")} for x in d])
    _w(flat, f"verification_summary_{mode}", RES / "tables")
    write_manifest(f"verification_{mode}", {"mode_config": cfg}, seeds={"master": seed, "note": "per-check seeds are fixed constants in verification/checks.py"})
    n_pass = sum(x["status"] == "PASS" for x in d)
    log(f"[verify] {n_pass}/{len(d)} PASS in {time.time()-t0:.1f}s -> {path}")
    return d


def run_tables(mode="quick", log=print):
    out = RES / "tables"
    ntr = 40_000 if mode == "full" else 8_000
    _w(T.table3(), "table3_per_worker_rate_excess", out)
    _w(T.table4(), "table4_breakeven_u", out)
    _w(T.table6(), "table6_stall_load", out)
    _w(T.table9(ntr=ntr), "table9_completion_moments", out)
    _w(T.table_designs(), "table_recovery_designs", out)
    _w(T.table5_regimes(), "table5_regime_summary", out)
    write_manifest(f"tables_{mode}", {"table9_trials": ntr}, seeds={"table9": 11})
    log("[tables] written")


def run_thm2_extension(log=print):
    from ..optimization import slack
    from ..verification.checks import _thm2_job
    job = _thm2_job()
    R, d, rho = 300 / 3600, 30 / 3600, 0.35
    rows = []
    for mt in (12.0, 6.0, 3.0, 1.5):
        lw = 1 / mt
        for n in np.linspace(job.n_D(), job.n_D() * 6, 120):
            rows.append({"mtbr_h": mt, "n": n, "cost": slack._spot_endpoint_cost(float(n), job, lw, R, d, rho, "fixed"), "cost_no_spot_nD": job.pi * job.D * job.n_D()})
    df = pd.DataFrame(rows)
    _w(df, "thm2_cost_vs_n", RES / "tables")
    return df


def run_robustness(mode="quick", log=print):
    cfg = load_yaml("robustness.yaml")
    runs = {"quick": 60, "full": cfg["runs_per_point"]}[mode]
    df = RB.run_robustness(cfg, runs=runs, log=log)
    (RES / "raw").mkdir(parents=True, exist_ok=True)
    df.to_csv(RES / "raw" / f"robustness_{mode}.csv", index=False)
    an = RB.analyse(df)
    eff = RB.effect_sizes(df)
    _w(eff, f"robustness_effect_sizes_{mode}", RES / "tables")
    _w(pd.DataFrame(an["series"]), f"robustness_corner_{mode}", RES / "tables")
    write_manifest(f"robustness_{mode}", cfg, seeds={"master": cfg["seed"]}, extra={"runs_per_point": runs, "status": "RECONSTRUCTED"})
    log(f"[robustness] {an['n_series']} (process,MTBR) series; significant interior minima: {sum(s['significant_interior_min'] for s in an['series'])}")
    return df, an, eff


def run_extensions(mode="quick", log=print):
    scfg = load_yaml("sensitivity.yaml")
    out = RES / "tables"
    sens = A.sensitivity(scfg)
    _w(sens, "sensitivity_one_at_a_time", out)
    tor = A.tornado(scfg)
    _w(tor, "sensitivity_tornado", out)
    _w(A.ablation(scfg), "ablation_modelling_choices", out)
    _w(A.integer_vs_continuous(), "integer_vs_continuous_share", out)
    trials = 100_000 if mode == "full" else 20_000
    _w(A.breakeven_scan(trials=trials), "breakeven_fine_scan", out)
    runs = 400 if mode == "full" else 100
    _w(A.market_wide_stress(runs=runs), "stress_market_wide_correlated", out)
    write_manifest(f"extensions_{mode}", {"sensitivity": scfg, "breakeven_trials": trials, "stress_runs": runs}, seeds={"breakeven": 5, "stress": 9})
    return tor


def run_figures(mode="quick", log=print, robustness_df=None, tornado=None):
    from ..plotting import figures as F
    paths = [F.fig3(), F.fig4(), F.fig5(), F.fig6(), F.fig_baselines()]
    thm2 = pd.read_csv(RES / "tables" / "thm2_cost_vs_n.csv") if (RES / "tables" / "thm2_cost_vs_n.csv").exists() else run_thm2_extension()
    paths.append(F.fig_thm2(thm2))
    if robustness_df is None:
        p = RES / "raw" / f"robustness_{mode}.csv"
        robustness_df = pd.read_csv(p) if p.exists() else None
    if robustness_df is not None:
        paths.append(F.fig_robustness(robustness_df))
    if tornado is None:
        p = RES / "tables" / "sensitivity_tornado.csv"
        tornado = pd.read_csv(p) if p.exists() else None
    if tornado is not None:
        paths.append(F.fig_sensitivity(tornado))
    log(f"[figures] {len(paths)} figures -> {RES/'figures'}")
    return paths


def reproduce(mode="quick", log=print):
    t0 = time.time()
    run_verification(mode, log=log)
    run_tables(mode, log=log)
    run_thm2_extension(log=log)
    df, an, eff = run_robustness(mode, log=log)
    tor = run_extensions(mode, log=log)
    run_figures(mode, log=log, robustness_df=df, tornado=tor)
    from ..reporting.report import generate_reports
    generate_reports(mode, log=log)
    log(f"[reproduce] mode={mode} finished in {time.time()-t0:.0f}s")


In [ ]:
%%writefile src/gnn_spot/experiments/robustness.py
"""Sec. IX-B robustness experiment: RECONSTRUCTED from the published description.

Original per-configuration data and scripts are unavailable (stated by the authors); every parameter not
given in the text is labelled in configs/robustness.yaml.  Cost factor is F(s) = m(s) E[W]/T_N (S5 billing:
all n workers billed throughout), relative to an all-on-demand cluster.  Checkpoint interval from the exponential
formula at the aggregate rate and NOT re-optimised; ages frozen during stalls; fixed failure-free stall R.
"""
from __future__ import annotations

import time
from typing import Dict, List

import numpy as np
import pandas as pd

from ..deadlines.completion import n_chunks
from ..models.spot import m_of_s, stretch
from ..simulation.processes import make_process
from ..simulation.results import make_rngs
from ..simulation.rollback import simulate_rollback_event_driven


def run_robustness(cfg: Dict, runs=None, seed=None, log=print, processes=None, mtbrs=None, n_s_values=None) -> pd.DataFrame:
    runs = runs or cfg["runs_per_point"]
    seed = cfg["seed"] if seed is None else seed
    procs = processes or cfg["processes"]
    mtbrs = mtbrs or cfg["mtbr_h"]
    nsv = n_s_values or cfg["n_s_values"]
    T_N, d, R = cfg["T_N_h"], cfg["delta_s"] / 3600.0, cfg["R_s"] / 3600.0
    n, rho = cfg["n"], cfg["rho"]
    configs = [(p, mt, ns) for p in procs for mt in mtbrs for ns in nsv]
    rngs = make_rngs(seed, len(configs))
    rows = []
    t0 = time.time()
    for idx, ((proc, mt, ns), rng) in enumerate(zip(configs, rngs)):
        lam_w = 1.0 / mt
        lam_a = ns * lam_w
        s = ns / n
        if ns == 0:
            W = np.full(runs, T_N)
            K = 1
        else:
            v = lam_a * d
            K = n_chunks(T_N, lam_a, v)
            W = np.empty(runs)
            for r in range(runs):
                p = make_process(proc, ns, mt, rng, group=cfg.get("burst_group", 4))
                W[r], _ = simulate_rollback_event_driven(T_N, K, R, d, p, rng)
        mean = W.mean()
        sem = W.std(ddof=1) / np.sqrt(runs) if runs > 1 else 0.0
        m = float(m_of_s(s, rho))
        theory = float(stretch(lam_a * R, lam_a * d, "fixed")) if ns > 0 else 1.0
        rows.append({"process": proc, "mtbr_h": mt, "n_s": ns, "s": s, "K": K, "runs": runs, "stretch": mean / T_N, "stretch_sem": sem / T_N,
                     "stretch_theory_exp": theory, "cost_factor": m * mean / T_N, "cost_factor_sem": m * sem / T_N})
        if (idx + 1) % 40 == 0:
            log(f"[robustness] {idx+1}/{len(configs)} configs ({time.time()-t0:.0f}s)")
    return pd.DataFrame(rows)


def analyse(df: pd.DataFrame) -> Dict:
    """Corner structure and effect sizes per (process, mtbr) series."""
    out = {"series": [], "n_series": 0}
    for (proc, mt), g in df.groupby(["process", "mtbr_h"]):
        g = g.sort_values("n_s")
        cf = g["cost_factor"].values
        se = g["cost_factor_sem"].values
        i = int(np.argmin(cf))
        endpoint_raw = i in (0, len(cf) - 1)
        # significant interior minimum: an interior point lower than BOTH endpoints by > 2 combined SE
        best_int = None
        for j in range(1, len(cf) - 1):
            lo = min(cf[0] - cf[j], cf[-1] - cf[j])
            if lo > 2 * np.sqrt(se[j]**2 + max(se[0], se[-1])**2):
                best_int = j
        lf = np.log(cf)
        d2 = np.diff(lf, 2)
        # noise scale of a second difference of ln F on a uniform grid: sqrt(6)*sem/F average
        noise = np.sqrt(6.0) * np.mean(se / cf)
        out["series"].append({"process": proc, "mtbr_h": mt, "argmin_n_s": int(g["n_s"].values[i]), "endpoint_raw_argmin": bool(endpoint_raw),
                              "significant_interior_min": best_int is not None, "max_second_diff_lnF": float(d2.max()), "second_diff_noise_scale": float(noise),
                              "all_spot_wins": bool(cf[-1] < cf[0])})
    out["n_series"] = len(out["series"])
    return out


def effect_sizes(df: pd.DataFrame) -> pd.DataFrame:
    """Stretch at n_s = max relative to the exponential process (the paper's 'magnitudes shift' claims)."""
    top = df[df["n_s"] == df["n_s"].max()]
    base = top[top["process"] == "exponential"].set_index("mtbr_h")
    rows = []
    for _, r in top.iterrows():
        b = base.loc[r["mtbr_h"]]
        se = np.hypot(r["stretch_sem"], b["stretch_sem"]) / b["stretch"]
        rows.append({"process": r["process"], "mtbr_h": r["mtbr_h"], "stretch": r["stretch"], "stretch_exp": b["stretch"], "stretch_theory_exp": b["stretch_theory_exp"],
                     "rel_change_pct": 100 * (r["stretch"] / b["stretch"] - 1), "rel_change_se_pct": 100 * se * r["stretch"] / b["stretch"]})
    return pd.DataFrame(rows)


In [ ]:
%%writefile src/gnn_spot/experiments/tables.py
"""Analytical tables computed from the equations (never typed in). Paper-printed values appear only as comparison columns."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..checkpointing import baselines as bl
from ..deadlines import completion as comp
from ..models import spot
from ..models.lambert import psi
from ..recovery import designs as des
from ..verification import reference as ref
from ..simulation import rollback as rb
from ..simulation.results import make_rngs, wilson_interval


def table3() -> pd.DataFrame:
    rows = []
    for n_s, row in ref.TABLE_III.items():
        for v, pct in row.items():
            c = 100 * float(bl.per_worker_rate_excess(n_s, v))
            rows.append({"n_s": n_s, "v": v, "excess_pct_computed": c, "excess_pct_paper": pct, "abs_diff_pct": c - pct, "rel_diff": (c - pct) / pct})
    return pd.DataFrame(rows)


def table4() -> pd.DataFrame:
    rows = []
    for (m, dr), (fo, a_, b_) in ref.TABLE_IV.items():
        rows.append({"m": m, "delta_over_R": dr, "u_star_firstorder": spot.breakeven_u_young(m, dr), "u_star_fixed_S4a": spot.breakeven_u(m, dr, "fixed"),
                     "u_star_overlap_S4b": spot.breakeven_u(m, dr, "overlap"), "paper_firstorder": fo, "paper_S4a": a_, "paper_S4b": b_})
    return pd.DataFrame(rows)


def table6(lam_w=None, ustar=0.9) -> pd.DataFrame:
    """Table VI. Default lam_w is the EXACT -ln(0.8)/730 (the printed 3.1e-4 is its rounding; the printed u values match the exact rate)."""
    lam_w = -np.log(0.8) / 730.0 if lam_w is None else lam_w
    rows = []
    for Rm in (5, 15, 30, 60):
        r = {"R_min": Rm}
        for ns in (64, 256, 1024):
            r[f"u_ns{ns}"] = ns * lam_w * Rm / 60
        r["n_s_at_ustar"] = ustar / (lam_w * Rm / 60)
        r["margin_ns64"] = ustar / r["u_ns64"]
        r["margin_ns1024"] = ustar / r["u_ns1024"]
        r["paper_n_s_at_ustar"] = ref.TABLE_VI[Rm]["n_at_ustar"]
        rows.append(r)
    return pd.DataFrame(rows)


def table9(ntr=40_000, seed=11) -> pd.DataFrame:
    T_N, d, R = 7.40, 30 / 3600, 300 / 3600
    rows = []
    for mt, rng in zip((12, 6, 3, 2), make_rngs(seed, 4)):
        lam = 8.0 / mt
        m = comp.completion_moments(T_N, lam, R, d)
        W, _ = rb.simulate_rollback_vectorized(T_N, lam, R, d, m.K, ntr, rng)
        K_p, mu_p, mus_p, sd_p, sds_p = ref.TABLE_IX[mt]
        rows.append({"MTBR_h": mt, "K": m.K, "mean_theory": m.mean, "mean_sim": W.mean(), "std_theory": m.std, "std_sim": W.std(ddof=1), "trials": ntr,
                     "paper_K": K_p, "paper_mean_th": mu_p, "paper_mean_sim": mus_p, "paper_std_th": sd_p, "paper_std_sim": sds_p,
                     "cantelli_5pct_D": comp.cantelli_deadline(m.mean, m.var), "D_over_T_N": comp.cantelli_deadline(m.mean, m.var) / T_N})
    return pd.DataFrame(rows)


def table_designs(n=32, s=0.5, rho=0.35, R_h=300 / 3600, delta_h=30 / 3600) -> pd.DataFrame:
    """Stretch and monetary cost factor of all designs vs stall load u (n_s=n*s fixed, lam_w varied)."""
    n_s = n * s
    rows = []
    for u in (0.01, 0.03, 0.1, 0.3, 1.0):
        lam_w = u / (n_s * R_h)
        lam_a = n_s * lam_w
        v = lam_a * delta_h
        for name, gam, cost in [
            ("rollback_S4a", float(des.gamma_rollback(u, v, "fixed")), float(des.rollback_cost_factor(s, rho, u, v, "fixed"))),
            ("rollback_S4b", float(des.gamma_rollback(u, v, "overlap")), float(des.rollback_cost_factor(s, rho, u, v, "overlap"))),
            ("halting", float(des.gamma_halting(lam_w, R_h, n_s)), float(des.halting_cost_factor(s, rho, lam_w, R_h, n_s))),
            ("elastic_down_unbilled", float(des.gamma_elastic(s, lam_w, R_h)), float(des.elastic_cost_factor(s, rho, lam_w, R_h))),
            ("elastic_all_billed", float(des.gamma_elastic(s, lam_w, R_h)), float(des.elastic_cost_factor(s, rho, lam_w, R_h, bill_down_spot=True))),
            ("rollback+warnings_phi0.9", float(des.warning_overhead(u, v, 0.9, v)), float(spot.m_of_s(s, rho) * des.warning_overhead(u, v, 0.9, v))),
        ]:
            rows.append({"u": u, "design": name, "stretch": gam, "cost_factor": cost, "n": n, "s": s})
    return pd.DataFrame(rows)


def table5_regimes(ex=None) -> pd.DataFrame:
    """Regime summary (Table V): conditions transcribed from the paper; 'example' evaluated live."""
    rows = [
        ("Does an extra worker lower cost?", "eps(n) > n/(n+z)", "Prop. 1"),
        ("Is the deadline minimum optimal?", "n_D >= n*, and no slack-buying (Eq. (5) with a global cost comparison)", "Prop. 1(d), Thm. 2"),
        ("Does spot pay?", "Lambda(u) Psi(v) < 1/m(s)", "Thm. 1(b), Table IV"),
        ("How much spot?", "s = 0 or the largest admissible s", "Thm. 1(a),(c)"),
        ("Does an extra worker buy useful slack?", "(1-rho) eps(n) > lam_w n ell(y_D)  (local)", "Thm. 2"),
        ("Is the cluster-size penalty unavoidable?", "No: Gamma_R, Gamma_P grow with n_s; Gamma_E does not", "Thm. 3"),
        ("Deadline slack?", "D >= mu + 4.36 sqrt(Var) (5% budget)", "Prop. 3"),
    ]
    return pd.DataFrame(rows, columns=["question", "condition", "result"])


def breakeven_curves(ms=(2/3, 1/2, 1/3), n=60) -> pd.DataFrame:
    rows = []
    dr = np.geomspace(0.1, 10.0, n)
    for m in ms:
        for stall in ("fixed", "overlap"):
            for x in dr:
                rows.append({"m": m, "stall": stall, "delta_over_R": x, "u_star": spot.breakeven_u(m, x, stall)})
    return pd.DataFrame(rows)


def deadline_multiple_curves(T_over_R=(30, 100, 300), d_over_R=0.1, eta=0.05, n=40) -> pd.DataFrame:
    """Fig. 6 data: (mu + 4.36 sigma)/T_N and mu/T_N vs u; units R = 1."""
    rows = []
    for TR in T_over_R:
        for u in np.geomspace(1e-2, 0.3, n):
            lam, d, R = u, d_over_R, 1.0
            m = comp.completion_moments(TR, lam, R, d)
            rows.append({"T_over_R": TR, "u": u, "K": m.K, "mean_over_T": m.mean / TR, "cantelli_over_T": comp.cantelli_deadline(m.mean, m.var, eta) / TR})
    return pd.DataFrame(rows)


def fig5_curves(n_s=16, d_over_R=0.1, n=200) -> pd.DataFrame:
    rows = []
    for u in np.geomspace(1e-2, 3.0, n):
        v = d_over_R * u
        lw_R = u / n_s
        rows.append({"u": u, "rollback_fixed": float(des.gamma_rollback(u, v, "fixed")), "rollback_overlap": float(des.gamma_rollback(u, v, "overlap")),
                     "halting": float((1 + lw_R) ** n_s), "elastic_s0.5": float(1 / (1 - 0.5 * lw_R / (1 + lw_R))),
                     "warn_phi0.9": float(des.warning_overhead(u, v, 0.9, v)), "warn_phi1_noperiodic": float(1 + u + v)})
    return pd.DataFrame(rows)


In [ ]:
%%writefile src/gnn_spot/models/__init__.py



In [ ]:
%%writefile src/gnn_spot/models/lambert.py
"""Checkpoint-optimum special functions (Lemma 1).

x* = 1 + W0(-exp(-1-v)),  Psi(v) = -1 / W0(-exp(-1-v)).

Numerical note
--------------
With t = -W0(-exp(-1-v)) in (0,1), t solves  v = t - 1 - ln t.  Writing L = -ln t = ln Psi,
this is  h(L) := L - 1 + exp(-L) = v.  Evaluating W0 directly near its branch point
(-1/e) suffers catastrophic cancellation for small v (the argument -exp(-1-v) is within ~v of
-1/e), so the production path solves h(L) = v by Newton's method (h is convex and increasing,
so Newton started on the right of the root is monotone) with a Taylor series for small L.
`x_star_lambertw` keeps the naive scipy evaluation purely as an independent cross-check.
"""
from __future__ import annotations

import numpy as np
from scipy.special import lambertw

_SERIES_CUTOFF = 1e-2


def _h(L):
    """h(L) = L - 1 + exp(-L), accurate for small L."""
    L = np.asarray(L, dtype=float)
    series = L**2 * (1/2 - L*(1/6 - L*(1/24 - L*(1/120 - L*(1/720 - L/5040)))))
    direct = L + np.expm1(-L)
    return np.where(L < _SERIES_CUTOFF, series, direct)


def log_psi(v):
    """L(v) = ln Psi(v) for v >= 0 (vectorised). Finite Psi requires v < ~700."""
    v_arr = np.asarray(v, dtype=float)
    if not np.all(np.isfinite(v_arr)) or np.any(v_arr < 0):
        raise ValueError("v must be finite and >= 0")
    scalar = v_arr.ndim == 0
    v1 = np.atleast_1d(v_arr)
    out = np.zeros_like(v1)
    pos = v1 > 0
    if np.any(pos):
        vp = v1[pos]
        L = np.where(vp <= 2.0, 2.0*np.sqrt(2.0*vp), vp + 1.0)
        L = np.where(_h(L) < vp, vp + 1.0, L)          # guarantee start right of the root
        for _ in range(200):
            step = (_h(L) - vp) / (-np.expm1(-L))
            L = L - step
            if np.all(np.abs(step) <= 1e-15 * np.abs(L)):
                break
        out[pos] = L
    return float(out[0]) if scalar else out


def psi(v):
    """Optimal checkpoint overhead factor Psi(v) = exp(L(v)); Psi(0)=1."""
    return np.exp(log_psi(v))


def x_star(v):
    """Optimal checkpoint interval in units of the mean time between aggregate revocations."""
    return -np.expm1(-log_psi(v))


def dlog_psi_dv(v):
    """(ln Psi)'(v) = 1/(1 - exp(-L)); diverges as v -> 0+ (Appendix B)."""
    L = np.asarray(log_psi(v))
    with np.errstate(divide="ignore"):
        r = 1.0 / (-np.expm1(-L))
    return float(r) if r.ndim == 0 else r


def x_star_lambertw(v):
    """Naive x* = 1 + W0(-exp(-1-v)) via scipy (cross-check only; cancels badly for tiny v)."""
    return 1.0 + np.real(lambertw(-np.exp(-1.0 - np.asarray(v, dtype=float)), 0))


def psi_lambertw(v):
    """Naive Psi = -1/W0(-exp(-1-v)) via scipy (cross-check only)."""
    return -1.0 / np.real(lambertw(-np.exp(-1.0 - np.asarray(v, dtype=float)), 0))


In [ ]:
%%writefile src/gnn_spot/models/performance.py
"""Synchronous-training performance model: paper Eq. (1)-(2) and Proposition 1.

T(n) = a n^-theta + (A/n) r(n) + sigma (1 - 1/n) + alpha n,   r(n) = kappa(n) (1 - 1/n),
kappa(n) = min{1, kappa0 n^gamma}.     J(n) = pi (n+z) T(n).     eps(n) = -d ln T / d ln n.

Continuous worker counts n are used by the analytical formulas; integer-constrained
counts are handled in `gnn_spot.optimization`.  Epoch times are in whatever unit `a`,
`A`, `sigma`, `alpha` are given (the paper uses seconds).
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Tuple

import numpy as np
from scipy.optimize import brentq


@dataclass(frozen=True)
class PerfParams:
    a: float                  # single-worker compute time
    theta: float = 1.0        # compute-scaling exponent in (0, 1]  (theta > 1 allowed for superlinear tests)
    A: float = 0.0            # time for one worker's link to move the full remote volume
    kappa0: float = 1.0       # cut-fraction scale
    gamma: float = 0.0        # partition-quality exponent in [0, 1)
    sigma: float = 0.0        # all-reduce limit of synchronisation
    alpha: float = 0.0        # per-worker latency

    def __post_init__(self):
        if self.a <= 0:
            raise ValueError("a must be > 0")
        if not (self.theta > 0):
            raise ValueError("theta must be > 0")
        if min(self.A, self.sigma, self.alpha, self.kappa0) < 0:
            raise ValueError("A, sigma, alpha, kappa0 must be >= 0")
        if not (0 <= self.gamma < 1):
            raise ValueError("gamma must be in [0, 1)")


def _check_n(n):
    n = np.asarray(n, dtype=float)
    if np.any(n < 1):
        raise ValueError("n must be >= 1")
    return n


def kappa(n, p: PerfParams):
    n = _check_n(n)
    return np.minimum(1.0, p.kappa0 * n**p.gamma)


def r_cut(n, p: PerfParams):
    n = _check_n(n)
    return kappa(n, p) * (1.0 - 1.0 / n)


def time_terms(n, p: PerfParams) -> Tuple:
    """(compute, fetch, sync, latency) contributions to T(n)."""
    n = _check_n(n)
    c = p.a * n**(-p.theta)
    f = p.A / n * r_cut(n, p)
    s = p.sigma * (1.0 - 1.0 / n)
    l = p.alpha * n
    return c, f, s, l


def T(n, p: PerfParams):
    c, f, s, l = time_terms(n, p)
    return c + f + s + l


def time_shares(n, p: PerfParams):
    """(p_c, p_f, p_s, p_l): shares of T(n); they sum to 1."""
    c, f, s, l = time_terms(n, p)
    tot = c + f + s + l
    return c/tot, f/tot, s/tot, l/tot


def elasticity(n, p: PerfParams):
    """Analytical eps(n) = -d ln T / d ln n (Proposition 1(b)), written without 0/0 at n=1:

    eps*T = theta*c + (1-gamma_eff)*f - A*kappa/n^2 - sigma/n - alpha*n
    which equals theta p_c + (1-gamma) p_f - (p_f+p_s)/(n-1) - p_l.
    gamma_eff = gamma while kappa<1 and 0 once the cut fraction is capped at 1 (kappa is
    non-differentiable at the cap; there the left/right derivatives differ).
    """
    n = _check_n(n)
    c, f, s, l = time_terms(n, p)
    kap = kappa(n, p)
    gam = np.where(p.kappa0 * n**p.gamma < 1.0, p.gamma, 0.0)
    num = p.theta*c + (1.0 - gam)*f - p.A*kap/n**2 - p.sigma/n - l
    return num / (c + f + s + l)


def elasticity_numeric(n, p: PerfParams, h: float = 1e-5):
    """Fourth-order central difference of -d ln T / d ln n (independent of `elasticity`)."""
    n = np.asarray(n, dtype=float)
    x = np.log(n)
    lt = lambda xx: np.log(T(np.exp(xx), p))
    d = (-lt(x + 2*h) + 8*lt(x + h) - 8*lt(x - h) + lt(x - 2*h)) / (12*h)
    return -d


def J(n, p: PerfParams, pi: float = 1.0, z: float = 0.0):
    """Per-epoch worker cost J(n) = pi (n+z) T(n)."""
    return pi * (np.asarray(n, dtype=float) + z) * T(n, p)


def dJ_dn(n, p: PerfParams, pi: float = 1.0, z: float = 0.0):
    """Proposition 1(a): J'(n) = pi T(n) [1 - (n+z) eps(n)/n]."""
    n = np.asarray(n, dtype=float)
    return pi * T(n, p) * (1.0 - (n + z) * elasticity(n, p) / n)


def extra_worker_lowers_cost(n, p: PerfParams, z: float = 0.0):
    """Proposition 1(a): an extra worker lowers cost iff eps(n) > n/(n+z)."""
    n = np.asarray(n, dtype=float)
    return elasticity(n, p) > n / (n + z)


def scale_out_margin(n, p: PerfParams, z: float):
    """eps(n) - n/(n+z); positive means scale-out pays at n."""
    n = np.asarray(n, dtype=float)
    return elasticity(n, p) - n / (n + z)


def cost_optimal_n_root(p: PerfParams, z: float, lo: float = 1.0 + 1e-9, hi: float = 1e7):
    """Continuous cost-minimising n as a root of eps(n) = n/(n+z) (Prop. 1(c), general theta).

    Returns the largest sign change found on a log grid (global minimum of a unimodal J);
    returns None if there is no interior root (J monotone on [lo, hi]).
    """
    g = lambda x: float(elasticity(x, p) - x/(x + z))
    grid = np.geomspace(lo, hi, 4000)
    vals = np.array([g(x) for x in grid])
    idx = np.where(np.sign(vals[:-1]) * np.sign(vals[1:]) < 0)[0]
    if idx.size == 0:
        return None
    # J'(n) < 0 where eps > n/(n+z); the minimiser is where g crosses + -> -
    cross = [i for i in idx if vals[i] > 0 > vals[i + 1]]
    i = cross[-1] if cross else idx[-1]
    return brentq(g, grid[i], grid[i + 1], xtol=1e-12, rtol=1e-14)


# ------------------------------------------------------------------ Proposition 1(c),(d)
def n_star_closed_form(b: float, sigma: float, z: float) -> float:
    """theta=1, A=alpha=0, T = sigma + b/n with b = a - sigma:  n* = sqrt(z b / sigma)."""
    return float(np.sqrt(z * b / sigma))


def j_min_closed_form(b: float, sigma: float, z: float, pi: float = 1.0) -> float:
    """Minimum value of J: pi (sqrt(b) + sqrt(z sigma))^2."""
    return pi * (np.sqrt(b) + np.sqrt(z * sigma))**2


def j_regret_closed_form(n, b: float, sigma: float, z: float, pi: float = 1.0):
    """J(n) - J(n*) = pi sigma (n - n*)^2 / n  (Prop. 1(c))."""
    n = np.asarray(n, dtype=float)
    ns = n_star_closed_form(b, sigma, z)
    return pi * sigma * (n - ns)**2 / n


def deadline_min_workers(a: float, sigma: float, D: float, N: float) -> float:
    """Prop. 1(d), theta=1, A=alpha=0:  n_D = (a - sigma)/(D/N - sigma)  (requires D/N > sigma)."""
    per_epoch = D / N
    if per_epoch <= sigma:
        return float("inf")
    return (a - sigma) / (per_epoch - sigma)


In [ ]:
%%writefile src/gnn_spot/models/spot.py
"""Spot capacity model: aggregate rates, stall semantics, cost factor F*, break-even (Eq. 3-4, Thm 1(b))."""
from __future__ import annotations

import numpy as np
from scipy.optimize import brentq

from .lambert import psi, log_psi, dlog_psi_dv

STALLS = ("fixed", "overlap")   # (S4a) fixed failure-free stall;  (S4b) overlapping replacements


def _check_stall(stall: str):
    if stall not in STALLS:
        raise ValueError(f"stall must be one of {STALLS}, got {stall!r}")


def Lambda(u, stall: str = "fixed"):
    """Lambda(u) = 1+u (S4a) or exp(u) (S4b). Always 1+u <= Lambda <= e^u."""
    _check_stall(stall)
    u = np.asarray(u, dtype=float)
    return 1.0 + u if stall == "fixed" else np.exp(u)


def dlog_Lambda(u, stall: str = "fixed"):
    """(ln Lambda)'(u): 1/(1+u) or 1."""
    _check_stall(stall)
    u = np.asarray(u, dtype=float)
    return 1.0 / (1.0 + u) if stall == "fixed" else np.ones_like(u)


def m_of_s(s, rho: float):
    """Price-weighted fraction of the on-demand bill m(s) = 1 - (1-rho) s."""
    return 1.0 - (1.0 - rho) * np.asarray(s, dtype=float)


def m_z(s, rho: float, n, z: float):
    """m_z(s) = (n m(s) + z)/(n + z) = 1 - (1-rho) n s/(n+z) (Appendix B)."""
    n = np.asarray(n, dtype=float)
    return (n * m_of_s(s, rho) + z) / (n + z)


def aggregate(n, s, lam_w, R, delta):
    """(lambda_a, u, v) = (n s lam_w, lambda_a R, lambda_a delta). R, delta in hours; lam_w per hour."""
    lam_a = np.asarray(n, dtype=float) * np.asarray(s, dtype=float) * lam_w
    return lam_a, lam_a * R, lam_a * delta


def stretch(u, v, stall: str = "fixed"):
    """Full-rollback wall-time stretch at the optimal interval: Lambda(u) Psi(v)."""
    return Lambda(u, stall) * psi(v)


def cost_factor(n, s, rho, lam_w, R, delta, stall="fixed", z: float = 0.0):
    """F*(n,s) = m(s) Lambda(u) Psi(v)  (z=0, Eq. 4);  m_z(s) replaces m(s) for z>0."""
    _, u, v = aggregate(n, s, lam_w, R, delta)
    mm = m_of_s(s, rho) if z == 0 else m_z(s, rho, n, z)
    return mm * stretch(u, v, stall)


def log_cost_factor(n, s, rho, lam_w, R, delta, stall="fixed", z: float = 0.0):
    _, u, v = aggregate(n, s, lam_w, R, delta)
    mm = m_of_s(s, rho) if z == 0 else m_z(s, rho, n, z)
    return np.log(mm) + np.log(Lambda(u, stall)) + log_psi(v)


# ----------------------------------------------------------------------- break-even
def breakeven_u(m: float, delta_over_R: float, stall: str = "fixed", u_max: float = 60.0) -> float:
    """Unique u* with Lambda(u*) Psi(u* delta/R) = 1/m  (Theorem 1(b)); spot pays for u < u*."""
    _check_stall(stall)
    if not (0 < m < 1):
        raise ValueError("m must be in (0,1)")
    target = 1.0 / m
    with np.errstate(over="ignore"):
        f = lambda u: float(Lambda(u, stall) * psi(u * delta_over_R)) - target
        if f(u_max) < 0:
            raise ValueError("break-even u exceeds u_max; increase u_max")
        return brentq(f, 0.0, u_max, xtol=1e-14, rtol=1e-14)


def breakeven_u_young(m: float, delta_over_R: float) -> float:
    """First-order (Young) break-even: 1 + u + sqrt(2 u delta/R) = 1/m, solved in closed form in sqrt(u)."""
    b = np.sqrt(2.0 * delta_over_R)
    c = 1.0 / m - 1.0
    w = (-b + np.sqrt(b*b + 4.0*c)) / 2.0
    return float(w*w)


def breakeven_lambda_w(n, s, m, R, delta, stall="fixed"):
    """Break-even per-worker rate: spot pays iff n s lam_w < lambda*(s)  (here returned per worker)."""
    ustar = breakeven_u(m, delta / R, stall)
    lam_a_star = ustar / R
    return lam_a_star / (np.asarray(n, dtype=float) * np.asarray(s, dtype=float))


# ------------------------------------------------------------------ H(y), ell(y) (Theorem 2)
def H(y, R, delta, stall="fixed"):
    """H(y) = Lambda(yR) Psi(y delta); y is the aggregate revocation rate (per hour)."""
    return Lambda(np.asarray(y, dtype=float) * R, stall) * psi(np.asarray(y, dtype=float) * delta)


def ell(y, R, delta, stall="fixed"):
    """ell(y) = (ln H)'(y) = R (ln Lambda)'(yR) + delta (ln Psi)'(y delta); -> infinity as y -> 0+."""
    y = np.asarray(y, dtype=float)
    return R * dlog_Lambda(y * R, stall) + delta * dlog_psi_dv(y * delta)


In [ ]:
%%writefile src/gnn_spot/optimization/__init__.py



In [ ]:
%%writefile src/gnn_spot/optimization/slack.py
"""Theorem 2: buying deadline slack with extra workers (continuous formulas and integer brute force)."""
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Optional

import numpy as np
from scipy.optimize import brentq

from ..models.spot import H, ell, Lambda
from ..models.lambert import psi
from ..models.performance import PerfParams, T, elasticity
from ..recovery.designs import stretch_of_design


@dataclass(frozen=True)
class Job:
    """Job description in HOURS: epoch time T(n) comes from `perf` (seconds) and N epochs."""
    perf: PerfParams
    N: float
    D: float          # deadline, hours
    pi: float = 1.0
    z: float = 0.0

    def T_N(self, n):
        """Failure-free job duration in hours at (possibly non-integer) n."""
        return self.N * T(n, self.perf) / 3600.0

    def n_D(self, hi: float = 1e6) -> float:
        """Smallest continuous n with T_N(n) <= D (root of T_N(n) = D) assuming T_N decreasing near it."""
        f = lambda n: self.T_N(n) - self.D
        grid = np.geomspace(1.0 + 1e-9, hi, 6000)
        vals = np.array([f(x) for x in grid])
        ok = np.flatnonzero(vals <= 0)
        if ok.size == 0:
            return float("inf")
        i = ok[0]
        if i == 0:
            return float(grid[0])
        return float(brentq(f, grid[i - 1], grid[i], xtol=1e-13))


def y_D(n, job: Job, lam_w_unused=None, R=None, delta=None, stall="fixed", y_hi: float = 1e3):
    """Aggregate revocation rate y (per hour) at which T_N(n) H(y) = D; 0 if n == n_D; nan if n < n_D."""
    target = job.D / float(job.T_N(n))
    if target < 1.0 - 1e-12:
        return float("nan")
    if target <= 1.0 + 1e-15:
        return 0.0
    with np.errstate(over="ignore"):
        f = lambda y: float(H(y, R, delta, stall)) - target
        hi = 1e-6
        while f(hi) < 0:
            hi *= 2
            if hi > y_hi:
                raise ValueError("y_D bracket failure")
        return float(brentq(f, 0.0, hi, xtol=1e-15, rtol=1e-14))


def binding_cost(n, job: Job, lam_w, R, delta, rho, stall="fixed"):
    """C(n) = pi D [ n - (1-rho) y_D(n)/lam_w ]   (z = 0, deadline binding, while s_D < 1)."""
    y = y_D(n, job, None, R, delta, stall)
    return job.pi * job.D * (n - (1.0 - rho) * y / lam_w)


def dC_dn_analytic(n, job: Job, lam_w, R, delta, rho, stall="fixed"):
    """C'(n) = pi D [1 - (1-rho) eps(n) / (lam_w n ell(y_D(n)))]."""
    y = y_D(n, job, None, R, delta, stall)
    return job.pi * job.D * (1.0 - (1.0 - rho) * float(elasticity(n, job.perf)) / (lam_w * n * float(ell(y, R, delta, stall))))


def marginal_condition(n, job: Job, lam_w, R, delta, rho, stall="fixed"):
    """Local test, Eq. (5): (1-rho) eps(n) > lam_w n ell(y_D(n)).  Returns (lhs, rhs, holds)."""
    y = y_D(n, job, None, R, delta, stall)
    lhs = (1.0 - rho) * float(elasticity(n, job.perf))
    rhs = lam_w * n * float(ell(y, R, delta, stall))
    return lhs, rhs, bool(lhs > rhs)


def global_condition(n, job: Job, lam_w, R, delta, rho, stall="fixed"):
    """Global comparison: C(n) < pi D n_D  iff (1-rho) y_D(n)/lam_w > n - n_D."""
    nD = job.n_D()
    y = y_D(n, job, None, R, delta, stall)
    return bool((1.0 - rho) * y / lam_w > n - nD)


def share_at_deadline(n, job: Job, lam_w, R, delta, stall="fixed"):
    """s_D(n) = y_D(n)/(n lam_w)."""
    return y_D(n, job, None, R, delta, stall) / (n * lam_w)


def _spot_endpoint_cost(n, job: Job, lam_w, R, delta, rho, stall):
    """Cost of the best spot corner at continuous n >= n_D (z = 0).

    If s_D(n) <= 1 the deadline binds and C = pi D [n - (1-rho) y_D/lam_w]  (Thm 2);
    otherwise all-spot is feasible with slack to spare:  C = pi n rho T_N(n) H(n lam_w).
    """
    y = y_D(n, job, None, R, delta, stall)
    if y / (n * lam_w) <= 1.0:
        return job.pi * job.D * (n - (1.0 - rho) * y / lam_w)
    return job.pi * (n * rho) * job.T_N(n) * float(H(n * lam_w, R, delta, stall))


def continuous_optimum(job: Job, lam_w, R, delta, rho, stall="fixed", n_max_factor=12.0, n_pts=600):
    """Minimise the spot-endpoint cost over continuous n >= n_D (coarse log-grid, then local refinement).

    Returns n_D, the continuous minimiser, its cost, and the no-spot benchmark pi D n_D.
    """
    from scipy.optimize import minimize_scalar
    nD = job.n_D()
    ns = nD * np.geomspace(1.0, n_max_factor, n_pts)
    costs = np.array([_spot_endpoint_cost(n, job, lam_w, R, delta, rho, stall) for n in ns])
    i = int(np.argmin(costs))
    lo, hi = ns[max(i - 1, 0)], ns[min(i + 1, n_pts - 1)]
    if hi > lo:
        res = minimize_scalar(lambda n: _spot_endpoint_cost(n, job, lam_w, R, delta, rho, stall),
                              bounds=(lo, hi), method="bounded", options={"xatol": 1e-9})
        n_opt, c_opt = float(res.x), float(res.fun)
    else:
        n_opt, c_opt = float(ns[i]), float(costs[i])
    return {"n_D": nD, "n_opt": n_opt, "cost_opt": c_opt, "cost_no_spot": job.pi * job.D * nD,
            "grid": (ns, costs)}


def brute_force_integer(job: Job, lam_w, R, delta, rho, design="rollback", stall="fixed",
                        n_min=1, n_max=200, mean_deadline=True, bill_down_spot=False):
    """Exhaustive search over integer (n, n_s) minimising expected cost s.t. T_N(n) Gamma <= D.

    Returns the best (n, n_s) with cost, plus the best no-spot cluster (n_s = 0).
    """
    from ..recovery.designs import hourly_bill_factor
    best = None
    best0 = None
    for n in range(n_min, n_max + 1):
        TN = float(job.T_N(n))
        if TN > job.D:
            continue
        n_s = np.arange(0, n + 1)
        s = n_s / n
        g = stretch_of_design(design, n, s, lam_w, R, delta, stall)
        feas = TN * g <= job.D
        b = hourly_bill_factor(design, s, rho, lam_w, R, bill_down_spot)
        cost = job.pi * (n * b + job.z) * TN * g
        cost = np.where(feas, cost, np.inf)
        k = int(np.argmin(cost))
        if best is None or cost[k] < best["cost"]:
            best = {"n": n, "n_s": int(n_s[k]), "s": float(s[k]), "cost": float(cost[k]), "T_N": TN,
                    "stretch": float(g[k])}
        if best0 is None or cost[0] < best0["cost"]:
            best0 = {"n": n, "n_s": 0, "s": 0.0, "cost": float(cost[0]), "T_N": TN, "stretch": 1.0}
    return {"best": best, "best_no_spot": best0}


In [ ]:
%%writefile src/gnn_spot/plotting/__init__.py



In [ ]:
%%writefile src/gnn_spot/plotting/figures.py
"""Figures regenerated from computed data (no hand-typed values). Each figure -> PNG (300 dpi) + PDF + data CSV."""
from __future__ import annotations

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ..config import ROOT, RESULTS_DIR
from ..experiments import tables as T

plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 100, "savefig.bbox": "tight"})
FIGDIR = RESULTS_DIR / "figures"


def _save(fig, name, data: pd.DataFrame, outdir=None):
    outdir = Path(outdir or FIGDIR)
    (outdir / "data").mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / f"{name}.png", dpi=300)
    fig.savefig(outdir / f"{name}.pdf")
    data.to_csv(outdir / "data" / f"{name}.csv", index=False)
    plt.close(fig)
    return outdir / f"{name}.png"


def fig3(outdir=None):
    """Fig. 3: (a) lever map, kappa'/kappa=1/3; (b) scale-out beyond the deadline minimum (large-n limit, p_l=0)."""
    f = np.linspace(0, 1, 201)
    rows = []
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
    for xi in (2, 4, 16):
        g = (1 - 1 / 3) * f / (xi - 1)
        ax[0].plot(f, np.minimum(g, 1), label=f"$\\xi={xi}$")
        rows += [{"panel": "a", "series": f"xi={xi}", "x": a, "y": b} for a, b in zip(f, np.minimum(g, 1))]
    ax[0].axvspan(0.44, 0.83, color="0.85", label="reported fetch share 44-83% [9]")
    ax[0].set(xlabel="fetch share of cost, $f$", ylabel="synchronisation share of cost, $g$", xlim=(0, 1), ylim=(0, 1),
              title="(a) Lever map, $\\kappa'/\\kappa=1/3$\npartitioning wins below each line")
    ax[0].legend(fontsize=7)
    pf = np.linspace(0, 1, 201)
    for zn, col in zip((0.1, 0.3, 1.0), ("C0", "C1", "C2")):
        thr = (zn) / (1 + zn)                      # theta=1, gamma=0: p_s < z/(n+z)
        ax[1].plot(pf, np.minimum(thr, 1 - pf), color=col, label=f"$z/n={zn}$")
        rows += [{"panel": "b", "series": f"z/n={zn}, theta=1, gamma=0", "x": a, "y": b} for a, b in zip(pf, np.minimum(thr, 1 - pf))]
        th, ga = 0.9, 0.5                          # p_s < [theta - (theta-(1-gamma)) p_f - 1/(1+z/n)] / theta
        y = (th - (th - (1 - ga)) * pf - 1 / (1 + zn)) / th
        y = np.where(y > 0, np.minimum(y, 1 - pf), np.nan)
        if np.any(np.isfinite(y)):
            ax[1].plot(pf, y, color=col, ls="--")
            rows += [{"panel": "b", "series": f"z/n={zn}, theta=0.9, gamma=0.5", "x": a, "y": b} for a, b in zip(pf, y)]
    ax[1].plot(pf, 1 - pf, "k", lw=0.8)
    ax[1].set(xlabel="fetch share of time, $p_f$", ylabel="synchronisation share, $p_s$", xlim=(0, 1), ylim=(0, 1),
              title="(b) Scale-out pays below each line\nsolid $\\theta=1,\\gamma=0$; dashed $\\theta=0.9,\\gamma=0.5$")
    ax[1].legend(fontsize=7)
    return _save(fig, "fig3_scaleout_regime_map", pd.DataFrame(rows), outdir)


def fig4(outdir=None):
    df = T.breakeven_curves()
    fig, ax = plt.subplots(figsize=(5.2, 4))
    for (m, stall), g in df.groupby(["m", "stall"]):
        col = {2/3: "C0", 1/2: "C1", 1/3: "C2"}[m]
        ax.plot(g["u_star"], g["delta_over_R"], color=col, ls="-" if stall == "fixed" else "--",
                label=f"m={m:.2f} " + ("(S4a)" if stall == "fixed" else "(S4b)"))
    u = np.geomspace(1e-2, 5, 100)
    ax.plot(u, 0.5 / u, "k:", lw=1, label="$v=1/2$")
    ax.set(xscale="log", yscale="log", xlim=(1e-2, 2), ylim=(0.1, 10), xlabel="stall load $u=n_s\\lambda_w R$", ylabel="checkpoint-to-stall ratio $\\delta/R$",
           title="Break-even boundary (spot pays lower-left)")
    ax.legend(fontsize=7)
    return _save(fig, "fig4_breakeven_boundary", df, outdir)


def fig5(outdir=None):
    df = T.fig5_curves()
    fig, ax = plt.subplots(figsize=(5.6, 4))
    sty = {"rollback_fixed": ("full rollback, fail-free stall", "C0", "-"), "rollback_overlap": ("full rollback, overlapping repl.", "C0", "--"),
           "halting": ("replica-preserving halting", "C3", "-"), "elastic_s0.5": ("elastic continuation", "C2", "-"),
           "warn_phi0.9": ("rollback + warnings, $\\phi=0.9$", "C1", "-"), "warn_phi1_noperiodic": ("warnings $\\phi=1$, no periodic ckpt", "C1", "--")}
    for k, (lab, c, ls) in sty.items():
        ax.plot(df["u"], df[k], color=c, ls=ls, label=lab)
    ax.set(xscale="log", yscale="log", xlabel="stall load $u=n_s\\lambda_w R$", ylabel="wall-time stretch $E[W]/T_N$",
           title="$n_s=16,\\ s=0.5,\\ \\delta/R=0.1,\\ \\delta_e=\\delta$")
    ax.legend(fontsize=6.5)
    return _save(fig, "fig5_recovery_design_stretch", df, outdir)


def fig6(outdir=None):
    df = T.deadline_multiple_curves()
    fig, ax = plt.subplots(figsize=(5.4, 4))
    for (TR), g in df.groupby("T_over_R"):
        l, = ax.plot(g["u"], g["cantelli_over_T"], label=f"$T_N/R={TR}$")
        ax.plot(g["u"], g["mean_over_T"], ls="--", color=l.get_color())
    ax.set(xscale="log", xlabel="stall load $u$", ylabel="deadline multiple $D/T_N$", title="5% miss budget (Cantelli, solid) vs mean (dashed), $\\delta/R=0.1$")
    ax.legend(fontsize=7)
    return _save(fig, "fig6_deadline_multiple", df, outdir)


def fig_thm2(job_rows: pd.DataFrame, outdir=None):
    """Cost vs cluster size (continuous formula) and integer optimum at several MTBR (extension figure)."""
    fig, ax = plt.subplots(figsize=(5.4, 4))
    for mt, g in job_rows.groupby("mtbr_h"):
        ax.plot(g["n"], g["cost"] / g["cost_no_spot_nD"], label=f"MTBR {mt:g} h")
    ax.axhline(1.0, color="k", lw=0.8)
    ax.set(xlabel="cluster size $n$", ylabel="$C(n)/(\\pi D n_D)$", title="Theorem 2: deadline-binding cost vs cluster size")
    ax.legend(fontsize=7)
    return _save(fig, "fig_thm2_cost_vs_n", job_rows, outdir)


def fig_robustness(df: pd.DataFrame, outdir=None, mtbrs=(0.5, 3.0, 12.0)):
    fig, axes = plt.subplots(1, len(mtbrs), figsize=(4 * len(mtbrs), 3.6), sharey=False)
    for ax, mt in zip(np.atleast_1d(axes), mtbrs):
        for proc, g in df[df["mtbr_h"] == mt].groupby("process"):
            g = g.sort_values("s")
            ax.errorbar(g["s"], g["cost_factor"], yerr=1.96 * g["cost_factor_sem"], marker="o", ms=3, capsize=2, label=proc)
        ax.axhline(1.0, color="k", lw=0.7)
        ax.set(xlabel="spot share $s$", ylabel="cost factor vs all on-demand", title=f"MTBR {mt:g} h", yscale="log")
    np.atleast_1d(axes)[0].legend(fontsize=7)
    fig.suptitle("Robustness (RECONSTRUCTED simulation): corner structure of cost in s", y=1.02)
    return _save(fig, "fig_robustness_reconstructed", df, outdir)


def fig_sensitivity(tor: pd.DataFrame, outdir=None):
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    t = tor.sort_values("swing")
    y = np.arange(len(t))
    ax.barh(y, t["F_high"] - t["F_base"], left=t["F_base"], alpha=0.7, label="+50%")
    ax.barh(y, t["F_low"] - t["F_base"], left=t["F_base"], alpha=0.7, label="-50%")
    ax.axvline(t["F_base"].iloc[0], color="k", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(t["parameter"])
    ax.set(xlabel="cost factor $F^*$ (rollback, S4a)", title="One-at-a-time sensitivity (base: n=32, s=0.5, MTBR 12 h)")
    ax.legend(fontsize=7)
    return _save(fig, "fig_sensitivity_tornado", tor, outdir)


def fig_baselines(outdir=None):
    from ..checkpointing import baselines as bl
    v = np.geomspace(1e-4, 1e2, 400)
    df = pd.DataFrame({"v": v, "young_regret_pct": 100 * bl.young_regret(v), "daly_regret_pct": 100 * bl.daly_regret(v)})
    fig, ax = plt.subplots(figsize=(5, 3.6))
    ax.plot(v, df["young_regret_pct"], label="Young")
    ax.plot(v, df["daly_regret_pct"], label="Daly")
    for r in (1, 5, 10):
        ax.axhline(r, color="0.6", lw=0.6, ls=":")
    ax.set(xscale="log", yscale="log", ylim=(1e-5, 1e4), xlabel="$v=\\lambda_a\\delta$", ylabel="regret over optimal interval (%)", title="Checkpoint-interval baselines")
    ax.legend()
    return _save(fig, "fig_baselines_regret", df, outdir)


In [ ]:
%%writefile src/gnn_spot/recovery/__init__.py



In [ ]:
%%writefile src/gnn_spot/recovery/designs.py
"""Recovery designs (Theorem 3) and preemption warnings (Proposition 2).

Billing assumptions are explicit parameters:
  * full rollback and replica-preserving halting follow (S5): all n workers billed throughout;
  * elastic continuation follows the paper's Eq. (7): down spot workers are NOT billed;
    `elastic_cost_factor(..., bill_down_spot=True)` gives the (S5)-consistent variant for comparison;
  * with warnings, all workers are billed throughout (S5).
Monetary cost factor = hourly-bill factor x wall-time stretch; it is NOT a ratio of completion times.
"""
from __future__ import annotations

import numpy as np
from scipy.stats import poisson

from ..models.lambert import psi
from ..models.spot import Lambda, m_of_s

DESIGNS = ("rollback", "halting", "elastic")


# ----------------------------------------------------------------- stretches (wall time / T_N)
def gamma_rollback(u, v, stall="fixed"):
    """Gamma_R = Lambda(u) Psi(v)."""
    return Lambda(u, stall) * psi(v)


def gamma_halting(lam_w, R, n_s):
    """Gamma_P = (1 + lam_w R)^{n_s} (finite-source semantics)."""
    return (1.0 + lam_w * R) ** np.asarray(n_s, dtype=float)


def down_fraction(lam_w, R):
    """q = lam_w R / (1 + lam_w R): long-run fraction of time one spot worker is down."""
    return lam_w * R / (1.0 + lam_w * R)


def gamma_elastic(s, lam_w, R):
    """Gamma_E = 1/(1 - s q) (ratio of means; fixed global batch, theta = 1)."""
    return 1.0 / (1.0 - np.asarray(s, dtype=float) * down_fraction(lam_w, R))


# ------------------------------------------------------------- billing / monetary cost factors
def rollback_cost_factor(s, rho, u, v, stall="fixed"):
    return m_of_s(s, rho) * gamma_rollback(u, v, stall)


def halting_cost_factor(s, rho, lam_w, R, n_s):
    return m_of_s(s, rho) * gamma_halting(lam_w, R, n_s)


def elastic_cost_factor(s, rho, lam_w, R, bill_down_spot=False):
    """(m - s q rho)/(1 - s q) (down spot unbilled, Eq. 7); or m/(1 - s q) if down spot are billed."""
    q = down_fraction(lam_w, R)
    s = np.asarray(s, dtype=float)
    m = m_of_s(s, rho)
    return (m if bill_down_spot else (m - s * q * rho)) / (1.0 - s * q)


def hourly_bill_factor(design, s, rho, lam_w, R, bill_down_spot=False):
    """Expected hourly bill relative to an all-on-demand cluster of the same size."""
    s = np.asarray(s, dtype=float)
    if design == "elastic" and not bill_down_spot:
        return m_of_s(s, rho) - s * down_fraction(lam_w, R) * rho
    return m_of_s(s, rho)


def stretch_of_design(design, n, s, lam_w, R, delta, stall="fixed"):
    """Wall-time stretch of a design at cluster size n, spot share s (n_s = n s)."""
    n_s = np.asarray(n, dtype=float) * np.asarray(s, dtype=float)
    lam_a = n_s * lam_w
    if design == "rollback":
        return gamma_rollback(lam_a * R, lam_a * delta, stall)
    if design == "halting":
        return gamma_halting(lam_w, R, n_s)
    if design == "elastic":
        return gamma_elastic(s, lam_w, R)
    raise ValueError(f"unknown design {design!r}")


def ordering_holds(s, lam_w, R, n_s, delta, tol=1e-12):
    """Theorem 3(c): Gamma_E <= Gamma_P <= e^u Psi(v) (u = n_s lam_w R, v = n_s lam_w delta)."""
    u = n_s * lam_w * R
    v = n_s * lam_w * delta
    gE, gP = gamma_elastic(s, lam_w, R), gamma_halting(lam_w, R, n_s)
    top = np.exp(u) * psi(v)
    return bool((gE <= gP + tol) and (gP <= top + tol))


# ----------------------------------------------------------------------- warnings (Prop. 2)
def warning_overhead(u, v, phi, v_e):
    """Omega_W = Psi((1-phi) v)(1 + u + phi v_e), v_e = lambda_a delta_e  (fixed failure-free stall)."""
    return psi((1.0 - phi) * v) * (1.0 + u + phi * v_e)


def warning_full_usable_mean(T_N, u, v_e):
    """phi = 1: no periodic checkpoint, E[W] = T_N (1 + u + v_e)."""
    return T_N * (1.0 + u + v_e)


def warning_full_usable_miss_prob(lam_a, T_N, D, delta_e, R):
    """phi = 1: P(W > D) = P(Poisson(lam_a T_N) > (D - T_N)/(delta_e + R))  (exact Poisson tail)."""
    k = (D - T_N) / (delta_e + R)
    # P(N > k) = sf(floor(k)).  The quotient is often mathematically an integer, where float error
    # (e.g. 9.999999999999998) would make floor() off by one, so add a relative tolerance.
    k_floor = np.floor(k * (1.0 + 1e-12) + 1e-12)
    return float(poisson.sf(k_floor, lam_a * T_N))


def warning_mean_exact_K(T_N, lam_a, phi, R, delta, delta_e, K):
    """Exact E[W] for K equal chunks with warnings (phi<1), derived here (not printed in the paper).

    Warned events (rate phi*lam_a in RUNNING time) add (delta_e+R) each; unwarned events (rate
    lam_u=(1-phi)*lam_a) force rollback + stall R.  Per segment of exposure X, with omega=exp(lam_u X):
        E[S] = (omega-1) [ (1 + phi*lam_a*(delta_e+R))/lam_u + R ].
    Minimising over a continuous interval gives the paper's Psi((1-phi)v)(1+u+phi*v_e) (Prop. 2).
    """
    lam_u = (1.0 - phi) * lam_a
    if lam_u <= 0:
        raise ValueError("phi must be < 1 for the periodic-checkpoint formula")
    w = T_N / K
    f = lambda X: np.expm1(lam_u * X) * ((1.0 + phi * lam_a * (delta_e + R)) / lam_u + R)
    return (K - 1) * f(w + delta) + f(w)


In [ ]:
%%writefile src/gnn_spot/reporting/__init__.py



In [ ]:
%%writefile src/gnn_spot/reporting/report.py
"""Generate docs/VALIDATION_REPORT.md and paper/reproduction_report.md from SAVED results."""
from __future__ import annotations

import json
import time
from pathlib import Path

import pandas as pd

from ..config import ROOT, RESULTS_DIR

RES = RESULTS_DIR

TRACE = [  # (paper item, source, test, experiment/output)
    ("Eq. (1) T(n); Eq. (2) J(n), eps(n)", "models/performance.py", "tests/unit/test_performance.py", "verify: prop1 (P1b, P1a)"),
    ("Prop. 1(a)-(d)", "models/performance.py", "tests/unit/test_performance.py", "verify: prop1 (P1c-*, P1d-nD)"),
    ("Eq. (3) lambda_a,u,v,m(s),Lambda(u)", "models/spot.py", "tests/unit/test_spot.py", "verify: theorem1"),
    ("Lemma 1 (x*, Psi, Eq. 4)", "models/lambert.py, checkpointing/baselines.py", "tests/unit/test_lambert.py, tests/property/test_properties.py", "verify: lemma1 (L1-*)"),
    ("Theorem 1(a) corner/log-concavity", "models/spot.py (log_cost_factor)", "tests/property/test_properties.py", "verify: theorem1 (T1a-*)"),
    ("Theorem 1(b) break-even; Table IV; Fig. 4", "models/spot.py (breakeven_u*)", "tests/regression/test_paper_values.py", "verify: table4_6; results/tables/table4_*.csv; fig4"),
    ("Theorem 1(c) deadline feasible share", "deadlines/feasibility.py", "tests/unit/test_deadlines.py", "verify: theorem2/theorem3 paths"),
    ("Theorem 2; Eq. (5)", "optimization/slack.py", "tests/unit/test_slack.py", "verify: theorem2 (T2-*); fig_thm2"),
    ("Theorem 3 (R/P/E designs, Eq. 6-7)", "recovery/designs.py, simulation/designs_sim.py", "tests/unit/test_designs.py", "verify: theorem3 (T3-*); Fig. 5; results/tables/table_recovery_designs.csv"),
    ("Prop. 2 (warnings)", "recovery/designs.py, simulation/rollback.py", "tests/unit/test_designs.py", "verify: prop2 (P2-*)"),
    ("Prop. 3 / Appendix A (Eq. 8, 10, 11), Table IX", "deadlines/completion.py", "tests/unit/test_completion.py", "verify: appendix_a; table9_completion_moments.csv; Fig. 6"),
    ("Remark 1 (Young/Daly), Table III", "checkpointing/baselines.py", "tests/regression/test_paper_values.py", "verify: remark1, table3; fig_baselines"),
    ("Sec. IX-A, Table VI", "experiments/tables.py", "tests/regression/test_paper_values.py", "verify: table4_6; table6_stall_load.csv"),
    ("Sec. IX-B robustness (Table VII)", "experiments/robustness.py, simulation/processes.py", "tests/integration/test_pipeline.py", "robustness_*.csv; fig_robustness_reconstructed (RECONSTRUCTED)"),
    ("Fig. 3 regime maps", "plotting/figures.py", "tests/integration/test_pipeline.py", "fig3_scaleout_regime_map"),
    ("Simulator (S1)-(S5) state machine", "simulation/rollback.py", "tests/unit/test_simulation.py", "verify: sim_crossval, overlap"),
]


def _fmt(x, maxlen=110):
    if x is None:
        return "-"
    if isinstance(x, float):
        return f"{x:.6g}"
    s = json.dumps(x, default=str) if isinstance(x, (dict, list)) else str(x)
    s = s.replace("|", "/")
    return s if len(s) <= maxlen else s[:maxlen - 3] + "..."


def _load(mode):
    p = RES / "raw" / f"verification_{mode}.json"
    if not p.exists():
        raise FileNotFoundError(f"{p} not found: run `python -m gnn_spot verify --mode {mode}` first")
    return json.load(open(p))


def validation_report(mode) -> str:
    d = _load(mode)
    rs = d["results"]
    n = len(rs)
    cnt = {s: sum(r["status"] == s for r in rs) for s in ("PASS", "FAIL", "INCONCLUSIVE")}
    L = [f"# Validation report (mode: `{mode}`)", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} from `results/raw/verification_{mode}.json` "
         f"({d['seconds']:.0f} s runtime). **{cnt['PASS']} PASS, {cnt['FAIL']} FAIL, {cnt['INCONCLUSIVE']} INCONCLUSIVE of {n} checks.**", "",
         "Evidence types: ANALYTICAL (algebra re-evaluated), NUMERICAL (independent numerics), MONTE_CARLO (simulation vs theory, |z| criterion), "
         "REPRODUCTION (compared with a number printed in the paper), RECONSTRUCTED (paper omits the configuration; ours is labelled).", "",
         "All statuses are produced by executing the checks in `src/gnn_spot/verification/checks.py`; none is hand-entered.", ""]
    fails = [r for r in rs if r["status"] != "PASS"]
    if fails:
        L += ["## Unresolved items (not PASS)", ""]
        for r in fails:
            L += [f"- **{r['id']}** ({r['paper_ref']}): {r['claim']} — expected {_fmt(r['expected'])}, computed {_fmt(r['computed'])}; criterion: {r['criterion']}. {r['notes']}"]
        L.append("")
    L += ["## All checks", "", "| ID | Paper ref | Claim | Evidence | Expected | Computed | Discrepancy | Criterion | Status |", "|---|---|---|---|---|---|---|---|---|"]
    for r in rs:
        L.append("| " + " | ".join(_fmt(x, 70) for x in (r["id"], r["paper_ref"], r["claim"], r["evidence"], r["expected"], r["computed"], r["discrepancy"], r["criterion"], r["status"])) + " |")
    L.append("")
    L += ["## Notes on individual checks", ""]
    for r in rs:
        if r["notes"]:
            L.append(f"- `{r['id']}`: {r['notes']}")
    return "\n".join(L) + "\n"


def _csv(name):
    p = RES / "tables" / name
    return pd.read_csv(p) if p.exists() else None


def reproduction_report(mode) -> str:
    d = _load(mode)
    rs = d["results"]
    by = lambda ev, st=None: [r for r in rs if r["evidence"] == ev and (st is None or r["status"] == st)]
    L = ["# Reproduction report", "",
         f"Paper: *When Does Spot Capacity Pay for Synchronous Training: Cost, Deadline and Recovery-Design Regimes* (Katkuri, Ananya JPS, Srividyambika).",
         f"Run mode: `{mode}`; generated {time.strftime('%Y-%m-%d', time.gmtime())}. Everything below is computed by this repository; nothing was measured on a real cluster, "
         "exactly as in the paper.", "",
         "## 1. Summary", ""]
    for ev in ("REPRODUCTION", "ANALYTICAL", "NUMERICAL", "MONTE_CARLO", "RECONSTRUCTED"):
        sub = by(ev)
        L.append(f"- **{ev}**: {sum(r['status']=='PASS' for r in sub)}/{len(sub)} pass")
    L += ["", "## 2. Exact reproduction of printed numbers (REPRODUCTION)", "",
          "Values printed in the paper compared with values recomputed from the equations:", ""]
    for r in by("REPRODUCTION"):
        L.append(f"- [{r['status']}] `{r['id']}` — {r['claim']}. Criterion: {r['criterion']}. {r['notes']}")
    L += ["", "## 3. Independent verification of derivations (ANALYTICAL / NUMERICAL)", ""]
    for r in by("ANALYTICAL") + by("NUMERICAL"):
        L.append(f"- [{r['status']}] `{r['id']}` — {r['claim']} ({r['criterion']})")
    L += ["", "## 4. Monte Carlo agreement (MONTE_CARLO)", ""]
    for r in by("MONTE_CARLO"):
        L.append(f"- [{r['status']}] `{r['id']}` — {r['claim']} ({r['criterion']}). {r['notes']}")
    L += ["", "## 5. Reconstructed experiments (parameters not in the paper)", ""]
    for r in by("RECONSTRUCTED"):
        L.append(f"- [{r['status']}] `{r['id']}` — {r['claim']}. {r['notes']}")
    fails = [r for r in rs if r["status"] != "PASS"]
    L += ["", "## 6. Unresolved discrepancies", ""]
    if not fails:
        L.append("None among the executed checks.")
    for r in fails:
        L.append(f"- **{r['id']}**: {r['claim']} — expected {_fmt(r['expected'])}; computed {_fmt(r['computed'])}. {r['notes']}")
    rob = _csv(f"robustness_corner_{mode}.csv")
    eff = _csv(f"robustness_effect_sizes_{mode}.csv")
    L += ["", "## 7. Robustness experiment (Sec. IX-B) — RECONSTRUCTED", ""]
    if rob is not None:
        L += [f"- {len(rob)} (process, MTBR) series. Raw argmin of cost at an endpoint: {int(rob.endpoint_raw_argmin.sum())}/{len(rob)}; "
              f"statistically significant interior minimum (2 SE below both endpoints): {int(rob.significant_interior_min.sum())}/{len(rob)}.",
              "- The paper's 32 'configurations' are (process × MTBR) = 4 × 8; we sweep n_s in {0,2,...,16} within each. The n_s grid and the eight MTBRs are our choice.",
              "- Runs per point: `%s` (paper: 250-400). With few runs the test has limited power, as the paper itself notes." % mode]
    if eff is not None:
        L += ["", "Stretch at n_s=16 relative to exponential (paper claims: Weibull 0.6 +11..13% at MTBR 3,12,48 h; Weibull 1.5 −4..6%; burst 1.35 vs 1.98 at 3 h):", "",
              "| process | MTBR h | stretch | exp. stretch (sim) | exp. theory (ΛΨ) | rel. change % | ±SE % |", "|---|---|---|---|---|---|---|"]
        for _, r in eff[eff.mtbr_h.isin([3.0, 12.0, 48.0])].iterrows():
            L.append(f"| {r['process']} | {r['mtbr_h']:g} | {r['stretch']:.3f} | {r['stretch_exp']:.3f} | {r['stretch_theory_exp']:.3f} | {r['rel_change_pct']:+.1f} | {r['rel_change_se_pct']:.1f} |")
    L += ["", "## 8. Extensions beyond the paper (clearly new, not claims of the paper)", ""]
    for name, desc in (("ablation_modelling_choices.csv", "Ablation of modelling choices (base n=32, s=0.5, MTBR 12 h)"),
                       ("sensitivity_tornado.csv", "One-at-a-time ±50% sensitivity of F*"),
                       ("breakeven_fine_scan.csv", "Fine-grained simulated scan around the predicted break-even u*"),
                       ("stress_market_wide_correlated.csv", "Market-wide correlated reclamation (all spot workers revoked together) — exploratory"),
                       ("integer_vs_continuous_share.csv", "Integer spot counts vs continuous share")):
        df = _csv(name)
        if df is not None:
            L += [f"### {desc}", "", df.round(4).to_markdown(index=False) if hasattr(df, "to_markdown") and _has_tabulate() else "```\n" + df.round(4).to_string(index=False) + "\n```", ""]
    L += ["## 9. Paper-to-code traceability matrix", "", "| Paper item | Source | Test | Experiment / output |", "|---|---|---|---|"]
    for a, b, c, e in TRACE:
        L.append(f"| {a} | `{b}` | `{c}` | {e} |")
    L += ["", "## 10. What is NOT established", "",
          "- No real-cluster measurement, trace replay, or fitted θ/σ exists in the paper or here; all evidence is analytical or simulation of the same model.",
          "- The original robustness data, Prop. 2/Thm 2 run configurations and simulation scripts are unavailable; those experiments are reconstructed and cannot be claimed as exact reproductions.",
          "- The supplied document's equations are illegible (PDF-style export); equations were reconstructed from surrounding text and then verified independently (see MATHEMATICAL_MODEL.md).", ""]
    return "\n".join(L) + "\n"


def _has_tabulate():
    try:
        import tabulate  # noqa
        return True
    except Exception:
        return False


def generate_reports(mode="full", log=print):
    base = ROOT if RES == ROOT / "results" else RES.parent          # scratch runs do not touch the committed docs
    (base / "docs").mkdir(exist_ok=True)
    (base / "paper").mkdir(exist_ok=True)
    (base / "docs" / "VALIDATION_REPORT.md").write_text(validation_report(mode), encoding="utf-8")
    (base / "paper" / "reproduction_report.md").write_text(reproduction_report(mode), encoding="utf-8")
    log("[report] docs/VALIDATION_REPORT.md and paper/reproduction_report.md written")


In [ ]:
%%writefile src/gnn_spot/simulation/__init__.py
"""Monte Carlo / event-driven simulators (independent of the analytical modules)."""


In [ ]:
%%writefile src/gnn_spot/simulation/designs_sim.py
"""Simulators for replica-preserving halting and elastic continuation (Theorem 3), alternating-renewal spot workers.

Each spot worker alternates Up ~ Exp(lam_w) and Down = R (fixed); workers evolve independently of the
job state ("halting assumes workers fail independently while the job is halted", paper Sec. VI).
* halting: the job progresses at rate 1 only while ALL spot workers are up.
* elastic: fixed global batch, theta=1: progress rate = (n - Z(t))/n with Z = number of spot workers down.
Billing: on-demand workers always; spot workers: halting -> billed throughout (S5);
elastic -> billed only while up (paper Eq. 7).  Both billing totals are returned.
These are NOT asserted equal to the formulas; the verification layer compares them.
"""
from __future__ import annotations

import numpy as np


def _simulate(design, T_N, n, n_s, lam_w, R, rng, stationary=True):
    q = lam_w * R / (1.0 + lam_w * R)
    up = np.ones(n_s, bool)
    nxt = np.empty(n_s)
    for i in range(n_s):
        if stationary and rng.random() < q:
            up[i] = False
            nxt[i] = rng.random() * R                  # residual of a deterministic down period
        else:
            nxt[i] = rng.exponential(1.0 / lam_w)
    t = 0.0
    prog = 0.0
    spot_up_time = 0.0
    while True:
        k_down = int((~up).sum())
        rate = (1.0 if k_down == 0 else 0.0) if design == "halting" else (n - k_down) / n
        i = int(nxt.argmin())
        t_next = nxt[i]
        dt = t_next - t
        if rate > 0 and prog + rate * dt >= T_N:
            fin = (T_N - prog) / rate
            spot_up_time += (n_s - k_down) * fin
            return t + fin, spot_up_time
        prog += rate * dt
        spot_up_time += (n_s - k_down) * dt
        t = t_next
        if up[i]:
            up[i] = False
            nxt[i] = t + R
        else:
            up[i] = True
            nxt[i] = t + rng.exponential(1.0 / lam_w)


def simulate_design(design, T_N, n, s, lam_w, R, rho, rng, n_trials, pi=1.0, stationary=True):
    """Returns (W, cost_all_billed, cost_down_unbilled) arrays for `design` in {'halting','elastic'}."""
    if design not in ("halting", "elastic"):
        raise ValueError(design)
    n_s = int(round(n * s))
    W = np.empty(n_trials)
    cost_all = np.empty(n_trials)
    cost_unb = np.empty(n_trials)
    for k in range(n_trials):
        if n_s == 0:
            w, sut = T_N, 0.0
        else:
            w, sut = _simulate(design, T_N, n, n_s, lam_w, R, rng, stationary)
        W[k] = w
        cost_all[k] = pi * (n - n_s + n_s * rho) * w
        cost_unb[k] = pi * ((n - n_s) * w + rho * sut)
    return W, cost_all, cost_unb


In [ ]:
%%writefile src/gnn_spot/simulation/processes.py
"""Revocation processes on the running clock (ages frozen during stalls)."""
from __future__ import annotations

from math import gamma as _gamma

import numpy as np


class PoissonProcess:
    """Aggregate Poisson process of rate lam (per running hour); memoryless so re-drawing is exact."""
    def __init__(self, lam, rng, start=0.0):
        self.lam = lam
        self._t = start + (rng.exponential(1.0 / lam) if lam > 0 else np.inf)

    def next_time(self):
        return self._t

    def consume(self, r, rng):
        self._t = r + (rng.exponential(1.0 / self.lam) if self.lam > 0 else np.inf)
        return 1


class WorkerRenewalProcess:
    """n_s spot workers, each an independent renewal process with i.i.d. lifetimes drawn by `draw(rng)`;
    only the revoked worker's age resets; all ages freeze during stalls (running clock)."""
    def __init__(self, n_s, draw, rng):
        self.n_s = n_s
        self.draw = draw
        self.t = np.array([draw(rng) for _ in range(n_s)]) if n_s > 0 else np.array([np.inf])

    def next_time(self):
        return float(self.t.min())

    def consume(self, r, rng):
        i = int(self.t.argmin())
        self.t[i] = r + self.draw(rng)
        return 1


def exponential_draw(mean):
    return lambda rng: rng.exponential(mean)


def weibull_draw(mean, shape):
    """Weibull with the requested MEAN: scale = mean / Gamma(1 + 1/shape)."""
    scale = mean / _gamma(1.0 + 1.0 / shape)
    return lambda rng: scale * rng.weibull(shape)


class BurstProcess:
    """Burst-correlated revocations: events at rate n_s lam_w / g, each revokes g workers at once
    (same mean per-worker rate lam_w).  g is capped at n_s."""
    def __init__(self, n_s, lam_w, group, rng):
        self.g = max(1, min(group, n_s))
        self.rate = n_s * lam_w / self.g if n_s > 0 else 0.0
        self._t = rng.exponential(1.0 / self.rate) if self.rate > 0 else np.inf

    def next_time(self):
        return self._t

    def consume(self, r, rng):
        self._t = r + (rng.exponential(1.0 / self.rate) if self.rate > 0 else np.inf)
        return self.g


def make_process(kind, n_s, mtbr_h, rng, group=4):
    """kind in {'exponential','weibull0.6','weibull1.5','burst'}; mtbr_h = per-worker mean time between revocations."""
    if n_s == 0:
        return PoissonProcess(0.0, rng)
    lam_w = 1.0 / mtbr_h
    if kind == "exponential":
        return WorkerRenewalProcess(n_s, exponential_draw(mtbr_h), rng)
    if kind.startswith("weibull"):
        return WorkerRenewalProcess(n_s, weibull_draw(mtbr_h, float(kind[len("weibull"):])), rng)
    if kind == "burst":
        return BurstProcess(n_s, lam_w, group, rng)
    raise ValueError(kind)


In [ ]:
%%writefile src/gnn_spot/simulation/results.py
"""Structured simulation output."""
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Any, Dict, Optional

import numpy as np


def make_rngs(seed: int, k: int):
    """k statistically independent Generators spawned from one master seed."""
    return [np.random.default_rng(s) for s in np.random.SeedSequence(seed).spawn(k)]


def wilson_interval(k: int, n: int, z: float = 1.959964):
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    den = 1 + z*z/n
    c = (p + z*z/(2*n)) / den
    h = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return (max(0.0, c - h), min(1.0, c + h))


@dataclass
class SimulationResult:
    config: Dict[str, Any]
    seed: int
    n_trials: int
    mean: float
    var: float
    std: float
    sem: float
    ci95: tuple
    miss_prob: Optional[float] = None
    miss_ci95: Optional[tuple] = None
    cost_mean: Optional[float] = None
    cost_ci95: Optional[tuple] = None
    diagnostics: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self):
        d = asdict(self)
        d["ci95"] = list(self.ci95)
        if self.miss_ci95 is not None:
            d["miss_ci95"] = list(self.miss_ci95)
        if self.cost_ci95 is not None:
            d["cost_ci95"] = list(self.cost_ci95)
        return d


def summarize(samples, config, seed, D=None, cost=None, diagnostics=None) -> SimulationResult:
    x = np.asarray(samples, dtype=float)
    n = x.size
    mean = float(x.mean())
    var = float(x.var(ddof=1)) if n > 1 else 0.0
    sem = float(np.sqrt(var / n)) if n > 1 else 0.0
    res = SimulationResult(config=dict(config), seed=int(seed), n_trials=int(n), mean=mean, var=var,
                           std=float(np.sqrt(var)), sem=sem, ci95=(mean - 1.959964*sem, mean + 1.959964*sem),
                           diagnostics=dict(diagnostics or {}))
    if D is not None:
        k = int(np.sum(x > D))
        res.miss_prob = k / n
        res.miss_ci95 = wilson_interval(k, n)
    if cost is not None:
        c = np.asarray(cost, dtype=float)
        cm = float(c.mean())
        cs = float(c.std(ddof=1) / np.sqrt(c.size)) if c.size > 1 else 0.0
        res.cost_mean = cm
        res.cost_ci95 = (cm - 1.959964*cs, cm + 1.959964*cs)
    return res


In [ ]:
%%writefile src/gnn_spot/simulation/rollback.py
"""Full-rollback simulators.

Two independent implementations of the same model (S1)-(S5):

* `simulate_rollback_vectorized`: segment-level, vectorised across trials (exact for exponential
  revocations because of memorylessness).  Used for the large-N verification runs.
* `simulate_rollback_event_driven`: explicit state machine (WORK -> CKPT -> WORK ... , STALL on a
  revocation) driven by an arbitrary failure process on the *running clock* (ages are frozen
  during stalls).  Used as the transparent reference and for non-exponential / correlated processes.

Conventions: all times in HOURS; `lam_a` per hour; K equal chunks of work w = T_N/K, a checkpoint of
duration delta after every chunk except the last (S3); a revocation during a write loses the write
and the chunk (rollback to the previous committed checkpoint).
"""
from __future__ import annotations

from enum import Enum
from typing import Callable, Optional

import numpy as np


# ---------------------------------------------------------------------------- stall models
def busy_period_md_inf(rng, size, lam, R):
    """Busy period of an M/D/infinity queue (S4b): the stall that starts at a revocation lasts until
    no replacement is pending; further revocations (rate lam) arriving during it extend it by R each.
    Mean (exp(lam R) - 1)/lam."""
    end = np.full(size, float(R))
    arr = rng.exponential(1.0 / lam, size)
    active = arr < end
    while np.any(active):
        idx = np.flatnonzero(active)
        end[idx] = arr[idx] + R
        arr[idx] += rng.exponential(1.0 / lam, idx.size)
        active = arr < end
    return end


# ------------------------------------------------------------------------ vectorised version
def simulate_rollback_vectorized(T_N, lam_a, R, delta, K, n_trials, rng, stall="fixed",
                                 warned_rate=0.0, delta_e=0.0):
    """Return (W, info) for n_trials independent jobs.

    warned_rate>0 adds Proposition-2 warned revocations: Poisson in RUNNING time at `warned_rate`,
    each costing (delta_e + R) of wall time and no lost work.  `lam_a` is then the UNWARNED rate.
    info: total revocations and warned events per trial.
    """
    w = T_N / K
    W = np.zeros(n_trials)
    n_fail = np.zeros(n_trials, dtype=np.int64)
    run_time = np.zeros(n_trials)               # running-clock time (for warned events)
    for k in range(K):
        X = w + delta if k < K - 1 else w
        idx = np.arange(n_trials)
        while idx.size:
            if lam_a > 0:
                Y = rng.exponential(1.0 / lam_a, idx.size)
                ok = Y > X
            else:
                Y = np.full(idx.size, np.inf)
                ok = np.ones(idx.size, bool)
            done = idx[ok]
            W[done] += X
            run_time[done] += X
            bad = idx[~ok]
            if bad.size:
                st = (np.full(bad.size, float(R)) if stall == "fixed"
                      else busy_period_md_inf(rng, bad.size, lam_a, R))
                W[bad] += Y[~ok] + st
                run_time[bad] += Y[~ok]
                n_fail[bad] += 1
            idx = bad
    warned = np.zeros(n_trials, dtype=np.int64)
    if warned_rate > 0:
        warned = rng.poisson(warned_rate * run_time)
        W = W + warned * (delta_e + R)
    return W, {"n_revocations": n_fail, "n_warned": warned}


# ----------------------------------------------------------------------------- event-driven
class Phase(Enum):
    WORK = "work"
    CKPT = "ckpt"
    STALL = "stall"
    DONE = "done"


def simulate_rollback_event_driven(T_N, K, R, delta, process, rng, stall="fixed", lam_for_stall=None,
                                   warned_process=None, delta_e=0.0):
    """One run of the explicit state machine.  Returns (W, stats dict).

    `process`: object with `.next_time()` (next revocation on the running clock) and `.consume(r, rng)`
    (process the event at running time r, returns number of workers revoked).
    `warned_process` (optional): same interface; its events cost delta_e + R and lose no work.
    For stall='overlap' `lam_for_stall` gives the arrival rate of replacements-extending revocations.
    """
    w = T_N / K
    wall = 0.0
    r = 0.0                      # running clock: advances only in WORK/CKPT
    seg = 0
    phase = Phase.WORK
    phase_end = w
    n_events = 0
    n_warn = 0
    lost_work = 0.0
    revoked_workers = 0
    while phase is not Phase.DONE:
        t_fail = process.next_time()
        t_warn = warned_process.next_time() if warned_process is not None else np.inf
        if t_warn < phase_end and t_warn < t_fail:
            wall += (t_warn - r) + delta_e + R
            r = t_warn
            warned_process.consume(r, rng)
            n_warn += 1
            continue
        if t_fail < phase_end:
            # revocation: lose the current chunk (and its checkpoint write if in progress)
            wall += t_fail - r
            if phase is Phase.WORK:
                lost_work += t_fail - (phase_end - w)      # work done in this chunk so far
            else:
                lost_work += w                              # whole chunk; the unfinished write is lost too
            r = t_fail
            revoked_workers += process.consume(r, rng)
            n_events += 1
            phase = Phase.STALL
            if stall == "fixed":
                wall += R
            else:
                wall += float(busy_period_md_inf(rng, 1, lam_for_stall, R)[0])
            phase = Phase.WORK
            phase_end = r + w
        else:
            wall += phase_end - r
            r = phase_end
            if phase is Phase.WORK:
                if seg == K - 1:
                    phase = Phase.DONE
                else:
                    phase = Phase.CKPT
                    phase_end = r + delta
            else:                                   # checkpoint written without interruption -> committed
                seg += 1
                phase = Phase.WORK
                phase_end = r + w
    return wall, {"n_events": n_events, "n_warned": n_warn, "lost_work": lost_work,
                  "revoked_workers": revoked_workers, "running_time": r}


In [ ]:
%%writefile src/gnn_spot/verification/__init__.py



In [ ]:
%%writefile src/gnn_spot/verification/checks.py
"""Verification suite: every check returns structured `CheckResult` records.

Evidence types:  ANALYTICAL (algebra/closed form re-evaluated),  NUMERICAL (independent numerics),
MONTE_CARLO (simulation vs theory with statistical criterion), REPRODUCTION (compares to a number
printed in the paper), RECONSTRUCTED (paper omits the configuration; ours is labelled).
Status: PASS / FAIL / INCONCLUSIVE.  Nothing here is hard-wired to pass.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, asdict, field
from typing import Any, Callable, Dict, List

import numpy as np
from scipy.optimize import minimize_scalar, brentq

from ..models import performance as perf
from ..models import spot
from ..models.lambert import psi, x_star, log_psi, psi_lambertw, x_star_lambertw
from ..checkpointing import baselines as bl
from ..recovery import designs as des
from ..deadlines import completion as comp
from ..deadlines.feasibility import largest_feasible_share, cantelli_feasible_counts, corner_decision
from ..optimization import slack
from ..simulation import rollback as rb
from ..simulation.designs_sim import simulate_design
from ..simulation.results import make_rngs
from . import reference as ref

MC_Z = 3.5          # |z| criterion for Monte Carlo agreement (many comparisons -> conservative-ish)


@dataclass
class CheckResult:
    id: str
    paper_ref: str
    claim: str
    evidence: str
    expected: Any
    computed: Any
    discrepancy: Any
    criterion: str
    status: str
    notes: str = ""

    def to_dict(self):
        return asdict(self)


def _res(id, ref_, claim, ev, exp, got, disc, crit, ok, notes=""):
    return CheckResult(id, ref_, claim, ev, _jsonable(exp), _jsonable(got), _jsonable(disc), crit,
                       "PASS" if ok else "FAIL", notes)


def _jsonable(x):
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, dict):
        return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    return x


H_ = 3600.0
D_S, R_S = 30.0, 300.0          # paper's main verification: delta=30 s, R=300 s


# ===================================================================== Proposition 1
def check_prop1(mode) -> List[CheckResult]:
    out = []
    a, sg, z = 3600.0, 20.0, 40.0
    b = a - sg
    p = perf.PerfParams(a=a, theta=1.0, sigma=sg)
    npts = 400_000 if mode == "full" else 20_000
    grid = np.linspace(1.0, 400.0, npts)
    Jg = perf.J(grid, p, 1.0, z)
    ng = grid[np.argmin(Jg)]
    nstar = perf.n_star_closed_form(b, sg, z)
    out.append(_res("P1c-root", "Prop. 1(c); Table VIII", "closed-form n* = sqrt(z b/sigma) equals grid argmin (theta=1)",
                    "NUMERICAL", nstar, ng, abs(nstar - ng), f"|n*-argmin| <= grid step {grid[1]-grid[0]:.2e}",
                    abs(nstar - ng) <= (grid[1] - grid[0])))
    reg_true = Jg - Jg.min()
    reg_form = perf.j_regret_closed_form(grid, b, sg, z)
    jmin = perf.j_min_closed_form(b, sg, z)
    err = float(np.max(np.abs(Jg - (jmin + reg_form)) / Jg))
    out.append(_res("P1c-regret", "Prop. 1(c); Table VIII ('12 digits')",
                    "J(n)=J* + sigma (n-n*)^2/n and J*=(sqrt b + sqrt(z sigma))^2 over the whole grid",
                    "NUMERICAL", 0.0, err, err, "max relative error <= 1e-12", err <= 1e-12))
    # general theta roots vs grid argmin
    for theta in (0.9, 0.7):
        for variant in ("model1", "a_minus_sigma"):
            if variant == "model1":
                pp = perf.PerfParams(a=a, theta=theta, sigma=sg)
                Jf = lambda n: perf.J(n, pp, 1.0, z)
                root = perf.cost_optimal_n_root(pp, z, 1 + 1e-9, 5e4)
            else:   # paper: "a replaced by a - sigma": T = sigma + (a-sigma) n^-theta
                class _P:   # tiny adapter: T(n)=sigma+b n^-theta
                    pass
                Tn = lambda n: sg + b * np.asarray(n, float) ** (-theta)
                Jf = lambda n: (np.asarray(n, float) + z) * Tn(n)
                g = lambda n: (-(np.log(Tn(n * np.exp(1e-6))) - np.log(Tn(n * np.exp(-1e-6)))) / 2e-6) - n / (n + z)
                gg = np.geomspace(1.0001, 5e4, 6000)
                gv = np.array([g(x) for x in gg])
                i = np.flatnonzero((gv[:-1] > 0) & (gv[1:] < 0))
                root = brentq(g, gg[i[-1]], gg[i[-1] + 1], xtol=1e-12) if i.size else None
            fine = np.linspace(max(1.0, root * 0.5), root * 1.5, 400_001) if root else None
            ga = fine[np.argmin(Jf(fine))] if root else None
            ok = root is not None and abs(root - ga) <= (fine[1] - fine[0]) * 1.01
            out.append(_res(f"P1c-theta{theta}-{variant}", "Prop. 1(c); Table VIII",
                            f"root of eps(n)=n/(n+z) equals grid argmin, theta={theta}, variant={variant}",
                            "NUMERICAL", ga, root, None if root is None else abs(root - ga),
                            "within one grid step of a fine local grid", ok))
    # elasticity analytic vs numerical, general model
    pg = perf.PerfParams(a=a, theta=0.9, A=500.0, kappa0=0.3, gamma=0.5, sigma=sg, alpha=0.5)
    ns = np.array([2.0, 3.0, 7.0, 20.0, 60.0, 150.0])
    e1, e2 = perf.elasticity(ns, pg), perf.elasticity_numeric(ns, pg)
    d = float(np.max(np.abs(e1 - e2)))
    out.append(_res("P1b-elasticity", "Prop. 1(b)", "analytical elasticity = numerical d ln T/d ln n (kappa<1 region and capped)",
                    "NUMERICAL", e2.tolist(), e1.tolist(), d, "max abs diff <= 1e-7", d <= 1e-7))
    # J' formula
    h = 1e-5
    dj = perf.dJ_dn(ns, pg, 1.0, 40.0)
    dn = (perf.J(ns + h, pg, 1.0, 40.0) - perf.J(ns - h, pg, 1.0, 40.0)) / (2 * h)
    rel = float(np.max(np.abs(dj - dn) / np.abs(dn)))
    out.append(_res("P1a-dJ", "Prop. 1(a)", "J'(n)=pi T [1-(n+z) eps/n] vs central difference", "NUMERICAL", dn.tolist(), dj.tolist(),
                    rel, "max rel diff <= 1e-6", rel <= 1e-6))
    # (d) deadline minimum
    N, Dh = 100.0, 100 * (sg + b / 9.0) / H_
    nD = perf.deadline_min_workers(a, sg, Dh * H_, N)
    out.append(_res("P1d-nD", "Prop. 1(d)", "n_D=(a-sigma)/(D/N-sigma) solves N T(n_D)=D", "ANALYTICAL", 9.0, nD, abs(nD - 9.0),
                    "|n_D - 9| <= 1e-9", abs(nD - 9.0) < 1e-9))
    return out


# ===================================================================== Lemma 1
def check_lemma1(mode) -> List[CheckResult]:
    out = []
    vs = np.geomspace(1e-4, 5.0, 400 if mode == "full" else 60)
    errs_psi, errs_x = [], []
    for v in vs:
        f = lambda lx: bl.reduced_overhead(np.exp(lx), v)       # numerical minimisation over ln x
        r = minimize_scalar(f, bounds=(np.log(1e-6), np.log(50.0)), method="bounded", options={"xatol": 1e-12})
        errs_psi.append(abs(r.fun - float(psi(v))) / r.fun)
        errs_x.append(abs(np.exp(r.x) - float(x_star(v))) / float(x_star(v)))
    out.append(_res("L1-psi", "Lemma 1; Table VIII", "closed-form Psi(v)=min_x (e^{x+v}-1)/x vs numerical minimisation, v in [1e-4,5]",
                    "NUMERICAL", 0.0, max(errs_psi), max(errs_psi), "max rel err <= 1e-9", max(errs_psi) <= 1e-9))
    out.append(_res("L1-xstar", "Lemma 1", "closed-form x* vs numerical argmin (flat minimum -> looser tol)",
                    "NUMERICAL", 0.0, max(errs_x), max(errs_x), "max rel err <= 1e-4", max(errs_x) <= 1e-4))
    # stable path vs naive lambertw
    vv = np.geomspace(1e-4, 5.0, 100)
    d = float(np.max(np.abs(psi(vv) - psi_lambertw(vv)) / psi(vv)))
    out.append(_res("L1-lambertw", "Lemma 1", "stable solver agrees with scipy W0 where W0 is well-conditioned", "NUMERICAL", 0.0, d, d,
                    "max rel diff <= 1e-9", d <= 1e-9))
    # properties
    vv = np.linspace(0, 6, 6001)
    L = log_psi(vv)
    mono = bool(np.all(np.diff(psi(vv)) > 0))
    conc = bool(np.all(np.diff(L, 2) < 1e-12))
    small = np.geomspace(1e-8, 1e-2, 50)
    asym = float(np.max(np.abs(psi(small) - 1 - np.sqrt(2 * small)) / small))
    out.append(_res("L1-props", "Lemma 1", "Psi(0)=1, Psi strictly increasing, ln Psi concave, Psi=1+sqrt(2v)+O(v)",
                    "NUMERICAL", True, bool(mono and conc and float(psi(0.0)) == 1.0 and asym < 2.0),
                    {"asym_const": asym}, "monotone & concave on grid; (Psi-1-sqrt(2v))/v bounded", mono and conc and asym < 2.0))
    # stationarity of Omega at x*: derivative zero
    v = 0.37
    xs = float(x_star(v))
    h = 1e-6
    dO = (bl.reduced_overhead(xs + h, v) - bl.reduced_overhead(xs - h, v)) / (2 * h)
    out.append(_res("L1-stationary", "Lemma 1 proof", "(1-x) e^{x+v} = 1 at x*; dOmega/dx = 0", "NUMERICAL", 0.0, float(dO), abs(dO),
                    "|dOmega/dx| <= 1e-6", abs(dO) <= 1e-6))
    # small-v conditioning: independent 60-digit reference via mpmath (solve L - 1 + e^-L = v)
    import mpmath as mp
    mp.mp.dps = 60
    worst = 0.0
    for v in (1e-12, 1e-9, 1e-6):
        Lref = mp.findroot(lambda L: L - 1 + mp.e**(-L) - mp.mpf(v), mp.sqrt(2 * mp.mpf(v)))
        Lp = float(log_psi(v))
        worst = max(worst, abs(Lp - float(Lref)) / float(Lref))
    out.append(_res("L1-smallv", "numerical stability", "branch-point regime v in {1e-12,1e-9,1e-6}: ln Psi vs 60-digit mpmath reference",
                    "NUMERICAL", 0.0, worst, worst, "max rel error <= 1e-12", worst <= 1e-12,
                    "naive scipy W0 gives Psi(1e-12)=" + str(float(psi_lambertw(1e-12))) + " (true ~1+1.41e-6)"))
    return out


# ===================================================================== Theorem 1
def check_theorem1(mode) -> List[CheckResult]:
    out = []
    rng = np.random.default_rng(20260101)
    ncfg = 3000
    s = np.linspace(0.0, 1.0, 201)
    viol_conc = viol_corner = viol_updown = 0
    for k in range(ncfg):
        n = int(rng.integers(2, 400))
        lam_w = 10 ** rng.uniform(-3.5, 0.0)
        R = rng.uniform(60, 1800) / H_
        d = rng.uniform(5, 300) / H_
        rho = rng.uniform(0.1, 0.9)
        stall = "fixed" if rng.random() < 0.5 else "overlap"
        z = 0.0 if rng.random() < 0.5 else rng.uniform(0, 100)
        with np.errstate(over="ignore"):
            lf = spot.log_cost_factor(n, s, rho, lam_w, R, d, stall, z)
        if not np.all(np.isfinite(lf)):
            continue
        d2 = np.diff(lf, 2)
        if np.any(d2 > 1e-9 * (1 + np.abs(lf[1:-1]))):
            viol_conc += 1
        i = int(np.argmin(lf))
        if 0 < i < s.size - 1:
            viol_corner += 1
        # up-then-down: sign pattern of first differences changes at most once from + to -
        sg = np.sign(np.diff(lf))
        sg = sg[sg != 0]
        if np.any((sg[1:] > sg[:-1])):
            viol_updown += 1
    out.append(_res("T1a-concave", "Thm 1(a); Table VIII", f"ln F*(s) strictly concave in s over {ncfg} random configs (z=0 and z>0, both stalls)",
                    "NUMERICAL", 0, viol_conc, viol_conc, "0 configurations with positive second difference > 1e-9 (rel.)",
                    viol_conc == 0, "random ranges are ours (paper does not state them)"))
    out.append(_res("T1a-corner", "Thm 1(a); Table VIII", "argmin over s in [0,1] at an endpoint in all configs", "NUMERICAL", 0, viol_corner,
                    viol_corner, "0 interior minima", viol_corner == 0))
    out.append(_res("T1a-updown", "Thm 1(a); Table VIII ('up-then-down')", "F*(s) pattern: once decreasing, never increasing again (concave shape)",
                    "NUMERICAL", 0, viol_updown, viol_updown, "0 violations", viol_updown == 0))
    # Theorem 1(b): monotone in n and lam_w; unique break-even
    ns = np.arange(2, 300)
    F = spot.cost_factor(ns, 0.5, 0.35, 0.01, R_S / H_, D_S / H_, "fixed")
    Fl = spot.cost_factor(16, 0.5, 0.35, np.geomspace(1e-4, 1.0, 300), R_S / H_, D_S / H_, "fixed")
    ok = bool(np.all(np.diff(F) > 0) and np.all(np.diff(Fl) > 0))
    out.append(_res("T1b-monotone", "Thm 1(b)", "F* strictly increasing in n and in lam_w (z=0)", "NUMERICAL", True, ok, None, "strict increase on grids", ok))
    # break-even root consistency: F*(at u*)=1
    errs = []
    for m in (2/3, 1/2, 1/3, 0.14):
        for dr in (0.05, 0.1, 0.5, 1.0, 2.0):
            for stall in ("fixed", "overlap"):
                u = spot.breakeven_u(m, dr, stall)
                errs.append(abs(m * float(spot.Lambda(u, stall)) * float(psi(u * dr)) - 1))
    out.append(_res("T1b-breakeven", "Thm 1(b)", "root of Lambda(u)Psi(u delta/R)=1/m satisfies F*=1", "NUMERICAL", 0.0, max(errs), max(errs), "<=1e-12", max(errs) <= 1e-12))
    return out


# ===================================================================== Theorem 2
def _thm2_job():
    perf_ = perf.PerfParams(a=3600.0, sigma=20.0)
    N = 100.0
    D = N * (20.0 + 3580.0 / 9.0) / H_      # RECONSTRUCTED: deadline chosen so that n_D = 9 (the paper's n_D)
    return slack.Job(perf_, N, D)


def check_theorem2(mode) -> List[CheckResult]:
    out = []
    job = _thm2_job()
    R, d, rho = R_S / H_, D_S / H_, 0.35
    # derivative identity C'(n)
    lam_w = 1 / 3.0
    errs = []
    for stall in ("fixed", "overlap"):
        for n in (12.0, 20.0, 33.0, 60.0):
            h = 1e-5
            num = (slack.binding_cost(n + h, job, lam_w, R, d, rho, stall) - slack.binding_cost(n - h, job, lam_w, R, d, rho, stall)) / (2 * h)
            ana = slack.dC_dn_analytic(n, job, lam_w, R, d, rho, stall)
            errs.append(abs(num - ana) / abs(ana))
    out.append(_res("T2-deriv", "Thm 2 proof", "C'(n)=pi D[1-(1-rho) eps/(lam_w n ell(y_D))] vs finite difference", "NUMERICAL", 0.0, max(errs), max(errs),
                    "max rel diff <= 1e-5", max(errs) <= 1e-5))
    # y_D' = eps/(n ell)
    n, h = 25.0, 1e-5
    yp = (slack.y_D(n + h, job, None, R, d) - slack.y_D(n - h, job, None, R, d)) / (2 * h)
    yf = float(perf.elasticity(n, job.perf)) / (n * float(spot.ell(slack.y_D(n, job, None, R, d), R, d)))
    out.append(_res("T2-yprime", "Thm 2 proof", "y_D'(n)=eps(n)/(n ell(y_D))", "NUMERICAL", yp, yf, abs(yp - yf) / abs(yp), "rel diff <= 1e-6", abs(yp - yf) / abs(yp) <= 1e-6))
    # local condition sign == sign of -C'
    ok = True
    for stall in ("fixed", "overlap"):
        for n in np.linspace(job.n_D() * 1.05, job.n_D() * 6, 40):
            lhs, rhs, hold = slack.marginal_condition(n, job, lam_w, R, d, rho, stall)
            c = slack.dC_dn_analytic(n, job, lam_w, R, d, rho, stall)
            ok &= (hold == (c < 0))
    out.append(_res("T2-local", "Eq. (5)", "(1-rho)eps > lam_w n ell  <=>  C'(n) < 0 on a grid of n", "ANALYTICAL", True, bool(ok), None, "equivalence at all grid points", ok))
    # ell diverges at n_D
    n1 = job.n_D() * (1 + 1e-9)
    c1 = slack.dC_dn_analytic(n1, job, lam_w, R, d, rho)
    out.append(_res("T2-nD-local", "Thm 2 / App. B", "C'(n) -> pi D > 0 just above n_D (condition (5) fails there)", "NUMERICAL", job.pi * job.D, c1,
                    abs(c1 - job.D) / job.D, "rel diff <= 1e-3", abs(c1 - job.D) / job.D <= 1e-3))
    # brute force at three rates (RECONSTRUCTED rates)
    nmax = 120 if mode == "full" else 60
    rates = (6.0, 3.0, 1.5)
    rows = []
    for mt in rates:
        lw = 1.0 / mt
        ce = slack.continuous_optimum(job, lw, R, d, rho, n_max_factor=nmax / job.n_D() if nmax / job.n_D() > 2 else 4.0)
        bf = slack.brute_force_integer(job, lw, R, d, rho, n_max=nmax)
        # integer n, continuous s: cost at integer n of the spot-endpoint
        ints = np.arange(int(np.ceil(job.n_D())), nmax + 1)
        c_int = np.array([slack._spot_endpoint_cost(float(k), job, lw, R, d, rho, "fixed") for k in ints])
        kbest = int(ints[np.argmin(c_int)])
        rows.append({"mtbr_h": mt, "n_D": job.n_D(), "n_opt_cont": ce["n_opt"], "cost_cont": ce["cost_opt"], "n_best_int_cont_s": kbest,
                     "cost_int_n_cont_s": float(c_int.min()), "n_best_int_int_s": bf["best"]["n"], "n_s_best": bf["best"]["n_s"],
                     "cost_int_int": bf["best"]["cost"], "cost_no_spot": bf["best_no_spot"]["cost"]})
    # formula-based decision (integer n, continuous s) = min(spot endpoint, best no-spot); it is a RELAXATION of the
    # integer-(n,n_s) problem, so it must lower-bound the brute force; the gap measures integrality of n_s.
    for r in rows:
        r["cost_formula_best"] = min(r["cost_int_n_cont_s"], r["cost_no_spot"])
    gaps = [(r["cost_int_int"] - r["cost_formula_best"]) / r["cost_formula_best"] for r in rows]
    # one spot worker changes the deadline-binding bill by at most pi D (1-rho): bound the gap by that, relative
    bound = [job.pi * job.D * (1 - rho) / r["cost_formula_best"] for r in rows]
    out.append(_res("T2-brute", "Thm 2; Table VIII", "deadline-binding formula (integer n, continuous s) vs exhaustive integer (n,n_s) search at 3 revocation rates",
                    "RECONSTRUCTED", "formula <= brute force and gap <= one spot worker's bill", rows, {"relative_gaps": gaps},
                    "formula never exceeds the integer optimum (relaxation) and the gap is at most the cost of one spot worker pi D (1-rho)",
                    all(g >= -1e-9 for g in gaps) and all(g <= b for g, b in zip(gaps, bound)),
                    "Revocation rates, N, D are not stated in the paper; MTBR in {6,3,1.5} h and D with n_D=9 are our reconstruction."))
    r3 = [r for r in rows if r["mtbr_h"] == 3.0][0]
    ok21 = r3["n_best_int_cont_s"] in (21, 22) and abs(r3["n_opt_cont"] - 21.5) < 1.0
    out.append(_res("T2-n21", "Thm 2; Table VIII ('optimum n=21 vs n_D=9 at MTBR 3 h')", "cost-optimal cluster at MTBR 3 h exceeds deadline minimum; paper reports n=21",
                    "RECONSTRUCTED", ref.THM2_REPORTED, {"n_opt_continuous": r3["n_opt_cont"], "n_int_continuous_s": r3["n_best_int_cont_s"],
                    "n_int_integer_s": r3["n_best_int_int_s"]}, r3["n_opt_cont"] - 21,
                    "continuous optimum within 1 worker of 21", abs(r3["n_opt_cont"] - 21) <= 1.0,
                    "Integer search over (n,n_s) gives n=%d (n_s=%d) because spot counts are integral." % (r3["n_best_int_int_s"], r3["n_s_best"])))
    return out


# ===================================================================== Remark 1 / baselines
def check_remark1(mode) -> List[CheckResult]:
    out = []
    rng = np.random.default_rng(7)
    N = 20_000
    u = 10 ** rng.uniform(-3, 1, N)
    v = 10 ** rng.uniform(-4, np.log10(5.0), N)
    viol = 0
    for stall in ("fixed", "overlap"):
        oy = bl.young_first_order_overhead(u, v) if stall == "fixed" else None
        gp = spot.Lambda(u, stall) * psi(v)
        # paper's Remark: first-order overhead 1+sqrt(2v)+u < Lambda*Psi (shown for S4a: u <= Lambda-1)
        viol += int(np.sum(bl.young_first_order_overhead(u, v) >= gp))
    out.append(_res("R1i", "Remark 1(i); Table VIII", "Young first-order overhead 1+sqrt(2v)+u < Lambda(u)Psi(v) over 20,000 random (u,v), both stalls",
                    "NUMERICAL", 0, viol, viol, "0 violations", viol == 0, "(u,v) sampling ranges are ours"))
    vv = np.geomspace(1e-4, 1e2, 200_000)
    yr = {t: float(brentq(lambda x: bl.young_regret(x) - t, 1e-3, 5)) for t in (0.01, 0.05, 0.10)}
    exp = ref.YOUNG_REGRET_V
    ok = all(abs(yr[t] - exp[t]) < 0.006 for t in exp)
    out.append(_res("R1ii", "Remark 1(ii)", "Young regret exceeds 1%,5%,10% at v ~ 0.17, 0.50, 0.79", "REPRODUCTION", exp, yr,
                    {t: yr[t] - exp[t] for t in exp}, "|v_computed - v_paper| < 0.006 (2-decimal rounding)", ok))
    dmax = 100 * float(np.max(bl.daly_regret(vv)))
    out.append(_res("R1iii", "Remark 1(iii)", "Daly regret <= 0.19% over v in [1e-4,1e2] (numerical)", "REPRODUCTION", ref.DALY_MAX_REGRET_PCT, dmax,
                    dmax - ref.DALY_MAX_REGRET_PCT, "computed max <= 0.195%", dmax <= 0.195))
    # Young regret unbounded
    rg = [float(bl.young_regret(v)) for v in (1.0, 10.0, 100.0, 500.0)]
    out.append(_res("R1ii-unbounded", "Remark 1(ii)", "Young's regret grows without bound in v (strictly increasing, exceeds 1e6 by v=500)", "NUMERICAL", "unbounded", rg, None,
                    "strictly increasing along v=1,10,100,500 and last > 1e6", all(np.diff(rg) > 0) and rg[-1] > 1e6))
    return out


def check_table3(mode) -> List[CheckResult]:
    out = []
    worst = 0.0
    cells = {}
    for n_s, row in ref.TABLE_III.items():
        for v, pct in row.items():
            c = 100 * float(bl.per_worker_rate_excess(n_s, v))
            cells[f"n_s={n_s},v={v}"] = (pct, round(c, 3))
            worst = max(worst, abs(c - pct) / pct)
    # allow 3 significant digits of rounding (<=0.4%); report the worst
    bad = {k: v for k, v in cells.items() if abs(v[1] - v[0]) / v[0] > 0.004}
    out.append(_res("TabIII", "Table III", "relative excess cost of per-worker-rate interval, 16 cells", "REPRODUCTION", "printed values", cells, worst,
                    "each cell within 0.4% relative (3-significant-digit rounding)", len(bad) == 0,
                    "Cells outside tolerance: %s" % bad if bad else
                    "Max relative deviation %.2f%% (cell n_s=24,v=0.01 shows 26.84 vs printed 26.9)." % (100 * worst)))
    return out


def check_table4_6(mode) -> List[CheckResult]:
    out = []
    cells = {}
    bad = {}
    for (m, dr), (fo, a_, b_) in ref.TABLE_IV.items():
        got = (spot.breakeven_u_young(m, dr), spot.breakeven_u(m, dr, "fixed"), spot.breakeven_u(m, dr, "overlap"))
        cells[f"m={m:.3f},d/R={dr}"] = {"paper": (fo, a_, b_), "computed": tuple(round(g, 4) for g in got)}
        for pv, g in zip((fo, a_, b_), got):
            dig = 2 if pv >= 0.1 else 3
            if abs(g - pv) > 0.5 * 10 ** (-dig) * 1.01 + 1e-12:
                bad[f"m={m:.3f},d/R={dr}"] = (pv, g)
    out.append(_res("TabIV", "Table IV", "break-even stall load u* (first-order, exact S4a, exact S4b), m in {2/3,1/3,0.14}", "REPRODUCTION", "printed", cells, bad,
                    "each entry equals the printed value to its printed precision (m=0.67 read as 2/3, 0.33 as 1/3)", len(bad) == 0,
                    "Young's first-order u* > exact u* in every row: %s" % all(c["computed"][0] > c["computed"][1] for c in cells.values())))
    # Table VI.  The printed u values match the EXACT rate -ln(0.8)/730 = 3.057e-4 (not its 2-digit rounding 3.1e-4).
    lam = -math.log(0.8) / 730.0
    cellsv, badv = {}, {}
    def _half_ulp(x):      # half a unit in the last printed place (printed to 2 significant digits)
        e = math.floor(math.log10(x))
        return 0.5 * 10 ** (e - 1)
    for Rm, row in ref.TABLE_VI.items():
        for ns in (64, 256, 1024):
            u = ns * lam * Rm / 60.0
            cellsv[f"R={Rm}min,n_s={ns}"] = (row[ns], round(u, 5))
            if abs(u - row[ns]) > _half_ulp(row[ns]) * 1.05:
                badv[f"R={Rm},n_s={ns}"] = (row[ns], u)
        n_at = 0.9 / (lam * Rm / 60.0)
        cellsv[f"R={Rm}min,n_s at u*=0.9"] = (row["n_at_ustar"], round(n_at))
        if abs(n_at - row["n_at_ustar"]) / row["n_at_ustar"] > 0.02:
            badv[f"R={Rm},n_at"] = (row["n_at_ustar"], n_at)
    out.append(_res("TabVI", "Table VI", "stall load at lam_w=-ln(0.8)/730 /h and n_s needed to reach u*=0.9", "REPRODUCTION", "printed", cellsv, badv,
                    "u equals the printed value to its printed precision (2 s.f.); n_s within 2% ('~')", len(badv) == 0,
                    "With the rounded rate 3.1e-4 the R=5 min, n_s=64 cell would be 0.00165, i.e. 0.0017 when rounded: the paper evidently used the exact rate."))
    lw_exact = -math.log(0.8) / 730.0
    out.append(_res("SecIX-rate", "Sec. IX-A", "lam_w >= -ln(0.8)/730 h = 3.1e-4 /h; MTBR below ~3300 h", "REPRODUCTION", (3.1e-4, 3300), (lw_exact, 1 / lw_exact),
                    None, "3.1e-4 to 2 digits; MTBR within 2% of 3300", abs(lw_exact - 3.1e-4) < 0.06e-4 and abs(1 / lw_exact - 3300) / 3300 < 0.02))
    stress = 64 * (5 / 60) / 0.5
    out.append(_res("SecIX-stress", "Sec. IX-A (third finding)", "u~0.5 at n_s=64,R=5 min needs per-worker MTBR ~ 10 h, ~300x more frequent than 3300 h", "REPRODUCTION",
                    (10, 300), (stress, 3300 / stress), None, "MTBR within 10% of 10 h; factor within 10% of 300", abs(stress - 10) / 10 < 0.1 and abs(3300 / stress - 300) / 300 < 0.1))
    return out


# ===================================================================== Appendix A / Table IX
def check_appendix_a(mode) -> List[CheckResult]:
    out = []
    T_N, d, R = 7.40, D_S / H_, R_S / H_
    ntr = 40_000 if mode == "full" else 8_000
    rows = {}
    thr_ok = True
    mc_ok = True
    z_vals = {}
    seeds = make_rngs(11, 4)
    for (mt, rng_) in zip((12, 6, 3, 2), seeds):
        lam = 8.0 / mt
        K_p, mu_p, _, sd_p, _ = ref.TABLE_IX[mt][0], ref.TABLE_IX[mt][1], ref.TABLE_IX[mt][2], ref.TABLE_IX[mt][3], ref.TABLE_IX[mt][4]
        m = comp.completion_moments(T_N, lam, R, d)
        thr_ok &= (m.K == K_p and abs(m.mean - mu_p) < 6e-4 and abs(m.std - sd_p) < 6e-4)
        W, _ = rb.simulate_rollback_vectorized(T_N, lam, R, d, m.K, ntr, rng_)
        se = W.std(ddof=1) / np.sqrt(ntr)
        z_mean = (W.mean() - m.mean) / se
        # std: se of sample std ~ s/sqrt(2n) (normal approx; kurtosis correction small)
        k = float(((W - W.mean()) ** 4).mean() / W.var() ** 2)
        se_sd = W.std() * np.sqrt((k - 1) / 4 / ntr)
        z_sd = (W.std(ddof=1) - m.std) / se_sd
        mc_ok &= abs(z_mean) < MC_Z and abs(z_sd) < MC_Z
        rows[f"MTBR={mt}h"] = {"K": m.K, "mean_th": round(m.mean, 4), "std_th": round(m.std, 4), "mean_sim": round(float(W.mean()), 4),
                               "std_sim": round(float(W.std(ddof=1)), 4), "z_mean": round(float(z_mean), 2), "z_std": round(float(z_sd), 2),
                               "paper": ref.TABLE_IX[mt]}
    out.append(_res("TabIX-theory", "Table IX (theory columns); Lemma 2", "K, mean and std from Eqs. (10)-(11)-equivalent moments match printed Table IX",
                    "REPRODUCTION", ref.TABLE_IX, {k: (v["K"], v["mean_th"], v["std_th"]) for k, v in rows.items()}, None,
                    "K equal; mean,std within 6e-4 h (3-decimal rounding)", thr_ok,
                    "Variance formula derived independently here (paper's Eq. (11) is illegible in the supplied file)."))
    out.append(_res("TabIX-sim", "Table IX; Table VIII", f"analytical mean/std vs {ntr:,} simulated runs per row", "MONTE_CARLO", "|z|<%.1f" % MC_Z, rows, None,
                    f"|z| < {MC_Z} for mean and std in all rows", mc_ok))
    # moments vs numerical differentiation of Laplace transform (independent of closed form)
    X, lam = 0.5, 2.0
    h = 1e-4
    phi = lambda th: comp.segment_laplace(th, X, lam, R)
    m1 = -(phi(h) - phi(-h)) / (2 * h)
    m2 = (phi(h) - 2 * phi(0.0) + phi(-h)) / h**2
    mean_c, var_c = comp.segment_mean(X, lam, R), comp.segment_var(X, lam, R)
    e1, e2 = abs(m1 - mean_c) / mean_c, abs((m2 - m1**2) - var_c) / var_c
    out.append(_res("AppA-laplace", "Appendix A, Eq. (10)", "closed-form segment mean/variance vs finite-difference derivatives of the Laplace transform",
                    "NUMERICAL", (mean_c, var_c), (m1, m2 - m1**2), (e1, e2), "rel diff <= 1e-5", max(e1, e2) <= 1e-5))
    out.append(_res("AppA-phi0", "Appendix A", "transform equals 1 at theta=0", "ANALYTICAL", 1.0, float(phi(0.0)), abs(float(phi(0.0)) - 1), "<=1e-12", abs(float(phi(0.0)) - 1) < 1e-12))
    # Cantelli multiplier and bound examples
    mult = comp.cantelli_multiplier(0.05)
    out.append(_res("Cantelli-4.36", "Eq. (8)", "5% miss budget -> multiplier sqrt(19) = 4.36", "REPRODUCTION", ref.CANTELLI_MULTIPLIER_5PCT, mult, mult - 4.36,
                    "|mult - 4.36| < 0.005", abs(mult - 4.36) < 0.005))
    lam = 8 / 3.0
    mm = comp.completion_moments(T_N, lam, R, d)
    rng = make_rngs(23, 1)[0]
    ntr2 = 200_000 if mode == "full" else 40_000
    W, _ = rb.simulate_rollback_vectorized(T_N, lam, R, d, mm.K, ntr2, rng)
    ex = {}
    ok = True
    for f, (sim_p, ch_p, ca_p) in ref.APP_A_EXAMPLE.items():
        D = f * mm.mean
        can = comp.cantelli_bound(D, mm.mean, mm.var)
        che = comp.chernoff_bound(D, T_N, lam, R, d, mm.K)
        k = int(np.sum(W > D))
        lo, hi = __import__("gnn_spot.simulation.results", fromlist=["x"]).wilson_interval(k, ntr2)
        ex[f"D={f}mu"] = {"cantelli": (round(can, 4), ca_p), "chernoff": (round(che, 4), ch_p), "sim_miss": (round(k / ntr2, 5), sim_p), "sim_ci": (lo, hi)}
        ok &= (abs(can - ca_p) <= 0.0006 + 0.005 * ca_p) and (abs(che - ch_p) <= 0.0006 + 0.005 * ch_p) and (k / ntr2 <= che) and (k / ntr2 <= can)
    out.append(_res("AppA-bounds", "Appendix A; Table VIII", "Chernoff & Cantelli bounds hold and reproduce the quoted values at MTBR 3 h (D=1.1mu, 1.25mu)",
                    "REPRODUCTION", ref.APP_A_EXAMPLE, ex, None, "bounds match printed values to rounding; simulated miss <= both bounds", ok))
    # K chunks and checkpoint count relation
    ks = {mt: comp.n_chunks(T_N, 8 / mt, 8 / mt * d) for mt in (12, 6, 3, 2)}
    out.append(_res("AppA-K", "Appendix A", "K=round(T_N/tau*), tau*=x*/lam_a", "REPRODUCTION", {k: v[0] for k, v in ref.TABLE_IX.items()}, ks, None, "equal", all(ks[k] == ref.TABLE_IX[k][0] for k in ks)))
    return out


# ===================================================================== Proposition 2 (warnings)
def check_prop2(mode) -> List[CheckResult]:
    out = []
    T_N, d, R = 7.40, D_S / H_, R_S / H_
    de = d
    lam_a_from = lambda mt: 8.0 / mt
    # (1) phi=0.9: theory vs vectorised MC vs event-driven MC at ASSUMED config (MTBR 3h, 8 spot workers)
    for label, mt in (("MTBR3h", 3.0),):
        lam = lam_a_from(mt)
        u, v, ve = lam * R, lam * d, lam * de
        phi = 0.9
        th = T_N * float(des.warning_overhead(u, v, phi, ve))
        lam_u = (1 - phi) * lam
        K = comp.n_chunks(T_N, lam_u, lam_u * d) if lam_u > 0 else 1
        th_K = float(des.warning_mean_exact_K(T_N, lam, phi, R, d, de, K))
        ntr = 40_000 if mode == "full" else 8_000
        rng = make_rngs(31, 2)
        W, _ = rb.simulate_rollback_vectorized(T_N, lam_u, R, d, K, ntr, rng[0], warned_rate=phi * lam, delta_e=de)
        se = W.std(ddof=1) / np.sqrt(ntr)
        z = (W.mean() - th_K) / se
        out.append(_res("P2-phi0.9-exactK", "Prop. 2 (derivation check)", f"simulation vs EXACT integer-K mean (derived here), phi=0.9 ({label}), {ntr:,} runs",
                        "MONTE_CARLO", th_K, float(W.mean()), float(W.mean() - th_K), f"|z| < {MC_Z}", abs(z) < MC_Z, f"z={z:.2f}; K={K}"))
        out.append(_res("P2-phi0.9", "Prop. 2; Table VIII", f"closed form Psi((1-phi)v)(1+u+phi v_e) (continuous interval) vs exact integer-K mean, phi=0.9 ({label}, delta_e=delta)",
                        "RECONSTRUCTED", th, th_K, th_K - th,
                        "relative difference <= 0.5% (continuous-K approximation; paper reports 9.072 vs 9.066, i.e. 0.07%)",
                        abs(th_K - th) / th <= 0.005, f"K-rounding effect {100*(th_K-th)/th:+.3f}%. MTBR, delta_e, K are not stated in the paper (assumed MTBR 3 h, delta_e=delta)."))
    # reverse-engineered MTBR for the paper's theory mean 9.072 h
    f = lambda mt: T_N * float(des.warning_overhead(lam_a_from(mt) * R, lam_a_from(mt) * d, 0.9, lam_a_from(mt) * d)) - 9.072
    mt_star = brentq(f, 1.0, 20.0)
    out.append(_res("P2-infer", "Prop. 2; Table VIII", "which per-worker MTBR (8 spot workers, delta_e=delta, T_N=7.40 h) makes the paper's quoted theory mean 9.072 h?",
                    "RECONSTRUCTED", 9.072, T_N * float(des.warning_overhead(lam_a_from(mt_star) * R, lam_a_from(mt_star) * d, 0.9, lam_a_from(mt_star) * d)),
                    {"inferred_mtbr_h": mt_star}, "informational (inference, not a claim about the paper)", True,
                    "Inference only: the paper does not state the rate; do not treat as the authors' configuration."))
    # (2) phi=1 exact Poisson tail vs MC and vs exact-pmf sum
    mt = 3.0
    lam = lam_a_from(mt)
    ntr = 200_000 if mode == "full" else 40_000
    rng = make_rngs(37, 1)[0]
    N_ev = rng.poisson(lam * T_N, ntr)
    W1 = T_N + N_ev * (d + R)
    kq = int(np.floor(lam * T_N))
    D = T_N + kq * (d + R)
    p_th = des.warning_full_usable_miss_prob(lam, T_N, D, d, R)
    # independent evaluation of the tail by direct pmf summation
    from math import exp, lgamma, log
    mean_ev = lam * T_N
    p_dir = 1.0 - sum(exp(-mean_ev + k * log(mean_ev) - lgamma(k + 1)) for k in range(0, kq + 1))
    k_miss = int(np.sum(W1 > D))
    p_sim = k_miss / ntr
    se = np.sqrt(p_th * (1 - p_th) / ntr)
    z = (p_sim - p_th) / se
    out.append(_res("P2-phi1-tail", "Prop. 2; Table VIII", f"phi=1: exact Poisson-tail miss probability vs {ntr:,} runs (D={D:.3f} h, assumed MTBR 3 h)", "RECONSTRUCTED",
                    p_th, p_sim, p_sim - p_th, f"|z| < {MC_Z}; also equals direct pmf sum to 1e-10", abs(z) < MC_Z and abs(p_dir - p_th) < 1e-10,
                    f"z={z:.2f}; paper's 0.4002 vs 0.4003 used an unstated D."))
    mean_th = des.warning_full_usable_mean(T_N, lam * R, lam * d)
    out.append(_res("P2-phi1-mean", "Prop. 2", "phi=1: E[W]=T_N(1+u+v_e)", "MONTE_CARLO", mean_th, float(W1.mean()), float(W1.mean() - mean_th),
                    f"|z| < {MC_Z}", abs((W1.mean() - mean_th) / (W1.std(ddof=1) / np.sqrt(ntr))) < MC_Z))
    # (3) event-driven simulator with warned process (independent implementation) at phi=0.9
    from ..simulation.processes import PoissonProcess
    lam_u = 0.1 * lam
    K = comp.n_chunks(T_N, lam_u, lam_u * d)
    rng = make_rngs(41, 1)[0]
    n_ev = 600 if mode == "full" else 200
    ws = []
    for _ in range(n_ev):
        W_, _ = rb.simulate_rollback_event_driven(T_N, K, R, d, PoissonProcess(lam_u, rng), rng, warned_process=PoissonProcess(0.9 * lam, rng), delta_e=de)
        ws.append(W_)
    ws = np.array(ws)
    # exact-K theory for the same K: running time chain + warned events
    th_exactK = None
    Wv, _ = rb.simulate_rollback_vectorized(T_N, lam_u, R, d, K, 100_000, make_rngs(43, 1)[0], warned_rate=0.9 * lam, delta_e=de)
    se = np.sqrt(ws.var(ddof=1) / n_ev + Wv.var(ddof=1) / 100_000)
    z = (ws.mean() - Wv.mean()) / se
    out.append(_res("P2-event-vs-vec", "Prop. 2", "explicit state-machine simulator vs vectorised simulator agree (same K), phi=0.9", "MONTE_CARLO", float(Wv.mean()), float(ws.mean()), float(ws.mean() - Wv.mean()),
                    f"|z|<{MC_Z}", abs(z) < MC_Z, f"z={z:.2f}"))
    return out


# ===================================================================== Theorem 3
def check_theorem3(mode) -> List[CheckResult]:
    out = []
    rng = np.random.default_rng(99)
    viol = 0
    n_cases = 5000
    for _ in range(n_cases):
        n_s = int(rng.integers(1, 200))
        lam_w = 10 ** rng.uniform(-4, -0.5)
        R = rng.uniform(60, 3600) / H_
        d = rng.uniform(5, 300) / H_
        s = rng.uniform(0.01, 1.0)
        u = n_s * lam_w * R
        if u > 30:
            continue
        gP = float(des.gamma_halting(lam_w, R, n_s))
        ok = (1 + u <= gP * (1 + 1e-12)) and (gP <= np.exp(u) * (1 + 1e-12)) and des.ordering_holds(s, lam_w, R, n_s, d)
        viol += (not ok)
    out.append(_res("T3-bounds", "Thm 3(b),(c); Eq. (6)", "1+u <= Gamma_P <= e^u and Gamma_E <= Gamma_P <= e^u Psi(v) over random parameters", "NUMERICAL", 0, viol, viol, "0 violations", viol == 0))
    # Gamma_E not growing with n at fixed s: compare n=10..1000
    ge = [float(des.gamma_elastic(0.5, 0.01, 0.0833)) for _ in range(3)]
    gr = [float(des.stretch_of_design("rollback", n, 0.5, 0.01, 0.0833, 0.0083)) for n in (10, 100, 1000)]
    gp = [float(des.stretch_of_design("halting", n, 0.5, 0.01, 0.0833, 0.0083)) for n in (10, 100, 1000)]
    ok = ge[0] == ge[1] == ge[2] and gr[0] < gr[1] < gr[2] and gp[0] < gp[1] < gp[2]
    out.append(_res("T3-n-scaling", "Thm 3", "at fixed s, lam_w, R: Gamma_E independent of n; Gamma_R, Gamma_P increase with n", "ANALYTICAL", True, ok, {"rollback": gr, "halting": gp, "elastic": ge[0]}, "strict ordering", ok))
    # cost factor is not a ratio of completion times
    s, rho, lw, R = 0.5, 0.35, 0.02, 0.0833
    cf = float(des.elastic_cost_factor(s, rho, lw, R))
    gE = float(des.gamma_elastic(s, lw, R))
    out.append(_res("T3-cost-vs-time", "Thm 3 / Eq. (7)", "elastic cost factor (m - s q rho)/(1 - s q) differs from the stretch Gamma_E (billing of down spot workers)", "ANALYTICAL",
                    "cost != stretch", (cf, gE), abs(cf - gE), "differ by > 1e-6", abs(cf - gE) > 1e-6))
    # Simulations (paper has none): halting & elastic vs formula, long job, stationary start
    n, s, lam_w, R, rho = 16, 0.5, 0.05, 0.25, 0.35
    ntr = 3000 if mode == "full" else 600
    T_N = 400.0
    for design in ("halting", "elastic"):
        rngs = make_rngs(53 if design == "halting" else 59, 1)[0]
        W, ca, cu = simulate_design(design, T_N, n, s, lam_w, R, rho, rngs, ntr, stationary=True)
        n_s = int(n * s)
        g = float(des.gamma_halting(lam_w, R, n_s)) if design == "halting" else float(des.gamma_elastic(s, lam_w, R))
        se = W.std(ddof=1) / np.sqrt(ntr)
        z = (W.mean() / T_N - g) / (se / T_N)
        # cost vs formula
        base = n * T_N
        if design == "halting":
            cth = float(des.halting_cost_factor(s, rho, lam_w, R, n_s))
            csim = ca.mean() / base
        else:
            cth = float(des.elastic_cost_factor(s, rho, lam_w, R))
            csim = cu.mean() / base
        out.append(_res(f"T3-sim-{design}", "Thm 3 (new: paper reports no simulation)", f"{design}: simulated stretch & cost factor vs formulas (n={n}, s={s}, T_N={T_N} h, stationary start, {ntr} runs)",
                        "MONTE_CARLO", {"stretch": g, "cost_factor": cth}, {"stretch": float(W.mean() / T_N), "cost_factor": float(csim)}, {"z_stretch": float(z)},
                        f"|z| < {MC_Z} on stretch; cost-factor relative error <= 1%", abs(z) < MC_Z and abs(csim - cth) / cth <= 0.01,
                        "Long-run formulas; first-passage of a finite job adds an O(R/T_N) boundary effect, reduced by the stationary start."))
    # Fig. 5 crossing: halting > fail-free rollback near u ~ 1.9 (n_s=16, d/R=0.1)
    f = lambda u: (1 + u / 16) ** 16 - (1 + u) * float(psi(0.1 * u))
    uc = brentq(f, 0.5, 4.0)
    out.append(_res("Fig5-crossing", "Fig. 5 caption", "halting curve crosses above fail-free rollback near u~1.9 (n_s=16, d/R=0.1)", "REPRODUCTION", ref.FIG5_HALTING_CROSSING_U, uc, uc - 1.9,
                    "|u_c - 1.9| < 0.1", abs(uc - 1.9) < 0.1))
    return out


# ===================================================================== overlap semantics (S4b)
def check_overlap(mode) -> List[CheckResult]:
    out = []
    lam, R = 4.0, 5.0 / 60
    u = lam * R
    rng = make_rngs(61, 1)[0]
    bp = rb.busy_period_md_inf(rng, 400_000, lam, R)
    th = np.expm1(u) / lam
    z = (bp.mean() - th) / (bp.std(ddof=1) / np.sqrt(bp.size))
    out.append(_res("S4b-busy", "(S4b); Lemma 1", "M/D/inf busy period mean (e^u-1)/lam_a", "MONTE_CARLO", th, float(bp.mean()), float(bp.mean() - th), f"|z|<{MC_Z}", abs(z) < MC_Z, f"z={z:.2f}"))
    T_N, d = 6.0, 30 / 3600
    lam = 3.0
    u = lam * R              # recompute: u depends on the rate used in THIS simulation
    v = lam * d
    K = comp.n_chunks(T_N, lam, v)
    w = T_N / K
    seg = lambda X: float(np.exp(u) * np.expm1(lam * X) / lam)
    th = (K - 1) * seg(w + d) + seg(w)
    ntr = 40_000 if mode == "full" else 10_000
    W, _ = rb.simulate_rollback_vectorized(T_N, lam, R, d, K, ntr, make_rngs(67, 1)[0], stall="overlap")
    z = (W.mean() - th) / (W.std(ddof=1) / np.sqrt(ntr))
    out.append(_res("S4b-mean", "Lemma 1 (S4b)", "mean completion under overlapping replacements = sum e^u (e^{lam X}-1)/lam", "MONTE_CARLO", th, float(W.mean()), float(W.mean() - th), f"|z|<{MC_Z}", abs(z) < MC_Z, f"z={z:.2f}"))
    return out


def check_event_vs_vector(mode) -> List[CheckResult]:
    """Two independent simulators of the same model must agree (guards shared coding errors)."""
    from ..simulation.processes import PoissonProcess
    out = []
    T_N, d, R, lam, K = 3.0, 30 / 3600, 300 / 3600, 2.0, 25
    rv, re_ = make_rngs(71, 2)
    n_ev = 3000 if mode == "full" else 800
    for stall in ("fixed", "overlap"):
        Wv, _ = rb.simulate_rollback_vectorized(T_N, lam, R, d, K, 100_000, rv, stall=stall)
        We = np.array([rb.simulate_rollback_event_driven(T_N, K, R, d, PoissonProcess(lam, re_), re_, stall=stall, lam_for_stall=lam)[0] for _ in range(n_ev)])
        se = np.sqrt(We.var(ddof=1) / n_ev + Wv.var(ddof=1) / Wv.size)
        z = (We.mean() - Wv.mean()) / se
        out.append(_res(f"SIM-xval-{stall}", "Sec. 'Simulation framework'", f"event-driven state machine vs vectorised segment simulator ({stall} stall)", "MONTE_CARLO", float(Wv.mean()), float(We.mean()),
                        float(We.mean() - Wv.mean()), f"|z|<{MC_Z}", abs(z) < MC_Z, f"z={z:.2f}"))
    # per-worker exponential renewal == aggregate Poisson
    from ..simulation.processes import make_process
    n_s, mt = 8, 3.0
    lam_a = n_s / mt
    r1 = make_rngs(73, 1)[0]
    Wc = np.array([rb.simulate_rollback_event_driven(T_N, K, R, d, make_process("exponential", n_s, mt, r1), r1)[0] for _ in range(n_ev)])
    Wa, _ = rb.simulate_rollback_vectorized(T_N, lam_a, R, d, K, 100_000, make_rngs(79, 1)[0])
    se = np.sqrt(Wc.var(ddof=1) / n_ev + Wa.var(ddof=1) / Wa.size)
    z = (Wc.mean() - Wa.mean()) / se
    out.append(_res("SIM-xval-perworker", "(S1)", "per-worker renewal processes (n_s=8) vs aggregate Poisson (aggregation principle)", "MONTE_CARLO", float(Wa.mean()), float(Wc.mean()), float(Wc.mean() - Wa.mean()), f"|z|<{MC_Z}", abs(z) < MC_Z, f"z={z:.2f}"))
    return out


def check_fig_claims(mode) -> List[CheckResult]:
    """Numerical statements in the text of Sec. IX-A about Fig. 6 (deadline multiple)."""
    from ..experiments import tables as T
    df = T.deadline_multiple_curves(n=60)
    lo = df[(df.u >= 0.0099) & (df.u <= 0.0105)]
    hi = df[(df.u >= 0.095) & (df.u <= 0.105)]
    near03 = df[(df.u >= 0.28)]
    r1 = (float(lo.cantelli_over_T.min()), float(hi.cantelli_over_T.max()))
    c1 = 1.1 <= r1[0] and r1[1] <= 1.85
    out = [_res("Fig6-text", "Sec. IX-A; Fig. 6", "deadline multiple 'about 1.2-1.8 at u=0.01-0.1 depending on T_N/R, above 2 near u=0.3'", "REPRODUCTION",
                "1.2-1.8 (u=0.01-0.1); >2 near 0.3", {"u=0.01-0.1 range": r1, "u~0.3 max over T_N/R": float(near03.cantelli_over_T.max()), "u~0.3 min": float(near03.cantelli_over_T.min())}, None,
                "range lies within [1.1,1.85] and the T_N/R=30 curve exceeds 2 near u=0.3", c1 and float(near03.cantelli_over_T.max()) > 2.0,
                "T_N/R=300 stays slightly below 2 at u=0.3 (about 1.93): 'above 2' holds only for shorter jobs.")]
    return out


# ===================================================================== registry
ALL_CHECKS: Dict[str, Callable] = {
    "prop1": check_prop1, "lemma1": check_lemma1, "theorem1": check_theorem1, "theorem2": check_theorem2,
    "remark1": check_remark1, "table3": check_table3, "table4_6": check_table4_6, "appendix_a": check_appendix_a,
    "fig_claims": check_fig_claims, "prop2": check_prop2, "theorem3": check_theorem3, "overlap": check_overlap, "sim_crossval": check_event_vs_vector,
}


def run_checks(mode="quick", only=None, log=print) -> List[CheckResult]:
    out: List[CheckResult] = []
    for name, fn in ALL_CHECKS.items():
        if only and name not in only:
            continue
        log(f"[verify] {name} ({mode})")
        out.extend(fn(mode))
    return out


In [ ]:
%%writefile src/gnn_spot/verification/reference.py
"""Numbers PRINTED in the paper (GNN.docx), transcribed for comparison only.

They are never used to generate tables/figures; `checks.py` compares computed values against them.
Transcription source: the supplied GNN.docx (PDF-style export; equations illegible but tables/text readable).
"""

# Table III: relative excess cost (%) from per-worker-rate interval; keys n_s -> {v: %}
TABLE_III = {
    4:  {0.01: 3.8,  0.05: 9.3,   0.2: 21.7,   0.5: 41.0},
    8:  {0.01: 9.3,  0.05: 23.7,  0.2: 61.0,   0.5: 131.0},
    24: {0.01: 26.9, 0.05: 78.9,  0.2: 278.0,  0.5: 945.0},
    64: {0.01: 62.3, 0.05: 237.0, 0.2: 1577.0, 0.5: 13797.0},
}

# Table IV: (m, delta/R) -> (first-order, exact S4a, exact S4b)
TABLE_IV = {
    (2/3, 0.1): (0.27, 0.21, 0.20),
    (2/3, 1.0): (0.086, 0.055, 0.055),
    (1/3, 0.1): (1.46, 0.90, 0.70),
    (1/3, 1.0): (0.76, 0.28, 0.27),
    (0.14, 0.1): (5.13, 2.32, 1.39),
    (0.14, 1.0): (3.50, 0.68, 0.61),
}

# Table VI: u = n_s lam_w R at lam_w = 3.1e-4 /h; R in minutes
TABLE_VI = {
    5:  {64: 0.0016, 256: 0.0065, 1024: 0.026, "n_at_ustar": 35000},
    15: {64: 0.0049, 256: 0.020,  1024: 0.078, "n_at_ustar": 11800},
    30: {64: 0.0098, 256: 0.039,  1024: 0.157, "n_at_ustar": 5900},
    60: {64: 0.020,  256: 0.078,  1024: 0.313, "n_at_ustar": 2900},
}

# Table IX: MTBR (h) -> (K, mean_th, mean_sim, std_th, std_sim);  T_N=7.40 h, n_s=8, delta=30 s, R=300 s
TABLE_IX = {
    12: (48, 8.686, 8.684, 0.418, 0.420),
    6:  (70, 9.570, 9.570, 0.520, 0.522),
    3:  (101, 11.241, 11.243, 0.687, 0.688),
    2:  (125, 12.908, 12.903, 0.832, 0.829),
}

# Text claims
CANTELLI_MULTIPLIER_5PCT = 4.36
YOUNG_REGRET_V = {0.01: 0.17, 0.05: 0.50, 0.10: 0.79}      # regret >1%,5%,10% at v ~ these
DALY_MAX_REGRET_PCT = 0.19                                  # over v in [1e-4, 1e2]
APP_A_EXAMPLE = {   # MTBR 3 h, T_N=7.40: D = f*mu -> (sim miss, Chernoff, Cantelli)
    1.10: (0.058, 0.311, 0.272),
    1.25: (0.0002, 0.0020, 0.056),
}
PROP2_REPORTED = {"phi0.9_mean_theory_h": 9.072, "phi0.9_mean_sim_h": 9.066,
                  "phi1_miss_theory": 0.4002, "phi1_miss_sim": 0.4003}
THM2_REPORTED = {"mtbr_h": 3.0, "n_opt": 21, "n_D": 9}
ROBUSTNESS_REPORTED = {"exp_stretch_mtbr3": 1.98, "burst_stretch_mtbr3": 1.35,
                       "weibull0.6_increase_pct": (11, 13), "weibull1.5_change_pct": (-6, -4),
                       "n_configs": 32}
FIG5_HALTING_CROSSING_U = 1.9
SECTION_IX = {"lambda_w_per_h": 3.1e-4, "mtbr_below_h": 3300, "margin_range": (3, 550),
              "stress_mtbr_h": 10, "factor_more_frequent": 300}


In [ ]:
%%writefile tests/__init__.py



In [ ]:
%%writefile tests/conftest.py
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))


In [ ]:
%%writefile tests/integration/test_pipeline.py
"""Integration: config -> verification -> tables -> figures -> report, all in a scratch results dir."""
import json, os, subprocess, sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parents[2]


def _run(args, tmp, check=True):
    env = dict(os.environ, GNN_SPOT_RESULTS=str(tmp / "results"), PYTHONPATH=str(ROOT / "src"))
    return subprocess.run([sys.executable, "-m", "gnn_spot", *args], cwd=ROOT, env=env, capture_output=True, text=True, check=check)


def test_cli_help():
    r = subprocess.run([sys.executable, "-m", "gnn_spot", "--help"], cwd=ROOT, env=dict(os.environ, PYTHONPATH=str(ROOT / "src")), capture_output=True, text=True)
    assert r.returncode == 0 and "reproduce" in r.stdout


def test_cli_rejects_bad_mode():
    r = _run(["verify", "--mode", "bogus"], Path("/tmp/gnn_spot_it"), check=False)
    assert r.returncode == 2 or "invalid choice" in r.stderr


def test_verify_subset_tables_figures_report(tmp_path):
    r = _run(["verify", "--mode", "quick", "--only", "prop1", "lemma1", "table3", "table4_6", "remark1", "appendix_a"], tmp_path, check=False)
    out = json.load(open(tmp_path / "results" / "raw" / "verification_quick.json"))
    assert len(out["results"]) >= 20
    assert all(x["status"] == "PASS" for x in out["results"]), [x["id"] for x in out["results"] if x["status"] != "PASS"]
    assert r.returncode == 0
    _run(["tables", "--mode", "quick"], tmp_path)
    for name in ("table3_per_worker_rate_excess", "table4_breakeven_u", "table6_stall_load", "table9_completion_moments"):
        assert (tmp_path / "results" / "tables" / f"{name}.csv").stat().st_size > 50
    _run(["figures", "--mode", "quick"], tmp_path)
    figs = tmp_path / "results" / "figures"
    for name in ("fig3_scaleout_regime_map", "fig4_breakeven_boundary", "fig5_recovery_design_stretch", "fig6_deadline_multiple"):
        assert (figs / f"{name}.png").exists() and (figs / f"{name}.pdf").exists() and (figs / "data" / f"{name}.csv").exists()
    _run(["report", "--mode", "quick"], tmp_path)
    assert (tmp_path / "docs" / "VALIDATION_REPORT.md").read_text().startswith("# Validation report")
    assert (tmp_path / "paper" / "reproduction_report.md").exists()


def test_simulate_command(tmp_path):
    r = _run(["simulate", "--trials", "2000", "--seed", "3"], tmp_path)
    d = json.loads(r.stdout)
    assert d["n_trials"] == 2000 and abs(d["mean"] - d["analytical_S4a"]["mean"]) < 5 * d["sem"]


def test_manifest_written(tmp_path):
    _run(["verify", "--mode", "quick", "--only", "lemma1"], tmp_path, check=False)
    m = json.load(open(tmp_path / "results" / "manifests" / "verification_quick.json"))
    assert m["environment"]["numpy"] and "mode_config" in m["config"]


def test_config_validation(tmp_path):
    from gnn_spot.config import load_baseline, ConfigError, load_mode
    cfg = load_baseline()
    assert cfg["_R_h"] == pytest.approx(300 / 3600)
    with pytest.raises(ConfigError):
        load_baseline("does_not_exist.yaml")
    with pytest.raises(ConfigError):
        load_mode("turbo")
    bad = tmp_path / "bad.yaml"
    bad.write_text("perf: {a: 1}\nspot: {R_s: 300, delta_s: 30, rho: 1.5, stall: fixed}\n")
    with pytest.raises(ConfigError):
        load_baseline(str(bad))


In [ ]:
%%writefile tests/property/test_properties.py
import numpy as np
from hypothesis import given, settings, strategies as st
from gnn_spot.models.lambert import psi, log_psi, x_star
from gnn_spot.models import spot
from gnn_spot.checkpointing import baselines as bl
from gnn_spot.recovery import designs as D
from scipy.optimize import minimize_scalar

v_st = st.floats(1e-4, 5.0)
pos = st.floats(1e-3, 10.0)


@given(v_st)
@settings(max_examples=60, deadline=None)
def test_closed_form_equals_numerical_minimisation(v):
    r = minimize_scalar(lambda lx: bl.reduced_overhead(np.exp(lx), v), bounds=(np.log(1e-6), np.log(50)), method="bounded", options={"xatol": 1e-12})
    assert abs(r.fun - psi(v)) / psi(v) < 1e-8


@given(v_st, st.floats(1e-4, 1.0))
@settings(max_examples=60, deadline=None)
def test_psi_strictly_increasing_and_ge_one(v, dv):
    assert psi(v) >= 1 and psi(v + dv) > psi(v) and 0 < x_star(v) < 1


@given(v_st, v_st)
@settings(max_examples=60, deadline=None)
def test_ln_psi_concave_midpoint(a, b):
    assert log_psi((a + b) / 2) >= (log_psi(a) + log_psi(b)) / 2 - 1e-12


@given(st.integers(2, 300), st.floats(1e-4, 0.5), st.floats(0.01, 1.0), st.floats(0.01, 0.5), st.floats(0.01, 0.9), st.sampled_from(["fixed", "overlap"]), st.floats(0, 100))
@settings(max_examples=80, deadline=None)
def test_cost_factor_log_concave_in_s_and_endpoint_optimal(n, lw, Rm, dm, rho, stall, z):
    R, d = Rm / 1.0, dm / 6.0
    s = np.linspace(0, 1, 101)
    with np.errstate(over="ignore"):
        lf = spot.log_cost_factor(n, s, rho, lw, R, d, stall, z)
    if not np.all(np.isfinite(lf)):
        return
    assert np.all(np.diff(lf, 2) < 1e-9 * (1 + np.abs(lf[1:-1])))
    i = int(np.argmin(lf))
    # interior minimiser impossible for strictly concave function (allow flat numerical ties)
    assert i in (0, len(s) - 1) or abs(lf[i] - min(lf[0], lf[-1])) < 1e-9


@given(st.integers(1, 500), st.floats(1e-4, 0.3), st.floats(0.01, 2.0))
@settings(max_examples=80, deadline=None)
def test_halting_between_bounds(n_s, lw, R):
    u = n_s * lw * R
    if u > 50:
        return
    g = D.gamma_halting(lw, R, n_s)
    assert 1 + u <= g * (1 + 1e-12) and g <= np.exp(u) * (1 + 1e-12)


@given(st.floats(1e-3, 20.0), v_st)
@settings(max_examples=60, deadline=None)
def test_young_first_order_below_exact(u, v):
    assert bl.young_first_order_overhead(u, v) < spot.Lambda(u, "fixed") * psi(v)


@given(st.floats(0.05, 0.95), st.floats(0.05, 5.0), st.sampled_from(["fixed", "overlap"]))
@settings(max_examples=40, deadline=None)
def test_breakeven_unique_and_ordering(m, dr, stall):
    u = spot.breakeven_u(m, dr, stall)
    f = lambda x: m * spot.Lambda(x, stall) * psi(x * dr)
    assert f(u * 0.9) < 1 < f(u * 1.1) and u <= spot.breakeven_u(m, dr, "fixed") + 1e-12


@given(st.floats(0.0, 1.0), st.floats(1e-3, 0.3), st.floats(0.01, 2.0), st.floats(0.01, 0.9))
@settings(max_examples=60, deadline=None)
def test_costs_nonnegative_and_finite(s, lw, R, rho):
    for f in (D.elastic_cost_factor(s, rho, lw, R), D.halting_cost_factor(s, rho, lw, R, 8)):
        assert np.isfinite(f) and f > 0


@given(st.integers(4, 200), st.floats(1e-4, 0.2), st.floats(0.01, 0.5), st.floats(0.01, 0.5), st.floats(0.05, 0.9), st.floats(0.5, 1.0))
@settings(max_examples=80, deadline=None)
def test_if_spot_pays_at_half_it_pays_more_at_larger_share(n, lw, R, d, rho, s_bar):
    """Corollary of Thm 1(a) (Discussion, 'benchmark corners'): F(0.5)<1  =>  F(s_bar) < F(0.5) for s_bar >= 0.5."""
    F = lambda s: spot.cost_factor(n, s, rho, lw, R, d / 6.0)
    if F(0.5) < 1.0 and s_bar > 0.5 + 1e-9:
        assert F(s_bar) < F(0.5) * (1 + 1e-12)


In [ ]:
%%writefile tests/regression/frozen_values.json
{
 "psi": {
  "0.0001": 1.0142764971895546,
  "0.01": 1.155848551745576,
  "0.1": 1.6212268350317864,
  "0.5": 3.3144458236686756,
  "1.0": 6.305395279271691,
  "5.0": 402.4275500032906
 },
 "x_star": {
  "0.0001": 0.014075547672763032,
  "0.01": 0.13483475106683465,
  "0.1": 0.38318316820829484,
  "0.5": 0.698290437315664,
  "1.0": 0.8414056604369606,
  "5.0": 0.9975150806648505
 },
 "breakeven_fixed": {
  "0.6666666666666666,0.1": 0.21197376590903227,
  "0.6666666666666666,1.0": 0.05520113374758747,
  "0.3333333333333333,0.1": 0.9003131566697365,
  "0.3333333333333333,1.0": 0.27890825302058614,
  "0.14,0.1": 2.3159756053133833,
  "0.14,1.0": 0.6817313961627574
 },
 "table_ix": {
  "12": [
   48,
   8.68608031497153,
   0.4177723840658736,
   10.507107918516734
  ],
  "6": [
   70,
   9.569605354860622,
   0.5197789101848599,
   11.83526909734013
  ],
  "3": [
   101,
   11.240608666428018,
   0.6869284216609655,
   14.234860237894063
  ],
  "2": [
   125,
   12.908146356225474,
   0.8318413468182492,
   16.53405872406499
  ]
 }
}


In [ ]:
%%writefile tests/regression/test_paper_values.py
"""Regression tests: values printed in the paper (tests/regression) and frozen computed values."""
import json
from pathlib import Path
import numpy as np, pytest
from gnn_spot.verification import reference as ref
from gnn_spot.checkpointing import baselines as bl
from gnn_spot.models import spot
from gnn_spot.models.lambert import psi, x_star

FROZEN = json.load(open(Path(__file__).with_name("frozen_values.json")))


def test_table3_within_rounding():
    for n_s, row in ref.TABLE_III.items():
        for v, pct in row.items():
            assert 100 * float(bl.per_worker_rate_excess(n_s, v)) == pytest.approx(pct, rel=0.004)


def test_table4_to_printed_precision():
    for (m, dr), vals in ref.TABLE_IV.items():
        got = (spot.breakeven_u_young(m, dr), spot.breakeven_u(m, dr, "fixed"), spot.breakeven_u(m, dr, "overlap"))
        for g, p in zip(got, vals):
            assert g == pytest.approx(p, abs=0.5 * 10 ** (-(2 if p >= 0.1 else 3)) * 1.01)


def test_remark1_thresholds():
    from scipy.optimize import brentq
    for t, v in ref.YOUNG_REGRET_V.items():
        assert brentq(lambda x: bl.young_regret(x) - t, 1e-3, 5) == pytest.approx(v, abs=0.006)
    vv = np.geomspace(1e-4, 1e2, 50000)
    assert 100 * bl.daly_regret(vv).max() == pytest.approx(0.19, abs=0.005)


def test_frozen_special_function_values():
    for k, val in FROZEN["psi"].items():
        assert float(psi(float(k))) == pytest.approx(val, rel=1e-12)
    for k, val in FROZEN["x_star"].items():
        assert float(x_star(float(k))) == pytest.approx(val, rel=1e-12)
    for k, val in FROZEN["breakeven_fixed"].items():
        m, dr = map(float, k.split(","))
        assert spot.breakeven_u(m, dr, "fixed") == pytest.approx(val, rel=1e-10)


def test_frozen_table_ix_theory():
    from gnn_spot.deadlines.completion import completion_moments, cantelli_deadline
    for k, (K, mean, std, D5) in FROZEN["table_ix"].items():
        m = completion_moments(7.40, 8 / float(k), 300 / 3600, 30 / 3600)
        assert m.K == K and m.mean == pytest.approx(mean, rel=1e-10) and m.std == pytest.approx(std, rel=1e-10)
        assert cantelli_deadline(m.mean, m.var) == pytest.approx(D5, rel=1e-10)


def test_table_vi_and_section_ix():
    lam = -np.log(0.8) / 730            # exact rate; printed 3.1e-4 is its rounding
    assert lam == pytest.approx(3.1e-4, abs=0.06e-4)
    assert 64 * lam * (5 / 60) == pytest.approx(0.0016, abs=5e-5)     # 0.00163 -> prints 0.0016
    assert 1024 * lam * (60 / 60) == pytest.approx(0.313, abs=5e-4)
    assert 0.9 / (lam * 5 / 60) == pytest.approx(35000, rel=0.02)


In [ ]:
%%writefile tests/unit/test_completion.py
import numpy as np, pytest
from gnn_spot.deadlines import completion as C


def test_laplace_normalised():
    assert C.segment_laplace(0.0, 0.7, 2.0, 0.08) == pytest.approx(1.0, rel=1e-12)


@pytest.mark.parametrize("X,lam,R", [(0.3, 1.0, 0.08), (1.2, 3.0, 0.083), (0.05, 10.0, 0.5)])
def test_moments_vs_laplace_derivatives(X, lam, R):
    h = 1e-4 / max(1.0, lam)
    phi = lambda t: C.segment_laplace(t, X, lam, R)
    m1 = -(phi(h) - phi(-h)) / (2 * h)
    m2 = (phi(h) - 2 * phi(0.0) + phi(-h)) / h**2
    assert C.segment_mean(X, lam, R) == pytest.approx(m1, rel=1e-5)
    assert C.segment_var(X, lam, R) == pytest.approx(m2 - m1**2, rel=1e-4)


def test_failure_free_limit():
    assert C.segment_mean(0.5, 1e-9, 0.1) == pytest.approx(0.5, rel=1e-6)
    assert C.completion_moments(3.0, 0.0, 0.1, 0.01).var == 0.0


def test_table_ix_theory():
    d, R = 30 / 3600, 300 / 3600
    for mt, (K, mu, _, sd, _) in {12: (48, 8.686, 0, 0.418, 0), 6: (70, 9.570, 0, 0.520, 0), 3: (101, 11.241, 0, 0.687, 0), 2: (125, 12.908, 0, 0.832, 0)}.items():
        m = C.completion_moments(7.40, 8 / mt, R, d)
        assert m.K == K and m.mean == pytest.approx(mu, abs=6e-4) and m.std == pytest.approx(sd, abs=6e-4)


def test_cantelli():
    assert C.cantelli_multiplier(0.05) == pytest.approx(4.3589, abs=1e-4)
    mu, var = 10.0, 4.0
    D = C.cantelli_deadline(mu, var, 0.05)
    assert C.cantelli_bound(D, mu, var) == pytest.approx(0.05)
    assert C.cantelli_bound(mu, mu, var) == 1.0


def test_chernoff_conservative_and_tighter_far_out():
    d, R, T, lam = 30 / 3600, 300 / 3600, 7.4, 8 / 3.0
    m = C.completion_moments(T, lam, R, d)
    assert C.chernoff_bound(1.25 * m.mean, T, lam, R, d) < C.cantelli_bound(1.25 * m.mean, m.mean, m.var)
    assert C.chernoff_bound(m.mean * 0.9, T, lam, R, d) == 1.0
    assert 0 < C.chernoff_tc(0.3, lam, R) < np.inf


In [ ]:
%%writefile tests/unit/test_deadlines.py
import pytest
from gnn_spot.deadlines import feasibility as F


def test_infeasible_when_TN_exceeds_D():
    assert F.largest_feasible_share(8, 10.0, 9.0, "rollback", 0.1, 0.08, 0.008) is None


def test_share_is_boundary():
    s = F.largest_feasible_share(8, 8.0, 10.0, "rollback", 0.5, 0.08, 0.008)
    from gnn_spot.recovery.designs import stretch_of_design
    assert 8.0 * stretch_of_design("rollback", 8, s, 0.5, 0.08, 0.008) == pytest.approx(10.0, rel=1e-9)


def test_all_spot_when_slack_large():
    assert F.largest_feasible_share(4, 1.0, 100.0, "elastic", 0.01, 0.08, 0.008) == 1.0


def test_corner_decision_is_endpoint():
    d = F.corner_decision(32, 1.0, 1.5, "rollback", 1 / 12, 300 / 3600, 30 / 3600, 0.35)
    assert d["feasible"] and d["s_opt"] in (0.0, d["s_bar"]) and d["s_bar"] <= 1


def test_mean_feasible_is_not_tail_safe():
    """Mean-deadline feasibility does NOT imply Cantelli (5% miss budget) feasibility (paper Sec. VII)."""
    T, lw, R, d = 7.4, 1 / 3.0, 300 / 3600, 30 / 3600
    D = 1.3 * T
    s_bar = F.largest_feasible_share(8, T, D, "rollback", lw, R, d)
    n_s = int(8 * s_bar + 1e-9)
    assert n_s == 4                                   # mean deadline allows 4 of 8 spot workers
    feas, rhs = F.cantelli_feasible_counts(8, T, D, lw, R, d)
    assert rhs[n_s] > D and list(feas) == [0]         # but the 5% tail rule allows none


In [ ]:
%%writefile tests/unit/test_designs.py
import numpy as np, pytest
from gnn_spot.recovery import designs as D
from gnn_spot.models.lambert import psi


def test_halting_bounds():
    for n_s, lw, R in [(4, 0.01, 0.08), (64, 0.002, 0.5), (1, 0.3, 0.1)]:
        u = n_s * lw * R
        g = D.gamma_halting(lw, R, n_s)
        assert 1 + u <= g + 1e-12 and g <= np.exp(u) + 1e-12


def test_elastic_closed_form():
    s, lw, R = 0.5, 0.05, 0.25
    q = lw * R / (1 + lw * R)
    assert D.gamma_elastic(s, lw, R) == pytest.approx(1 / (1 - s * q))
    assert D.gamma_elastic(s, lw, R) == pytest.approx(1 + s * lw * R, rel=0.01)   # ~ 1 + u/n


def test_elastic_cost_billing_variants():
    s, rho, lw, R = 0.5, 0.35, 0.05, 0.25
    q = D.down_fraction(lw, R)
    assert D.elastic_cost_factor(s, rho, lw, R) == pytest.approx((1 - (1 - rho) * s - s * q * rho) / (1 - s * q))
    assert D.elastic_cost_factor(s, rho, lw, R, bill_down_spot=True) > D.elastic_cost_factor(s, rho, lw, R)


def test_ordering_theorem3c():
    assert D.ordering_holds(0.5, 0.05, 0.25, 16, 0.01)


def test_size_scaling():
    g = lambda d, n: D.stretch_of_design(d, n, 0.5, 0.01, 0.08, 0.008)
    assert g("elastic", 10) == g("elastic", 1000)
    assert g("rollback", 10) < g("rollback", 100) and g("halting", 10) < g("halting", 100)


def test_unknown_design():
    with pytest.raises(ValueError):
        D.stretch_of_design("x", 10, 0.5, 0.01, 0.08, 0.008)


def test_warnings_limits():
    u, v = 0.3, 0.03
    assert D.warning_overhead(u, v, 0.0, v) == pytest.approx((1 + u) * psi(v))      # no warnings -> full rollback
    assert D.warning_overhead(u, v, 1.0, v) == pytest.approx(1 + u + v)              # all warned -> no periodic ckpt
    assert D.warning_overhead(u, v, 0.9, v) < D.warning_overhead(u, v, 0.0, v)


def test_phi1_tail_exact():
    from scipy.stats import poisson
    lam, T, de, R = 2.0, 7.4, 30 / 3600, 300 / 3600
    D_ = T + 10 * (de + R)
    assert D.warning_full_usable_miss_prob(lam, T, D_, de, R) == pytest.approx(1 - poisson.cdf(10, lam * T))


def test_exact_K_mean_reduces_to_formula_for_large_K():
    T, lam, phi, R, d = 7.4, 2.667, 0.9, 300 / 3600, 30 / 3600
    lam_u = (1 - phi) * lam
    from gnn_spot.models.lambert import x_star
    K = int(round(T * lam_u / float(x_star(lam_u * d))))
    exact = D.warning_mean_exact_K(T, lam, phi, R, d, d, K)
    form = T * D.warning_overhead(lam * R, lam * d, phi, lam * d)
    assert exact == pytest.approx(form, rel=0.01)


In [ ]:
%%writefile tests/unit/test_lambert.py
import numpy as np, pytest, mpmath as mp
from gnn_spot.models.lambert import psi, x_star, log_psi, dlog_psi_dv, psi_lambertw, x_star_lambertw


def test_zero_failure_limit():
    assert psi(0.0) == 1.0 and x_star(0.0) == 0.0


@pytest.mark.parametrize("v", [1e-4, 1e-2, 0.1, 0.5, 1.0, 5.0])
def test_matches_lambertw_where_conditioned(v):
    assert psi(v) == pytest.approx(psi_lambertw(v), rel=1e-9)
    assert x_star(v) == pytest.approx(x_star_lambertw(v), rel=1e-8, abs=1e-12)


@pytest.mark.parametrize("v", [1e-12, 1e-9, 1e-6, 1e-3, 3.0, 50.0])
def test_against_60_digit_reference(v):
    mp.mp.dps = 60
    L = mp.findroot(lambda L: L - 1 + mp.e ** (-L) - mp.mpf(v), mp.sqrt(2 * mp.mpf(v)) if v < 1 else mp.mpf(v) + 1)
    assert float(log_psi(v)) == pytest.approx(float(L), rel=1e-13)


def test_optimality_equation():
    v = 0.37
    x = float(x_star(v))
    assert (1 - x) * np.exp(x + v) == pytest.approx(1.0, rel=1e-12)


def test_derivative_of_ln_psi():
    v, h = 0.8, 1e-6
    num = (np.log(psi(v + h)) - np.log(psi(v - h))) / (2 * h)
    assert dlog_psi_dv(v) == pytest.approx(num, rel=1e-6)
    assert dlog_psi_dv(1e-10) > 1e4          # diverges at 0+


def test_invalid_inputs():
    with pytest.raises(ValueError):
        log_psi(-0.1)
    with pytest.raises(ValueError):
        log_psi(np.nan)


def test_vectorised_shape():
    v = np.geomspace(1e-4, 5, 7)
    assert psi(v).shape == (7,) and np.all(np.diff(psi(v)) > 0)


In [ ]:
%%writefile tests/unit/test_performance.py
import numpy as np, pytest
from gnn_spot.models import performance as P

PG = P.PerfParams(a=3600, theta=0.9, A=500, kappa0=0.3, gamma=0.5, sigma=20, alpha=0.5)


def test_shares_sum_to_one():
    assert sum(P.time_shares(10.0, PG)) == pytest.approx(1.0)


@pytest.mark.parametrize("n", [2.0, 5.0, 40.0, 150.0])
def test_elasticity_matches_numeric(n):
    assert P.elasticity(n, PG) == pytest.approx(P.elasticity_numeric(n, PG), abs=1e-7)


def test_elasticity_closed_form_shares():
    n = 5.0                                           # kappa0 n^gamma = 0.67 < 1: uncapped region
    pc, pf, ps, pl = P.time_shares(n, PG)
    assert float(P.elasticity(n, PG)) == pytest.approx(PG.theta * pc + (1 - PG.gamma) * pf - (pf + ps) / (n - 1) - pl, rel=1e-10)


def test_dJ_formula():
    n, h = 17.0, 1e-5
    num = (P.J(n + h, PG, 1, 40) - P.J(n - h, PG, 1, 40)) / (2 * h)
    assert P.dJ_dn(n, PG, 1, 40) == pytest.approx(num, rel=1e-6)


def test_prop1c_closed_forms():
    a, sg, z = 3600.0, 20.0, 40.0
    p = P.PerfParams(a=a, sigma=sg)
    b = a - sg
    ns = P.n_star_closed_form(b, sg, z)
    assert ns == pytest.approx(np.sqrt(z * b / sg))
    for n in (5.0, 50.0, 200.0):
        assert P.J(n, p, 1, z) - P.j_min_closed_form(b, sg, z) == pytest.approx(P.j_regret_closed_form(n, b, sg, z), rel=1e-12)
    assert P.j_regret_closed_form(ns, b, sg, z) == pytest.approx(0, abs=1e-9)


def test_root_equals_closed_form_theta1():
    p = P.PerfParams(a=3600, sigma=20)
    assert P.cost_optimal_n_root(p, 40.0) == pytest.approx(P.n_star_closed_form(3580, 20, 40), rel=1e-8)


def test_deadline_min():
    nD = P.deadline_min_workers(3600, 20, 100 * (20 + 3580 / 9), 100)
    p = P.PerfParams(a=3600, sigma=20)
    assert nD == pytest.approx(9.0, rel=1e-12)
    assert 100 * P.T(nD, p) == pytest.approx(100 * (20 + 3580 / 9), rel=1e-9)
    assert P.deadline_min_workers(3600, 20, 10, 1) == float("inf")


def test_validation_errors():
    with pytest.raises(ValueError):
        P.PerfParams(a=-1)
    with pytest.raises(ValueError):
        P.PerfParams(a=1, gamma=1.0)
    with pytest.raises(ValueError):
        P.T(0.5, PG)


def test_n_equals_one_finite():
    assert np.isfinite(P.elasticity(1.0, PG)) and P.T(1.0, PG) == pytest.approx(PG.a + PG.alpha)


In [ ]:
%%writefile tests/unit/test_simulation.py
import numpy as np, pytest
from gnn_spot.simulation import rollback as RB, processes as PR
from gnn_spot.simulation.results import make_rngs, summarize, wilson_interval
from gnn_spot.simulation.designs_sim import simulate_design
from gnn_spot.deadlines.completion import completion_moments

T, D_, R = 3.0, 30 / 3600, 300 / 3600


def test_determinism():
    a, _ = RB.simulate_rollback_vectorized(T, 2.0, R, D_, 20, 500, make_rngs(1, 1)[0])
    b, _ = RB.simulate_rollback_vectorized(T, 2.0, R, D_, 20, 500, make_rngs(1, 1)[0])
    assert np.array_equal(a, b)


def test_independent_streams_differ():
    r = make_rngs(5, 2)
    assert r[0].random() != r[1].random()


def test_no_failures_equals_ideal():
    W, info = RB.simulate_rollback_vectorized(T, 0.0, R, D_, 10, 50, make_rngs(2, 1)[0])
    assert np.allclose(W, T + 9 * D_)           # K-1 checkpoint writes, none after the last chunk
    assert info["n_revocations"].sum() == 0


def test_event_driven_no_failures():
    rng = make_rngs(3, 1)[0]
    w, st = RB.simulate_rollback_event_driven(T, 10, R, D_, PR.PoissonProcess(0.0, rng), rng)
    assert w == pytest.approx(T + 9 * D_) and st["n_events"] == 0


def test_W_at_least_ideal_and_state_machine_consistency():
    rng = make_rngs(4, 1)[0]
    for _ in range(50):
        w, st = RB.simulate_rollback_event_driven(T, 12, R, D_, PR.PoissonProcess(3.0, rng), rng)
        assert w >= T + 11 * D_ + st["n_events"] * R - 1e-9           # every revocation costs at least R of stall
        assert w == pytest.approx(T + 11 * D_ + st["n_events"] * R + st["running_time"] - (T + 11 * D_), rel=1e-9)  # wall = running + stalls


def test_mean_matches_theory_vectorised():
    lam, K = 2.0, 25
    W, _ = RB.simulate_rollback_vectorized(T, lam, R, D_, K, 60000, make_rngs(7, 1)[0])
    m = completion_moments(T, lam, R, D_, K)
    assert abs(W.mean() - m.mean) < 4 * W.std() / np.sqrt(W.size)
    assert abs(W.std() - m.std) / m.std < 0.03


def test_busy_period_mean():
    lam, Rr = 4.0, 0.1
    bp = RB.busy_period_md_inf(make_rngs(9, 1)[0], 200000, lam, Rr)
    assert abs(bp.mean() - np.expm1(lam * Rr) / lam) < 4 * bp.std() / np.sqrt(bp.size)
    assert bp.min() >= Rr


def test_weibull_mean_scaling():
    rng = make_rngs(11, 1)[0]
    draw = PR.weibull_draw(5.0, 0.6)
    x = np.array([draw(rng) for _ in range(200000)])
    assert abs(x.mean() - 5.0) < 4 * x.std() / np.sqrt(x.size)


def test_burst_same_mean_worker_rate_fewer_events():
    rng = make_rngs(13, 1)[0]
    p = PR.BurstProcess(16, 0.1, 4, rng)
    assert p.rate == pytest.approx(16 * 0.1 / 4) and p.consume(0.0, rng) == 4
    assert PR.BurstProcess(2, 0.1, 4, rng).g == 2          # group capped at n_s


def test_halting_elastic_limits():
    rng = make_rngs(17, 1)[0]
    W, ca, cu = simulate_design("elastic", 50.0, 10, 0.0, 0.1, 0.2, 0.35, rng, 5)
    assert np.allclose(W, 50.0) and np.allclose(ca, 10 * 50.0)
    W, _, _ = simulate_design("halting", 100.0, 8, 0.5, 1e-6, 0.2, 0.35, rng, 5)
    assert np.allclose(W, 100.0, rtol=1e-3)


def test_summarize_and_wilson():
    r = summarize(np.arange(100.0), {"a": 1}, 3, D=89.5, cost=np.ones(100))
    assert r.n_trials == 100 and r.miss_prob == pytest.approx(0.10) and r.ci95[0] < r.mean < r.ci95[1] and r.cost_mean == 1.0
    lo, hi = wilson_interval(10, 100)
    assert lo < 0.1 < hi


In [ ]:
%%writefile tests/unit/test_slack.py
import numpy as np, pytest
from gnn_spot.optimization import slack as S
from gnn_spot.models.performance import PerfParams

R, D_, RHO = 300 / 3600, 30 / 3600, 0.35
JOB = S.Job(PerfParams(a=3600, sigma=20), 100.0, 100 * (20 + 3580 / 9) / 3600)


def test_nD_is_nine():
    assert JOB.n_D() == pytest.approx(9.0, rel=1e-9)


def test_yD_zero_at_nD_and_nan_below():
    assert S.y_D(JOB.n_D(), JOB, None, R, D_) == 0.0
    assert np.isnan(S.y_D(5.0, JOB, None, R, D_))


def test_yD_solves_deadline():
    from gnn_spot.models.spot import H
    n = 20.0
    y = S.y_D(n, JOB, None, R, D_)
    assert JOB.T_N(n) * H(y, R, D_) == pytest.approx(JOB.D, rel=1e-10)


@pytest.mark.parametrize("stall", ["fixed", "overlap"])
def test_derivative_identity(stall):
    n, h, lw = 25.0, 1e-5, 1 / 3
    num = (S.binding_cost(n + h, JOB, lw, R, D_, RHO, stall) - S.binding_cost(n - h, JOB, lw, R, D_, RHO, stall)) / (2 * h)
    assert S.dC_dn_analytic(n, JOB, lw, R, D_, RHO, stall) == pytest.approx(num, rel=1e-5)


def test_local_condition_fails_just_above_nD():
    lhs, rhs, holds = S.marginal_condition(JOB.n_D() * 1.0001, JOB, 1 / 3, R, D_, RHO)
    assert not holds and rhs > lhs


def test_binding_cost_at_nD_is_pi_D_nD():
    assert S.binding_cost(JOB.n_D(), JOB, 1 / 3, R, D_, RHO) == pytest.approx(JOB.D * JOB.n_D())


def test_global_condition_consistent_with_cost():
    lw = 1 / 3
    for n in (12.0, 21.0, 40.0):
        better = S.binding_cost(n, JOB, lw, R, D_, RHO) < JOB.D * JOB.n_D()
        assert S.global_condition(n, JOB, lw, R, D_, RHO) == better


def test_integer_brute_force_never_beats_relaxation():
    lw = 1 / 3
    bf = S.brute_force_integer(JOB, lw, R, D_, RHO, n_max=40)
    ints = np.arange(9, 41)
    relax = min(min(S._spot_endpoint_cost(float(k), JOB, lw, R, D_, RHO, "fixed") for k in ints), bf["best_no_spot"]["cost"])
    assert bf["best"]["cost"] >= relax * (1 - 1e-9)


In [ ]:
%%writefile tests/unit/test_spot.py
import numpy as np, pytest
from gnn_spot.models import spot
from gnn_spot.models.lambert import psi


def test_lambda_bounds():
    u = np.linspace(0, 5, 50)
    assert np.all(spot.Lambda(u, "fixed") <= spot.Lambda(u, "overlap") + 1e-15) and np.all(spot.Lambda(u, "fixed") >= 1 + u - 1e-15)


def test_bad_stall():
    with pytest.raises(ValueError):
        spot.Lambda(1.0, "x")


def test_aggregate_units():
    la, u, v = spot.aggregate(16, 0.5, 0.1, 300 / 3600, 30 / 3600)
    assert la == pytest.approx(0.8) and u == pytest.approx(0.8 * 300 / 3600) and v == pytest.approx(0.8 * 30 / 3600)


def test_cost_factor_zero_spot_is_one():
    assert spot.cost_factor(10, 0.0, 0.35, 0.1, 0.1, 0.01) == pytest.approx(1.0)


def test_cost_factor_formula():
    n, s, rho, lw, R, d = 20, 0.4, 0.35, 0.05, 0.1, 0.01
    la, u, v = spot.aggregate(n, s, lw, R, d)
    assert spot.cost_factor(n, s, rho, lw, R, d) == pytest.approx((1 - (1 - rho) * s) * (1 + u) * psi(v))


@pytest.mark.parametrize("stall", ["fixed", "overlap"])
@pytest.mark.parametrize("m,dr", [(2/3, 0.1), (1/3, 1.0), (0.14, 0.5)])
def test_breakeven_root(m, dr, stall):
    u = spot.breakeven_u(m, dr, stall)
    assert m * spot.Lambda(u, stall) * psi(u * dr) == pytest.approx(1.0, rel=1e-12)


def test_young_breakeven_exceeds_exact():
    for m in (2/3, 1/3, 0.14):
        for dr in (0.1, 1.0):
            assert spot.breakeven_u_young(m, dr) > spot.breakeven_u(m, dr, "fixed")


def test_mz_affine_in_s():
    s = np.linspace(0, 1, 5)
    mz = spot.m_z(s, 0.35, 32, 40.0)
    assert np.allclose(np.diff(mz, 2), 0)


def test_ell_diverges_at_zero():
    assert spot.ell(1e-12, 0.1, 0.01) > 1e3 and spot.ell(1.0, 0.1, 0.01) < spot.ell(1e-6, 0.1, 0.01)


def test_ell_matches_numeric():
    y, h = 0.7, 1e-6
    num = (np.log(spot.H(y + h, 0.1, 0.01, "overlap")) - np.log(spot.H(y - h, 0.1, 0.01, "overlap"))) / (2 * h)
    assert spot.ell(y, 0.1, 0.01, "overlap") == pytest.approx(num, rel=1e-6)


## 2. Environment check and tests

In [ ]:
import importlib
missing=[m for m in ['numpy','scipy','pandas','matplotlib','yaml','mpmath','pytest','hypothesis'] if importlib.util.find_spec(m) is None]
print('missing:', missing)
if missing:
    pipnames={'yaml':'pyyaml'}
    subprocess.run([sys.executable,'-m','pip','install','-q',*[pipnames.get(m,m) for m in missing]])
os.environ['PYTHONPATH']=os.path.join(PROJ,'src')
sys.path.insert(0, os.path.join(PROJ,'src'))

In [ ]:
r = subprocess.run([sys.executable,'-m','pytest','-q','-p','no:cacheprovider'],cwd=PROJ,capture_output=True,text=True)
print(r.stdout[-1500:], r.stderr[-500:])

## 3. Reproduce the paper (quick mode, ~2 min). Change to `full` for paper-scale trial counts (~4 min).

In [ ]:
MODE='quick'
r = subprocess.run([sys.executable,'-m','gnn_spot','reproduce','--mode',MODE],cwd=PROJ,capture_output=True,text=True)
print(r.stdout[-2500:]); print(r.stderr[-1500:])

## 4. Results

In [ ]:
import json, pandas as pd
d=json.load(open(f'results/raw/verification_{MODE}.json'))
rs=d['results']
print(sum(x['status']=='PASS' for x in rs),'/',len(rs),'checks PASS')
for x in rs:
    if x['status']!='PASS': print('NOT PASS:',x['id'],'-',x['claim'],'| computed',x['computed'])
pd.DataFrame([{k:x[k] for k in ('id','paper_ref','evidence','status')} for x in rs])

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f, width=700))

In [ ]:
pd.read_csv(f'results/tables/robustness_effect_sizes_{MODE}.csv').round(3)

In [ ]:
print(open('paper/reproduction_report.md').read()[:6000])